# Phase 6 — Web Development with FastAPI

**Outcome:** build, secure, test, observe and deploy a FastAPI service, and explain
every layer from the TCP socket to the response model.

| Unit | Topic |
| --- | --- |
| 6.1 | HTTP and the Python web stack from scratch |
| 6.2 | FastAPI fundamentals and Pydantic v2 |
| 6.3 | Dependency injection and application structure |
| 6.4 | Databases with SQLAlchemy 2.0 |
| 6.5 | Authentication, authorization and security |
| 6.6 | Background jobs, real time and caching |
| 6.7 | API testing and observability |
| 6.8 | Deployment and operations |
| Project | Notes API in production |

**The Notes API.** One application grows through Units 6.2–6.8. Its code lives in
`projects/notes_api/`, a separate uv project (its own `pyproject.toml`, `uv.lock`,
`src/notes_api/` package and `tests/`) so it can be containerized in Unit 6.8. The
notebook **writes those files** with `%%writefile` cells, imports the app in the
kernel, tests it with `TestClient` and `httpx.AsyncClient`, and serves it with
Uvicorn. Re-running the notebook rewrites the same files, so the folder always matches
the last unit you ran. To serve it from a terminal:

```bash
cd projects/notes_api
uv run uvicorn notes_api.main:app --reload
# then open http://127.0.0.1:8000/docs
```

**How to use this notebook**

- Select the **Python 3.15 (course)** kernel, then run the *Shared setup* cell first.
- Each unit starts with its own setup cell, so you can jump straight to any unit.
- **Servers are started and stopped by the cell that needs them.** When a demo needs
  a real server, the cell starts it in a background process on a free port, waits
  for its `/health` endpoint, runs the demo and stops the server, even if the demo
  fails. No server outlives its cell.
- **No public network and no cloud account.** HTTP demos talk to servers on
  `127.0.0.1`, to `httpx.MockTransport` / ASGI transports, or to the local shop API
  from Unit 4.6. Third-party services (OAuth providers, email) are faked locally.
- **PostgreSQL and Redis** (from Unit 6.4) run in Docker when it is available;
  otherwise the cells run on SQLite / fakeredis and print a clear skip message for
  the parts that need the real service.
- **Secrets** are never committed: `.env` is git-ignored, `.env.example` is committed,
  and demo keys are generated when the cell runs.
- **Optional packages are detected, never assumed.** Packages outside the course
  dependencies (Flask, Django, Litestar, Gunicorn, aiohttp, h2) are checked with
  `importlib.util.find_spec`; when one is missing its cell prints a skip message.
- *Predict the output* questions hide the answer in a collapsible block: decide
  first, then run the proof cell.
- Labs: replace the `raise NotImplementedError` in each stub, then run the check
  cell. Until then the checks print FAIL; that is expected.
- Timings are measurements on your machine, not facts; no cell asserts a speedup
  except the loose sanity checks named in a lab.
- Features newer than Python 3.12 are labelled (*New in 3.13*, *Changed in 3.14*, …).

## What you need from Phases 1–5

A web framework is mostly the language features you already know, wired together.

| Earlier idea | Unit | Where Phase 6 uses it |
| --- | --- | --- |
| `bytes` vs `str`, encodings | 1.4 | HTTP is bytes on a socket; headers are Latin-1, bodies are usually UTF-8 |
| Dict dispatch (`ROUTES`, `dispatch`) | 1.6.14 | the router of the raw socket server (6.1.4) |
| Decorators, `functools.wraps` | 1.12 | `@app.get(...)` registers a function and returns it unchanged |
| Annotation-based injection (`inject`) | 1.12.11 | FastAPI reads parameter annotations to decide where each value comes from |
| Exceptions with `status_code` (`AppError`) | 1.14.5 | `HTTPException` and exception handlers |
| Descriptors, metaclasses, `__init_subclass__` | 2.5 | how Pydantic builds a model class from annotations |
| `dataclass`, `TypedDict`, enums | 2.6 | request records, enum path parameters |
| `inspect.signature`, annotations | 3.x, 4.2 | 6.2.10: what FastAPI does with your function signature |
| `pytest`, fixtures | 4.3 | the Notes API test suite |
| Pydantic, `httpx`, the shop API | 4.6 | request/response models; HTTP clients (6.1.8) |
| Threads, the event loop, blocking calls | 5.1–5.4 | `async def` vs `def` handlers and the thread pool (6.2.8) |

## Shared setup

This cell runs first in every notebook of the course. It imports common modules and
defines three helpers used everywhere:

| Helper | What it does |
| --- | --- |
| `run_checks(test_cases)` | runs `(label, thunk, expected)` cases, prints PASS/FAIL |
| `show_size(label, obj)` | prints `sys.getsizeof(obj)`, type name and `id()` |
| `find_repo_root()` | finds the folder holding `pyproject.toml` |

`run_checks` catches every exception, so a failing check never stops Run All.

> **Note:** each unit re-imports what it needs, so you can run any unit on its own
> after this cell.

In [1]:
import inspect
import os
import platform
import sys
from collections.abc import Callable
from pathlib import Path

type TestCase = tuple[str, Callable[[], object], object]


def run_checks(test_cases: list[TestCase]) -> None:
    """Run (label, thunk, expected) cases and print PASS/FAIL for each. Never raises."""
    passed_count = 0
    for label, thunk, expected in test_cases:
        try:
            actual = thunk()
        except Exception as error:  # a check cell must never stop Run All
            print(f"FAIL  {label}: raised {type(error).__name__}: {error}")
            continue
        if actual == expected:
            passed_count += 1
            print(f"PASS  {label}")
        else:
            print(f"FAIL  {label}: expected {expected!r}, got {actual!r}")
    print(f"{passed_count}/{len(test_cases)} checks passed")


def show_size(label: str, obj: object) -> None:
    """Print the shallow size in bytes, the type and the id of an object."""
    print(f"{label}: {sys.getsizeof(obj)} bytes, type={type(obj).__name__}, id={id(obj)}")


def find_repo_root(start: Path | None = None) -> Path:
    """Walk up from start (default: the working directory) to the folder holding pyproject.toml."""
    current = (start or Path.cwd()).resolve()
    for folder in (current, *current.parents):
        if (folder / "pyproject.toml").exists():
            return folder
    raise FileNotFoundError("pyproject.toml not found above the working directory")


print(f"Python {platform.python_version()} ({sys.implementation.name}) on {platform.system()}")
print(f"Repository root: {find_repo_root().name}/")

Python 3.15.1 (cpython) on Linux
Repository root: python-course/


## Unit 6.1 — HTTP and the Python Web Stack from Scratch

**Objectives**

- Explain what happens between a browser and a Python function: TCP/IP, ports,
  DNS, TLS, and the `socket` API.
- Read and write HTTP/1.1 messages by hand: request and status lines, headers,
  bodies, CRLF, keep-alive, chunked encoding, content negotiation.
- Use method semantics and status codes correctly (safe vs idempotent; 401 vs 403;
  409, 422, 429) and design REST resources with pagination, versioning and
  idempotency keys.
- Build an HTTP server on raw sockets three ways (blocking, a thread per
  connection, asyncio streams) and benchmark them, next to Node's `http` module.
- Write a WSGI app and an ASGI app with no framework, add middleware to each, and
  serve them; see exactly what Uvicorn and FastAPI sit on.
- Pick an HTTP client (`urllib`, `requests`, `httpx`, `aiohttp`) and use it well:
  connection reuse, timeouts, retries, streaming.

**JS bridge**

| Node.js | Python | The difference that bites |
| --- | --- | --- |
| `net.createServer` | `socket.create_server` / `asyncio.start_server` | Python's plain socket API is blocking by default; Node's is always event-driven |
| `http.createServer((req, res) => …)` | a WSGI or ASGI callable | Python separates the **server** (Gunicorn, Uvicorn) from the **app** by a standard interface |
| `req.on("data")` / streams | ASGI `receive()` messages with `more_body` | the body arrives as messages you pull, not events pushed at you |
| Express / Fastify / NestJS | Flask / Starlette + FastAPI / Django, Litestar | FastAPI validates with type hints (like zod + NestJS DTOs) |
| `fetch` / `undici` / `axios` | `httpx` (sync and async), `requests` | `requests` has **no default timeout**; always pass one |
| `ws` / Socket.IO | ASGI `websocket` scope | WebSockets are part of the ASGI interface itself |

Run the three setup cells before the sections below. The first is the local shop
API from Unit 4.6 (copied unchanged): a `BaseHTTPRequestHandler` served by
`ThreadingHTTPServer` in a daemon thread on `127.0.0.1:18046`, used as a real
HTTP/1.1 server to talk to.

In [2]:
import http.server
import json
import random
import threading
import time
import urllib.parse

SHOP_API_PORT = 18046  # fixed so outputs (URLs in messages) are stable; falls back to any free port


def demo_orders() -> list[dict[str, object]]:
    """40 seeded orders as an API would return them, with 3 invalid records planted."""
    generator = random.Random(46)
    cities = ["Pune", "Mumbai", "Bengaluru", "Hubli & Dharwad"]
    prices = {"TEA-500": "240.00", "MUG-02": "199.00", "JAR-750": "350.50", "KETTLE-9": "899.00"}
    orders: list[dict[str, object]] = []
    for order_id in range(1, 41):
        sku = generator.choice(sorted(prices))
        orders.append(
            {
                "order_id": order_id,
                "customer": f"customer-{order_id:03d}",
                "city": generator.choice(cities),
                "sku": sku,
                "quantity": generator.randint(1, 5),
                "unit_price": prices[sku],
                "placed_at": f"2026-11-{generator.randint(1, 7):02d}"
                f"T{generator.randint(9, 20):02d}:00:00+05:30",
            }
        )
    orders[6]["quantity"] = -2  # invalid: not positive
    del orders[17]["city"]  # invalid: missing field
    orders[29]["unit_price"] = "two hundred"  # invalid: not a number
    return orders


PRODUCTS_HTML = """<!doctype html>
<html><head><title>Chai Corner - Products</title></head>
<body>
  <h1>Products</h1>
  <ul id="catalogue">
    <li class="product" data-sku="TEA-500">
      <a href="/products/tea-500">Masala tea 500 g</a> <span class="price">₹240.00</span>
    </li>
    <li class="product" data-sku="MUG-02">
      <a href="/products/mug-02">Steel mug</a> <span class="price">₹199.00</span>
    </li>
    <li class="product sold-out" data-sku="JAR-750">
      <a href="/products/jar-750">Pickle jar 750 ml</a> <span class="price">₹350.50</span>
    </li>
  </ul>
  <p class="note">Prices include GST &amp; free delivery above ₹500.</p>
</body></html>
"""
ROBOTS_TXT = "User-agent: *\nDisallow: /admin/\nCrawl-delay: 1\n\nUser-agent: BadBot\nDisallow: /\n"


class ShopAPIHandler(http.server.BaseHTTPRequestHandler):
    """A tiny local HTTP API used by every HTTP demo (no external network)."""

    protocol_version = "HTTP/1.1"  # keep-alive, so clients can reuse connections
    hits: dict[str, int] = {}
    connections = 0

    def setup(self) -> None:
        """Count each new TCP connection."""
        type(self).connections += 1  # one per TCP connection, not per request
        super().setup()

    def log_message(self, format: str, *args: object) -> None:  # keep the notebook quiet
        """Silence the default request log."""
        pass

    def _send(self, status: int, body: object, content_type: str = "application/json") -> None:
        """Send status and a JSON or text body."""
        data = body.encode() if isinstance(body, str) else json.dumps(body).encode()
        try:
            self.send_response(status)
            self.send_header("Content-Type", f"{content_type}; charset=utf-8")
            self.send_header("Content-Length", str(len(data)))
            self.end_headers()
            self.wfile.write(data)
        except BrokenPipeError, ConnectionResetError:  # the client gave up (a timeout demo)
            self.close_connection = True

    def _count(self, key: str) -> int:
        """Increment and return the hit counter for key."""
        self.hits[key] = self.hits.get(key, 0) + 1
        return self.hits[key]

    def do_GET(self) -> None:
        """Route GET requests to the demo endpoints."""
        url = urllib.parse.urlsplit(self.path)
        query = dict(urllib.parse.parse_qsl(url.query))
        parts = url.path.strip("/").split("/")
        fail_first = int(query.get("fail", 0))
        if url.path == "/rates":
            base = query.get("base", "INR")
            self._send(
                200,
                {
                    "base": base,
                    "date": "2026-11-07",
                    "rates": {"USD": 0.012, "EUR": 0.011, "GBP": 0.0094},
                },
            )
        elif url.path == "/orders":
            if "key" in query and self._count(query["key"]) <= fail_first:
                self._send(503, {"error": "service busy, try again"})
            else:
                self._send(200, demo_orders())
        elif parts[0] == "flaky" and len(parts) == 2:
            attempt = self._count(parts[1])
            if attempt <= fail_first:
                self._send(503, {"error": "busy", "attempt": attempt})
            else:
                self._send(200, {"ok": True, "attempt": attempt})
        elif url.path == "/hits":
            self._send(
                200, {"key": query.get("key"), "hits": self.hits.get(query.get("key", ""), 0)}
            )
        elif url.path == "/connections":
            self._send(200, {"connections": type(self).connections})
        elif url.path == "/reset":
            self.hits.clear()
            self._send(200, {"reset": True})
        elif url.path == "/slow":
            time.sleep(float(query.get("seconds", "2")))
            self._send(200, {"slow": True})
        elif parts[0] == "status" and len(parts) == 2:
            self._send(int(parts[1]), {"status": int(parts[1])})
        elif url.path == "/headers":
            self._send(
                200,
                {
                    name.lower(): value
                    for name, value in self.headers.items()
                    if name.lower() in ("user-agent", "x-request-id", "authorization")
                },
            )
        elif url.path == "/products.html":
            self._send(200, PRODUCTS_HTML, "text/html")
        elif url.path == "/robots.txt":
            self._send(200, ROBOTS_TXT, "text/plain")
        else:
            self._send(404, {"error": f"no route {url.path}"})

    def do_POST(self) -> None:
        """Echo the posted JSON back."""
        length = int(self.headers.get("Content-Length", 0))
        payload = json.loads(self.rfile.read(length) or b"null")
        self._send(201 if self.path == "/orders" else 200, {"received": payload})


def start_shop_api() -> str:
    """Start the local API once (in a daemon thread) and return its base URL."""
    server = globals().get("_SHOP_API_SERVER")
    if server is None:
        try:
            server = http.server.ThreadingHTTPServer(("127.0.0.1", SHOP_API_PORT), ShopAPIHandler)
        except OSError:
            server = http.server.ThreadingHTTPServer(("127.0.0.1", 0), ShopAPIHandler)
        server.daemon_threads = True
        threading.Thread(target=server.serve_forever, daemon=True).start()
        globals()["_SHOP_API_SERVER"] = server
    return f"http://127.0.0.1:{server.server_address[1]}"

Unit setup, part 1: imports, the scratch folder `notebooks/_scratch/phase6/`
(git-ignored; `%%writefile` cells save demo apps there), flags for optional
packages, and `SHOP_API`, the base URL of the shop API.

In [3]:
import asyncio
import base64
import contextlib
import dataclasses
import http
import http.client
import importlib
import importlib.util
import inspect
import json
import os
import platform
import re
import shutil
import socket
import ssl
import subprocess
import sys
import sysconfig
import threading
import time
import urllib.error
import urllib.parse
import urllib.request
from collections.abc import Awaitable, Callable, Iterable, Iterator
from http import HTTPStatus
from pathlib import Path
from types import ModuleType
from typing import Any

import httpx

REPO_ROOT = find_repo_root()
scratch_dir = REPO_ROOT / "notebooks" / "_scratch"  # git-ignored
PHASE6_DIR = scratch_dir / "phase6"
PHASE6_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(REPO_ROOT / "notebooks")  # %%writefile cells use paths relative to notebooks/
if str(PHASE6_DIR) not in sys.path:
    sys.path.insert(0, str(PHASE6_DIR))
CHILD_ENV = {**os.environ, "PYTHON_COLORS": "0", "PYTHONUNBUFFERED": "1", "NO_COLOR": "1"}
CHILD_ENV.pop("FORCE_COLOR", None)


def has_module(name: str) -> bool:
    """True if the module can be imported (checked without importing it)."""
    return importlib.util.find_spec(name) is not None


OPTIONAL_PACKAGES = {
    name: has_module(name)
    for name in (
        "requests",
        "h2",
        "aiohttp",
        "websockets",
        "uvicorn",
        "starlette",
        "fastapi",
        "flask",
        "django",
        "litestar",
        "gunicorn",
    )
}
HAS_NODE = shutil.which("node") is not None
HAS_OPENSSL = shutil.which("openssl") is not None
SHOP_API = start_shop_api()  # the local API from Unit 4.6

Unit setup, part 2: the error and timing helpers shared since earlier phases
(`raised_error_name`, `raised_message`, `interpreter_label`, `Timer`),
`fresh_import` (re-imports a `%%writefile` module) and `mask_dates`, which hides the
parts of raw HTTP output that change on every run (the `Date` header, ports).

In [4]:
def raised_error_name(thunk: Callable[[], object]) -> str:
    """Call thunk and return the name of the exception it raised, or "no error"."""
    try:
        thunk()
    except Exception as error:
        return type(error).__name__
    return "no error"


def raised_message(thunk: Callable[[], object]) -> str:
    """Call thunk and return "ErrorName: message" (memory addresses hidden), or "no error"."""
    try:
        thunk()
    except Exception as error:
        return re.sub(r" at 0x[0-9a-f]+", "", f"{type(error).__name__}: {error}")
    return "no error"


def interpreter_label() -> str:
    """Implementation, version and build flavour: print it next to every benchmark."""
    free_threaded = bool(sysconfig.get_config_var("Py_GIL_DISABLED"))
    build = "free-threaded build" if free_threaded else "default (GIL) build"
    implementation = platform.python_implementation()
    return f"{implementation} {platform.python_version()}, {build}, {platform.machine()}"


class Timer:
    """Timing context manager (shared since Unit 1.14): reports on exit, keeps elapsed_seconds."""

    def __init__(self, label: str, report: Callable[[str], None] | None = print) -> None:
        """Remember the label and where to report the elapsed time."""
        self.label = label
        self.report = report
        self.elapsed_seconds = 0.0

    def __enter__(self) -> Timer:
        """Start the clock."""
        self._start = time.perf_counter()
        return self

    def __exit__(self, *exc_info: object) -> None:
        """Stop the clock and report the elapsed milliseconds."""
        self.elapsed_seconds = time.perf_counter() - self._start
        if self.report is not None:
            self.report(f"{self.label} took {self.elapsed_seconds * 1000:.2f} ms")


def fresh_import(module_name: str) -> ModuleType:
    """Import a module saved by a %%writefile cell, always re-reading the file from disk."""
    for cached in (PHASE6_DIR / "__pycache__").glob(f"{module_name}.*.pyc"):
        cached.unlink()
    importlib.invalidate_caches()
    sys.modules.pop(module_name, None)
    return importlib.import_module(module_name)


def mask_dates(text: str) -> str:
    """Replace HTTP Date header values and ephemeral port numbers with stable placeholders."""
    text = re.sub(r"(?im)^(date: ).*$", r"\1<date>", text)
    return re.sub(r"127\.0\.0\.1:\d+", "127.0.0.1:<port>", text)

Unit setup, part 3: the **server helpers** every Phase 6 unit uses.

| Helper | What it does |
| --- | --- |
| `free_port()` | asks the OS for an unused port (bind to port 0) |
| `wait_for_health(url, process)` | polls `url` until it answers 200; fails fast if the process died |
| `background_server(command, port)` | context manager: starts the process, waits for `/health`, yields the base URL, **always** terminates the process in `finally` |
| `uvicorn_server("module:app")` | `background_server` for `python -m uvicorn module:app` on a free port |

Server output goes to a log file in the scratch folder (a pipe nobody reads can
fill up and freeze the server); when a server fails to start, the helper prints the
end of its log.

In [5]:
NO_PROXY_OPENER = urllib.request.build_opener(urllib.request.ProxyHandler({}))


def free_port() -> int:
    """Ask the OS for a free TCP port on 127.0.0.1 (bind to port 0, read it, release it)."""
    with socket.socket() as probe:
        probe.bind(("127.0.0.1", 0))
        return probe.getsockname()[1]


def wait_for_health(
    health_url: str, process: subprocess.Popen[bytes] | None = None, timeout: float = 15.0
) -> None:
    """Poll health_url until it answers 200; fail fast if the server process exits."""
    deadline = time.monotonic() + timeout
    while time.monotonic() < deadline:
        if process is not None and process.poll() is not None:
            raise RuntimeError(f"server exited early with code {process.returncode}")
        try:
            with NO_PROXY_OPENER.open(health_url, timeout=1) as response:
                if response.status == 200:
                    return
        except OSError:
            time.sleep(0.05)
    raise TimeoutError(f"{health_url} did not become healthy within {timeout} s")


@contextlib.contextmanager
def background_server(
    command: list[str],
    port: int,
    *,
    health_path: str = "/health",
    cwd: Path = PHASE6_DIR,
    extra_env: dict[str, str] | None = None,
    timeout: float = 15.0,
) -> Iterator[str]:
    """Start a server process, wait for its health check, yield its base URL, always stop it."""
    log_path = PHASE6_DIR / f"server_{port}.log"
    with log_path.open("w", encoding="utf-8") as log_file:
        process = subprocess.Popen(
            command,
            stdout=log_file,
            stderr=subprocess.STDOUT,
            cwd=cwd,
            env={**CHILD_ENV, **(extra_env or {})},
        )
    base_url = f"http://127.0.0.1:{port}"
    started = False
    try:
        try:
            wait_for_health(base_url + health_path, process, timeout)
        except RuntimeError, TimeoutError:
            print(log_path.read_text(encoding="utf-8")[-2000:])  # show why it did not start
            raise
        started = True
        yield base_url
    finally:
        process.terminate()
        try:
            process.wait(timeout=5)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait()
        if started:
            log_path.unlink(missing_ok=True)  # keep the log only when startup failed


def uvicorn_server(
    app_spec: str, *, app_dir: Path = PHASE6_DIR, extra_args: Iterable[str] = (), **options: Any
) -> contextlib.AbstractContextManager[str]:
    """Serve "module:attribute" with Uvicorn in a child process (see background_server)."""
    port = free_port()
    host_and_port = ["--host", "127.0.0.1", "--port", str(port)]
    command = [sys.executable, "-m", "uvicorn", app_spec, "--app-dir", str(app_dir), *host_and_port]
    command += ["--log-level", "warning", *extra_args]
    return background_server(command, port, **options)


print(interpreter_label())
print(f"{SHOP_API=}")
found_flags = (f"{name} {'yes' if found else 'no'}" for name, found in OPTIONAL_PACKAGES.items())
print("optional packages: " + ", ".join(found_flags))
print(f"{HAS_NODE=}  {HAS_OPENSSL=}")

CPython 3.15.1, default (GIL) build, x86_64
SHOP_API='http://127.0.0.1:18046'
optional packages: requests yes, h2 no, aiohttp no, websockets yes, uvicorn yes, starlette yes, fastapi yes, flask no, django no, litestar no, gunicorn no
HAS_NODE=True  HAS_OPENSSL=True


### 6.1.1 Networking basics

An HTTP request is the top of a stack of protocols. Each layer only talks to the
layer below it:

| Layer | Protocol | What it gives you | Python |
| --- | --- | --- | --- |
| Application | HTTP/1.1, HTTP/2, WebSocket | requests, responses, headers | `http.client`, `httpx`, your framework |
| Security (optional) | TLS 1.2 / 1.3 | encryption, server identity (certificates) | `ssl` |
| Transport | **TCP** (or UDP) | a reliable, ordered **byte stream** between two ports | `socket.SOCK_STREAM` |
| Internet | IPv4 / IPv6 | delivery of packets between addresses | `socket.AF_INET`, `AF_INET6` |
| Link | Ethernet, Wi-Fi, loopback | frames on one network | the OS |

- A **port** (0–65535) picks the program on a host: 80 for HTTP, 443 for HTTPS, 8000
  is the Uvicorn default. A connection is identified by four numbers: client
  address and **ephemeral port** (picked by the OS), server address and port.
- **DNS** turns a name into addresses. `getaddrinfo` returns every address to try
  (IPv6 and IPv4) for a name and port.
- **TCP** is a byte stream with no message boundaries. Everything HTTP does to mark
  "where a request ends" (CRLF lines, `Content-Length`, chunked encoding) exists
  because TCP will not do it.
- **TLS** sits between TCP and HTTP: a handshake agrees on keys and proves the
  server's identity with a certificate; then every byte is encrypted.

```mermaid
flowchart LR
    Client["httpx / browser"] -->|"HTTP bytes"| TLSc["TLS"]
    TLSc -->|"encrypted bytes"| TCPc["TCP :54321"]
    TCPc -->|"IP packets"| TCPs["TCP :443"]
    TCPs --> TLSs["TLS"]
    TLSs -->|"HTTP bytes"| Server["Uvicorn → your app"]
```

<details><summary>ASCII version</summary>

```text
client app ──HTTP──▶ TLS ──▶ TCP (ephemeral port) ══ IP ══▶ TCP :443 ──▶ TLS ──▶ server app
```

</details>

`getaddrinfo` for `localhost` (no network needed). The list order and whether IPv6
appears depend on the OS configuration:

In [6]:
for family, kind, _proto, _canonical_name, address in socket.getaddrinfo(
    "localhost", 8000, type=socket.SOCK_STREAM
):
    print(f"{family.name:<9} {kind.name:<12} {address}")
print(f"{socket.getservbyname('http')=}  {socket.getservbyname('https')=}")

AF_INET6  SOCK_STREAM  ('::1', 8000, 0, 0)
AF_INET   SOCK_STREAM  ('127.0.0.1', 8000)
socket.getservbyname('http')=80  socket.getservbyname('https')=443


#### The socket API

| Step | Server | Client | Node `net` |
| --- | --- | --- | --- |
| create | `socket.socket(AF_INET, SOCK_STREAM)` | same | `net.createServer()` / `new net.Socket()` |
| address | `bind(("127.0.0.1", port))` | — | `server.listen(port)` |
| wait | `listen(backlog)` then `accept()` → new socket per client | `connect((host, port))` | `"connection"` event |
| talk | `recv(n)` / `sendall(data)` | same | `"data"` event / `write()` |
| finish | `close()` | `close()` | `end()` |

`socket.create_server` and `socket.create_connection` wrap the first three steps.
Below, an echo server runs in a thread; the client sends bytes and reads them back.
The client's port is an ephemeral port the OS picked, so it changes on every run:

In [7]:
def echo_once(server_socket: socket.socket) -> None:
    """Accept one client, echo what it sends in upper case, then close."""
    connection, _client_address = server_socket.accept()
    with connection:
        data = connection.recv(1024)
        connection.sendall(data.upper())


with socket.create_server(("127.0.0.1", 0)) as echo_server:
    server_port = echo_server.getsockname()[1]
    echo_thread = threading.Thread(target=echo_once, args=(echo_server,))
    echo_thread.start()
    with socket.create_connection(("127.0.0.1", server_port), timeout=5) as client:
        client.sendall(b"hello tcp")
        reply = client.recv(1024)
        client_port = client.getsockname()[1]
    echo_thread.join()
print(f"{reply=}")
print(f"the client got an ephemeral port from the OS: {client_port > 1024=}")

reply=b'HELLO TCP'
the client got an ephemeral port from the OS: client_port > 1024=True


**TCP has no message boundaries.** The client below calls `sendall` twice; the
server waits a moment and then reads once. It gets both writes glued together,
because TCP delivers a stream of bytes, not messages. HTTP therefore needs its own
framing: a blank line ends the headers, and `Content-Length` (or chunked encoding)
says where the body ends. (On a loaded machine the read could, in rare cases, return
only the first part; that is exactly why a server must loop until it has a whole
request.)

In [8]:
def read_after_pause(server_socket: socket.socket, received: list[bytes]) -> None:
    """Accept one client, wait 100 ms so both writes arrive, then read once."""
    connection, _client_address = server_socket.accept()
    with connection:
        time.sleep(0.1)
        received.append(connection.recv(1024))


received_chunks: list[bytes] = []
with socket.create_server(("127.0.0.1", 0)) as stream_server:
    reader_thread = threading.Thread(target=read_after_pause, args=(stream_server, received_chunks))
    reader_thread.start()
    with socket.create_connection(stream_server.getsockname(), timeout=5) as client:
        client.sendall(b"hello")
        client.sendall(b"world")
    reader_thread.join()
print(f"{received_chunks=}")

received_chunks=[b'helloworld']


For contrast, **UDP** (`SOCK_DGRAM`) keeps message boundaries but promises no
delivery or order. DNS queries use it, and so does HTTP/3 (through QUIC). Each
`recvfrom` returns exactly one datagram:

In [9]:
with (
    socket.socket(socket.AF_INET, socket.SOCK_DGRAM) as udp_server,
    socket.socket(socket.AF_INET, socket.SOCK_DGRAM) as udp_client,
):
    udp_server.bind(("127.0.0.1", 0))
    udp_server.settimeout(5)
    for datagram in (b"hello", b"world"):
        udp_client.sendto(datagram, udp_server.getsockname())
    datagrams = [udp_server.recvfrom(1024)[0] for _ in range(2)]
print(f"{datagrams=}")

datagrams=[b'hello', b'world']


#### TLS in one cell

A TLS 1.3 handshake, simplified: the client sends *ClientHello* (versions, ciphers,
the server name it wants (**SNI**), the application protocols it speaks (**ALPN**:
`h2`, `http/1.1`)); the server answers with its certificate and key share; both
derive the same keys; from then on every record is encrypted. The client checks
that the certificate is signed by a trusted authority **and** names the host it
dialled.

`ssl.create_default_context()` gives safe client defaults: certificate checks on,
hostname checks on, old protocol versions off:

In [10]:
client_context = ssl.create_default_context()
print(f"{ssl.OPENSSL_VERSION=}")
print(f"{client_context.minimum_version=}")
print(f"{client_context.verify_mode=}  {client_context.check_hostname=}")
print(f"{ssl.HAS_ALPN=}  {ssl.HAS_TLSv1_3=}")

ssl.OPENSSL_VERSION='OpenSSL 3.5.9 29 Sep 2026'
client_context.minimum_version=<TLSVersion.TLSv1_2: 771>
client_context.verify_mode=<VerifyMode.CERT_REQUIRED: 2>  client_context.check_hostname=True
ssl.HAS_ALPN=True  ssl.HAS_TLSv1_3=True


A real handshake on `127.0.0.1`. The cell creates a throwaway self-signed
certificate for `localhost` with the `openssl` command (skipped if it is missing),
starts a TLS echo server in a thread, and connects twice: once with the default
context (fails: the certificate is not signed by a trusted authority), once with a
context that trusts this one certificate. The negotiated version, cipher and ALPN
protocol are printed. Nothing secret is stored: the key lives in the scratch folder
and is regenerated every run.

In [11]:
def make_localhost_certificate(folder: Path) -> tuple[Path, Path] | None:
    """Create a 1-day self-signed certificate for localhost; None without openssl."""
    if not HAS_OPENSSL:
        return None
    cert_path, key_path = folder / "localhost.crt", folder / "localhost.key"
    subprocess.run(
        [
            "openssl",
            "req",
            "-x509",
            "-newkey",
            "rsa:2048",
            "-nodes",
            "-days",
            "1",
            "-keyout",
            str(key_path),
            "-out",
            str(cert_path),
            "-subj",
            "/CN=localhost",
            "-addext",
            "subjectAltName=DNS:localhost",
        ],
        check=True,
        capture_output=True,
    )
    return cert_path, key_path


def tls_echo_once(server_socket: socket.socket, server_context: ssl.SSLContext) -> None:
    """Accept one client, do the TLS handshake, echo one message."""
    connection, _client_address = server_socket.accept()
    try:
        with server_context.wrap_socket(connection, server_side=True) as tls_connection:
            tls_connection.sendall(tls_connection.recv(1024))
    except ssl.SSLError, OSError:  # the client rejected our certificate
        connection.close()

Run the two connections. (`except A, B:` without brackets is *new in 3.14*, PEP 758.)

In [12]:
def tls_round_trip(client_context: ssl.SSLContext, server_context: ssl.SSLContext) -> str:
    """Connect to a one-shot TLS echo server; describe the session or the error."""
    with socket.create_server(("127.0.0.1", 0)) as tls_server:
        server_thread = threading.Thread(target=tls_echo_once, args=(tls_server, server_context))
        server_thread.start()
        try:
            with (
                socket.create_connection(tls_server.getsockname(), timeout=5) as raw,
                client_context.wrap_socket(raw, server_hostname="localhost") as tls,
            ):
                tls.sendall(b"secret")
                echoed = tls.recv(1024)
                protocol = tls.selected_alpn_protocol()
                return f"{tls.version()} {tls.cipher()[0]} alpn={protocol} {echoed=}"
        except ssl.SSLCertVerificationError as error:
            return f"SSLCertVerificationError: {error.verify_message}"
        finally:
            server_thread.join()


certificate = make_localhost_certificate(PHASE6_DIR)
if certificate is None:
    print("SKIP: the openssl command is not installed, so no certificate can be made")
else:
    server_context = ssl.create_default_context(ssl.Purpose.CLIENT_AUTH)
    server_context.load_cert_chain(*certificate)
    server_context.set_alpn_protocols(["http/1.1"])
    print("default context:", tls_round_trip(ssl.create_default_context(), server_context))
    trusting_context = ssl.create_default_context(cafile=certificate[0])
    trusting_context.set_alpn_protocols(["h2", "http/1.1"])
    print("trusting context:", tls_round_trip(trusting_context, server_context))

default context: SSLCertVerificationError: self-signed certificate
trusting context: TLSv1.3 TLS_AES_256_GCM_SHA384 alpn=http/1.1 echoed=b'secret'


> **Note:** in production TLS usually ends at a load balancer or reverse proxy
> (nginx, Caddy, a cloud load balancer) and Uvicorn speaks plain HTTP behind it.
> Uvicorn can also terminate TLS itself (`--ssl-keyfile`, `--ssl-certfile`).

### 6.1.2 HTTP/1.1 anatomy

An HTTP/1.1 message is text lines ending in **CRLF** (`\r\n`), a blank line, then
an optional body:

```text
POST /notes?draft=true HTTP/1.1\r\n        ← request line: method, target, version
Host: api.example.com\r\n                  ← headers: "Name: value"
Content-Type: application/json\r\n
Content-Length: 17\r\n
\r\n                                       ← blank line: end of headers
{"title": "Tea"}                         ← body: exactly Content-Length bytes

HTTP/1.1 201 Created\r\n                   ← status line: version, code, reason
Content-Type: application/json\r\n
Location: /notes/7\r\n
Content-Length: 25\r\n
\r\n
{"id": 7, "title": "Tea"}
```

Header names are case-insensitive; the request line and headers are Latin-1 /
ASCII bytes; the body is whatever `Content-Type` says (usually UTF-8 JSON).

Below, a raw request is written by hand to the shop API's socket and the raw
response is printed exactly as it arrives (the `Date` header is masked):

In [13]:
def raw_http_exchange(base_url: str, raw_request: bytes) -> bytes:
    """Send raw_request on a fresh TCP connection and return every byte of the reply."""
    address = urllib.parse.urlsplit(base_url)
    with socket.create_connection((address.hostname, address.port), timeout=5) as sock:
        sock.sendall(raw_request)
        reply = b""
        while chunk := sock.recv(4096):  # the server closes: Connection: close
            reply += chunk
    return reply


raw_request = (
    b"GET /rates?base=INR HTTP/1.1\r\n"
    b"Host: 127.0.0.1\r\n"
    b"Accept: application/json\r\n"
    b"Connection: close\r\n"
    b"\r\n"
)
raw_reply = raw_http_exchange(SHOP_API, raw_request)
print(mask_dates(raw_reply.decode("latin-1")))

HTTP/1.1 200 OK
Server: BaseHTTP Python/3.15.1
Date: <date>
Content-Type: application/json; charset=utf-8
Content-Length: 91

{"base": "INR", "date": "2026-11-07", "rates": {"USD": 0.012, "EUR": 0.011, "GBP": 0.0094}}


The same reply with escapes visible: every line ends with `\r\n` and the head
ends with `\r\n\r\n`. Splitting on that marker separates head and body; the
body length matches `Content-Length`:

In [14]:
head, _, body = raw_reply.partition(b"\r\n\r\n")
status_line, *header_lines = head.split(b"\r\n")
print(f"{status_line=}")
print(f"{header_lines[0]=}")
content_length = next(
    int(line.split(b":")[1]) for line in header_lines if line.lower().startswith(b"content-length")
)
print(f"{content_length=}  {len(body)=}")
show_size("raw request bytes", raw_request)

status_line=b'HTTP/1.1 200 OK'
header_lines[0]=b'Server: BaseHTTP Python/3.15.1'
content_length=91  len(body)=91
raw request bytes: 127 bytes, type=bytes, id=140575377740080


#### Method semantics

| Method | Safe (no state change) | Idempotent (repeat = same effect) | Body | Typical use |
| --- | --- | --- | --- | --- |
| `GET` | yes | yes | no | read a resource or a list |
| `HEAD` | yes | yes | no | headers only (size, `ETag`) |
| `OPTIONS` | yes | yes | no | CORS preflight, allowed methods |
| `POST` | no | **no** | yes | create, or "do this action" |
| `PUT` | no | yes | yes | replace a resource at a known URL |
| `PATCH` | no | not by rule (usually yes) | yes | change some fields |
| `DELETE` | no | yes | rare | remove (the second call: 404 or 204, same server state) |

**Safe** methods may be prefetched and cached. **Idempotent** requests can be
retried after a timeout without fear; a `POST` cannot, unless it carries an
idempotency key (6.1.3). `http.HTTPMethod` (*new in 3.11*) lists the methods with
their descriptions:

In [15]:
IDEMPOTENT_METHODS = {"GET", "HEAD", "OPTIONS", "PUT", "DELETE", "TRACE"}
SAFE_METHODS = {"GET", "HEAD", "OPTIONS", "TRACE"}
for method in (
    http.HTTPMethod.GET,
    http.HTTPMethod.POST,
    http.HTTPMethod.PUT,
    http.HTTPMethod.PATCH,
    http.HTTPMethod.DELETE,
):
    safe = "safe" if method in SAFE_METHODS else "unsafe"
    repeat = "idempotent" if method in IDEMPOTENT_METHODS else "not idempotent"
    print(f"{method:<7} {safe:<7} {repeat:<15} {method.description}")

GET     safe    idempotent      Retrieve the target.
POST    unsafe  not idempotent  Perform target-specific processing with the request payload.
PUT     unsafe  idempotent      Replace the target with the request payload.
PATCH   unsafe  not idempotent  Apply partial modifications to a target.
DELETE  unsafe  idempotent      Remove the target.


#### Status codes

| Class | Meaning | Common codes |
| --- | --- | --- |
| 1xx | informational | 101 Switching Protocols (WebSocket upgrade) |
| 2xx | success | 200 OK, 201 Created (+ `Location`), 202 Accepted, 204 No Content |
| 3xx | redirection | 301/308 permanent, 302/307 temporary, 304 Not Modified |
| 4xx | **client** error: fix the request | 400, 401, 403, 404, 405, 409, 413, 415, 422, 429 |
| 5xx | **server** error: retrying may help | 500, 502, 503 (+ `Retry-After`), 504 |

The pairs people mix up:

| Code | Means | Example |
| --- | --- | --- |
| **401** Unauthorized | "who are you?": no or bad credentials; send `WWW-Authenticate` | missing or expired token |
| **403** Forbidden | "I know who you are, and no" | a user deleting someone else's note |
| **409** Conflict | the request clashes with the current state | username taken; stale version |
| **422** Unprocessable Content | well-formed JSON, invalid values | `"limit": -5`; FastAPI's validation errors |
| **429** Too Many Requests | rate limit hit; send `Retry-After` | 101st call in a minute |

`HTTPStatus` (*`is_success`, `is_client_error`, … new in 3.12*) knows every code.
*Changed in 3.13:* 422's phrase is now "Unprocessable Content" (RFC 9110), and 413
is "Content Too Large":

In [16]:
for status_code in (200, 201, 204, 301, 304, 400, 401, 403, 404, 409, 413, 422, 429, 503):
    status = HTTPStatus(status_code)
    kind = next(
        label
        for label, matches in (
            ("success", status.is_success),
            ("redirect", status.is_redirection),
            ("client error", status.is_client_error),
            ("server error", status.is_server_error),
        )
        if matches
    )
    print(f"{status.value} {status.phrase:<22} {kind}")

200 OK                     success
201 Created                success
204 No Content             success
301 Moved Permanently      redirect
304 Not Modified           redirect
400 Bad Request            client error
401 Unauthorized           client error
403 Forbidden              client error
404 Not Found              client error
409 Conflict               client error
413 Content Too Large      client error
422 Unprocessable Content  client error
429 Too Many Requests      client error
503 Service Unavailable    server error


#### Key headers

| Header | Direction | Purpose |
| --- | --- | --- |
| `Host` | request | which site (required in HTTP/1.1: many sites share one IP) |
| `Content-Type` / `Content-Length` | both | what the body is / how many bytes |
| `Accept`, `Accept-Encoding`, `Accept-Language` | request | content negotiation |
| `Authorization` | request | credentials (`Bearer <token>`) |
| `Cookie` / `Set-Cookie` | request / response | session state |
| `Cache-Control`, `ETag`, `If-None-Match` | both | caching and conditional requests (304) |
| `Location` | response | the new resource (201) or the redirect target (3xx) |
| `Retry-After` | response | when to retry (429, 503) |
| `Connection`, `Keep-Alive` | both | reuse the TCP connection or close it |
| `Transfer-Encoding: chunked` | both | body length not known in advance |

**Keep-alive.** HTTP/1.1 connections stay open by default, so several requests
can reuse one TCP (and TLS) handshake. The shop API counts TCP connections at
`/connections`. Three requests on one `http.client.HTTPConnection` open one
connection; three separate `urlopen` calls (which send `Connection: close`) open
three:

In [17]:
def shop_connection_count() -> int:
    """How many TCP connections the shop API has accepted so far."""
    with NO_PROXY_OPENER.open(f"{SHOP_API}/connections", timeout=5) as response:
        return json.load(response)["connections"]


shop_address = urllib.parse.urlsplit(SHOP_API)
before = shop_connection_count()
keep_alive = http.client.HTTPConnection(shop_address.hostname, shop_address.port, timeout=5)
for _ in range(3):
    keep_alive.request("GET", "/rates")
    keep_alive.getresponse().read()  # read the body before reusing the connection
keep_alive.close()
reused = shop_connection_count() - before - 1  # minus the counting request itself

before = shop_connection_count()
for _ in range(3):
    with NO_PROXY_OPENER.open(f"{SHOP_API}/rates", timeout=5) as response:
        response.read()
separate = shop_connection_count() - before - 1
print(f"keep-alive: 3 requests used {reused} connection(s)")
print(f"urlopen:    3 requests used {separate} connection(s)")

keep-alive: 3 requests used 1 connection(s)
urlopen:    3 requests used 3 connection(s)


**Chunked transfer encoding.** When a server does not know the body length up
front (a stream, a large export), it sends `Transfer-Encoding: chunked`: each
chunk is its size in hex, CRLF, the bytes, CRLF; a zero-size chunk ends the body.
A hand-written encoder and decoder (Uvicorn produces this automatically in 6.1.6):

In [18]:
def encode_chunked(parts: Iterable[bytes]) -> bytes:
    """Encode parts as an HTTP/1.1 chunked body (size in hex, CRLF, data, CRLF)."""
    encoded = b"".join(b"%x\r\n%s\r\n" % (len(part), part) for part in parts if part)
    return encoded + b"0\r\n\r\n"


def decode_chunked(data: bytes) -> bytes:
    """Decode a complete chunked body back into the original bytes."""
    decoded, position = b"", 0
    while True:
        line_end = data.index(b"\r\n", position)
        size = int(data[position:line_end], 16)
        if size == 0:
            return decoded
        decoded += data[line_end + 2 : line_end + 2 + size]
        position = line_end + 2 + size + 2  # skip the data and its CRLF


chunked_body = encode_chunked([b"first line\n", b"second, longer line\n"])
print(f"{chunked_body=}")
print(f"{decode_chunked(chunked_body)=}")

chunked_body=b'b\r\nfirst line\n\r\n14\r\nsecond, longer line\n\r\n0\r\n\r\n'
decode_chunked(chunked_body)=b'first line\nsecond, longer line\n'


**Content negotiation.** The client lists what it accepts with quality values
(`q`, default 1); the server picks the best match it can produce, or answers
**406 Not Acceptable**. A small negotiator:

In [19]:
def negotiate(accept_header: str, available: list[str]) -> str | None:
    """Pick the available media type the Accept header prefers most (None → 406)."""
    preferences: list[tuple[float, str]] = []
    for item in accept_header.split(","):
        media_type, *parameters = (piece.strip() for piece in item.split(";"))
        quality = 1.0
        for parameter in parameters:
            name, _, value = parameter.partition("=")
            if name.strip() == "q":
                quality = float(value)
        preferences.append((quality, media_type))
    for quality, media_type in sorted(preferences, key=lambda pair: -pair[0]):
        for candidate in available:
            pattern = media_type.replace("*", ".*")
            if quality > 0 and re.fullmatch(pattern, candidate):
                return candidate
    return None


produced = ["application/json", "text/csv"]
for accept in ("text/csv;q=0.9, application/json", "text/*", "*/*", "image/png"):
    print(f"{accept!r:<40} -> {negotiate(accept, produced)}")

'text/csv;q=0.9, application/json'       -> application/json
'text/*'                                 -> text/csv
'*/*'                                    -> application/json
'image/png'                              -> None


#### HTTP/2 and HTTP/3

| | HTTP/1.1 | HTTP/2 | HTTP/3 |
| --- | --- | --- | --- |
| Wire format | text lines | binary frames | binary frames |
| Transport | TCP | TCP (+ TLS in practice) | **QUIC over UDP** (TLS 1.3 built in) |
| Requests per connection | one at a time (pipelining unused) | many, **multiplexed** as streams | many, and a lost packet stalls only its own stream |
| Headers | repeated as text | compressed (HPACK) | compressed (QPACK) |
| Picked by | default | ALPN `h2` during the TLS handshake | `Alt-Svc` header, then QUIC |

The semantics (methods, status codes, headers) are the same in all three; only
the framing changes, so your FastAPI code never sees the difference. Uvicorn speaks
HTTP/1.1 (and WebSocket); HTTP/2 and HTTP/3 are usually terminated by the proxy in
front (or by servers such as Hypercorn and Granian).

What an HTTP/1.1-only server does with the HTTP/2 *connection preface* (the fixed
bytes an HTTP/2 client sends first when it already knows the server speaks h2):

In [20]:
HTTP2_PREFACE = b"PRI * HTTP/2.0\r\n\r\nSM\r\n\r\n"
preface_reply = raw_http_exchange(SHOP_API, HTTP2_PREFACE)
print(mask_dates(preface_reply.decode("latin-1")).splitlines()[0])

HTTP/1.1 505 Invalid HTTP version (2.0)


### 6.1.3 REST design

REST models an API as **resources** (nouns with URLs) manipulated through the
standard **verbs**. The Notes API (started in Unit 6.2) follows this table:

| Verb | URL | CRUD | Success | Notes |
| --- | --- | --- | --- | --- |
| `GET` | `/notes` | read (list) | 200 | filtering, sorting, pagination in the query string |
| `POST` | `/notes` | create | 201 + `Location: /notes/7` | not idempotent |
| `GET` | `/notes/7` | read (one) | 200 / 404 | |
| `PUT` | `/notes/7` | replace | 200 | send the whole resource |
| `PATCH` | `/notes/7` | update | 200 | send only the changed fields |
| `DELETE` | `/notes/7` | delete | 204 | |
| `GET` | `/notes/7/tags` | sub-resource | 200 | nesting shows ownership; keep it shallow |

Rules of thumb:

- **Plural nouns, no verbs** in paths (`/notes`, not `/getNotes`); actions that do
  not fit CRUD become sub-resources (`POST /notes/7/archive`).
- **Stateless:** every request carries everything needed (auth token, ids); the
  server keeps no per-client conversation state between requests, so any instance
  behind a load balancer can answer.
- **Filtering and sorting** go in the query string:
  `GET /notes?tag=work&pinned=true&sort=-created_at`.
- **Pagination:** `limit` + `offset` (simple, can skip or repeat rows when data
  changes) or a **cursor** (an opaque token for "after this row"; stable and fast).
  Return the total or a `next` link, and/or a `Link` header.
- **Versioning:** in the path (`/v1/notes`, the most common), in a header
  (`Api-Version: 2`), or in the media type (`Accept: application/vnd.notes.v2+json`).
  Add fields freely; remove or rename only in a new version.
- **Idempotency keys:** the client sends `Idempotency-Key: <uuid>` with a `POST`; the
  server stores the first response under that key and replays it for retries, so a
  timed-out payment or note creation is never done twice.

Pagination helpers: an offset `Link` header and an opaque cursor (base64 of JSON,
so clients cannot depend on its contents):

In [21]:
def pagination_links(base_url: str, total: int, limit: int, offset: int) -> str:
    """Build an RFC 8288 Link header with first/prev/next/last for limit/offset paging."""
    last_offset = max(0, (total - 1) // limit * limit)
    pages = {"first": 0, "last": last_offset}
    if offset > 0:
        pages["prev"] = max(0, offset - limit)
    if offset + limit < total:
        pages["next"] = offset + limit
    return ", ".join(
        f'<{base_url}?limit={limit}&offset={page_offset}>; rel="{relation}"'
        for relation, page_offset in pages.items()
    )


def encode_cursor(position: dict[str, object]) -> str:
    """Opaque cursor: URL-safe base64 of compact JSON."""
    return base64.urlsafe_b64encode(json.dumps(position, separators=(",", ":")).encode()).decode()


def decode_cursor(cursor: str) -> dict[str, object]:
    """Inverse of encode_cursor."""
    return json.loads(base64.urlsafe_b64decode(cursor))


print(pagination_links("/notes", total=95, limit=20, offset=40))
cursor = encode_cursor({"created_at": "2026-11-17T09:00:00Z", "id": 41})
print(f"{cursor=}")
print(f"{decode_cursor(cursor)=}")

</notes?limit=20&offset=0>; rel="first", </notes?limit=20&offset=80>; rel="last", </notes?limit=20&offset=20>; rel="prev", </notes?limit=20&offset=60>; rel="next"
cursor='eyJjcmVhdGVkX2F0IjoiMjAyNi0xMS0xN1QwOTowMDowMFoiLCJpZCI6NDF9'
decode_cursor(cursor)={'created_at': '2026-11-17T09:00:00Z', 'id': 41}


An idempotency-key store in a few lines. The first request with a key runs the
handler and remembers its response; a retry with the same key and the same body
gets the stored response (the note is not created twice); the same key with a
*different* body is a client bug, answered with **422**:

In [22]:
@dataclasses.dataclass
class IdempotencyStore:
    """Remember (request fingerprint, response) per Idempotency-Key."""

    responses: dict[str, tuple[str, tuple[int, dict[str, object]]]] = dataclasses.field(
        default_factory=dict
    )

    def run(
        self,
        key: str,
        body: dict[str, object],
        handler: Callable[[], tuple[int, dict[str, object]]],
    ) -> tuple[int, dict[str, object]]:
        """Run handler once per key; replay its response for identical retries."""
        fingerprint = json.dumps(body, sort_keys=True)
        if key in self.responses:
            stored_fingerprint, stored_response = self.responses[key]
            if stored_fingerprint != fingerprint:
                return 422, {"error": "Idempotency-Key reused with a different body"}
            return stored_response
        response = handler()
        self.responses[key] = (fingerprint, response)
        return response


created_notes: list[dict[str, object]] = []


def create_note(body: dict[str, object]) -> tuple[int, dict[str, object]]:
    """Pretend handler: store a note and answer 201."""
    created_notes.append({"id": len(created_notes) + 1, **body})
    return 201, created_notes[-1]


store = IdempotencyStore()
first = store.run("key-7f3a", {"title": "Tea"}, lambda: create_note({"title": "Tea"}))
retry = store.run("key-7f3a", {"title": "Tea"}, lambda: create_note({"title": "Tea"}))
misuse = store.run("key-7f3a", {"title": "Coffee"}, lambda: create_note({"title": "Coffee"}))
print(f"{first=}\n{retry=}\n{misuse=}\n{len(created_notes)=}")

first=(201, {'id': 1, 'title': 'Tea'})
retry=(201, {'id': 1, 'title': 'Tea'})
misuse=(422, {'error': 'Idempotency-Key reused with a different body'})
len(created_notes)=1


#### REST vs RPC vs GraphQL

| Style | Shape | Strengths | Costs | Python |
| --- | --- | --- | --- | --- |
| **REST** | resources + verbs, many URLs | HTTP caching, status codes, simple tooling, OpenAPI | over/under-fetching; several calls for a screen | FastAPI, Django REST framework |
| **RPC** (JSON-RPC, gRPC) | one endpoint (or one per method), "call this function" | natural for actions; gRPC is fast and typed (Protobuf, HTTP/2) | no HTTP caching; errors hidden in bodies | `grpcio`, plain FastAPI `POST /rpc` |
| **GraphQL** | one `POST /graphql`, client sends a query | client picks exactly the fields; one round trip | caching and rate limiting are harder; N+1 queries on the server | Strawberry, Graphene |

The same question, "title of note 7", in the three styles:

In [23]:
rest_request = "GET /notes/7"
rpc_request = {"jsonrpc": "2.0", "method": "notes.get", "params": {"id": 7}, "id": 1}
graphql_request = {"query": "query { note(id: 7) { title } }"}
print(f"REST:    {rest_request}   (the response has every field of note 7)")
print(f"RPC:     POST /rpc     {json.dumps(rpc_request)}")
print(f"GraphQL: POST /graphql {json.dumps(graphql_request)}")

REST:    GET /notes/7   (the response has every field of note 7)
RPC:     POST /rpc     {"jsonrpc": "2.0", "method": "notes.get", "params": {"id": 7}, "id": 1}
GraphQL: POST /graphql {"query": "query { note(id: 7) { title } }"}


### 6.1.4 A raw socket HTTP server

Everything a web server does, by hand, in one module:

1. `accept()` a TCP connection;
2. read bytes until the blank line (`\r\n\r\n`), then `Content-Length` more;
3. **parse** the request line and headers (`parse_head`);
4. **route** on `(method, path)` with `match` (the dict-dispatch idea from Unit
   1.6.14) and produce a JSON payload (`route`);
5. **serialize** status line, headers and body (`build_response`) and `sendall`.

`route` returns how long the request should "wait on I/O" (`/slow?ms=50` stands in
for a database query) instead of sleeping itself, so each server can wait in its
own way: `time.sleep` in the blocking and threaded servers, `await asyncio.sleep`
in the asyncio one. The module ends with the three ways to serve connections:

| Server | Concurrency | Cost per waiting client | Node equivalent |
| --- | --- | --- | --- |
| `serve_blocking` | none: one connection at a time | nothing (it just waits in the backlog) | — |
| `serve_threaded` | one OS thread per connection | a thread (~8 MiB stack reserved) | — |
| `serve_asyncio` | one thread, event loop, `asyncio.start_server` | a Task (~1 KB) | `http.createServer` |

Part 1 of `raw_http.py`: imports and the request-head parser.

In [24]:
%%writefile _scratch/phase6/raw_http.py
"""Hand-written HTTP/1.1 servers on raw sockets: blocking, thread-per-connection, asyncio.

Run one with:  python raw_http.py blocking|threaded|asyncio PORT
"""

import asyncio
import contextlib
import json
import socket
import sys
import threading
import time
from http import HTTPStatus
from urllib.parse import parse_qsl, urlsplit

type RouteResult = tuple[int, object, float]  # status, JSON payload, seconds of simulated I/O


def parse_head(head: bytes) -> tuple[str, str, dict[str, str]]:
    """Split a request head (without the blank line) into method, target and headers."""
    request_line, *header_lines = head.decode("latin-1").split("\r\n")
    method, target, _version = request_line.split(" ", 2)
    headers: dict[str, str] = {}
    for line in header_lines:
        name, _, value = line.partition(":")
        headers[name.strip().lower()] = value.strip()
    return method, target, headers

Overwriting _scratch/phase6/raw_http.py


Part 2: the application (`route`) and the response serializer.

In [25]:
%%writefile -a _scratch/phase6/raw_http.py


def route(method: str, target: str, body: bytes) -> RouteResult:
    """The application: map (method, path) to (status, payload, simulated I/O wait)."""
    url = urlsplit(target)
    query = dict(parse_qsl(url.query))
    match method, url.path:
        case "GET", "/health":
            return 200, {"status": "ok"}, 0.0
        case "GET", "/hello":
            return 200, {"message": f"hello, {query.get('name', 'world')}"}, 0.0
        case "GET", "/slow":  # stands in for a database query or a call to another service
            milliseconds = float(query.get("ms", "50"))
            return 200, {"slept_ms": milliseconds}, milliseconds / 1000
        case "POST", "/echo":
            try:
                return 200, {"received": json.loads(body or b"null")}, 0.0
            except json.JSONDecodeError:
                return 400, {"error": "body is not valid JSON"}, 0.0
        case _:
            return 404, {"error": f"no route {method} {url.path}"}, 0.0


def build_response(status: int, payload: object) -> bytes:
    """Serialize a JSON response: status line, headers, blank line, body."""
    body = json.dumps(payload).encode()
    head = (
        f"HTTP/1.1 {status} {HTTPStatus(status).phrase}\r\n"
        "Content-Type: application/json\r\n"
        f"Content-Length: {len(body)}\r\n"
        "Connection: close\r\n"
        "\r\n"
    )
    return head.encode("latin-1") + body

Appending to _scratch/phase6/raw_http.py


Part 3: read one request from a blocking socket and serve it.

In [26]:
%%writefile -a _scratch/phase6/raw_http.py


def read_request(connection: socket.socket) -> tuple[str, str, bytes] | None:
    """Read one request: bytes up to the blank line, then Content-Length more bytes."""
    data = b""
    while b"\r\n\r\n" not in data:
        chunk = connection.recv(65536)
        if not chunk:
            return None  # the client closed before sending a whole head
        data += chunk
    head, _, body = data.partition(b"\r\n\r\n")
    method, target, headers = parse_head(head)
    content_length = int(headers.get("content-length", "0"))
    while len(body) < content_length:
        chunk = connection.recv(65536)
        if not chunk:
            return None
        body += chunk
    return method, target, body[:content_length]


def handle_connection(connection: socket.socket) -> None:
    """Serve one request on a connection with blocking calls, then close it."""
    with connection:
        request = read_request(connection)
        if request is None:
            return
        status, payload, wait_seconds = route(*request)
        if wait_seconds:
            time.sleep(wait_seconds)  # blocking wait: this thread can do nothing else
        connection.sendall(build_response(status, payload))

Appending to _scratch/phase6/raw_http.py


Part 4: the blocking and threaded servers, and the asyncio handler.

In [27]:
%%writefile -a _scratch/phase6/raw_http.py


def serve_blocking(port: int) -> None:
    """One connection at a time: later clients wait in the kernel's listen backlog."""
    with socket.create_server(("127.0.0.1", port), backlog=128) as server:
        while True:
            connection, _client_address = server.accept()
            handle_connection(connection)


def serve_threaded(port: int) -> None:
    """A new thread per connection: blocking code, but clients are served concurrently."""
    with socket.create_server(("127.0.0.1", port), backlog=128) as server:
        while True:
            connection, _client_address = server.accept()
            threading.Thread(target=handle_connection, args=(connection,), daemon=True).start()


async def handle_stream(reader: asyncio.StreamReader, writer: asyncio.StreamWriter) -> None:
    """Serve one request with asyncio streams: every wait is an await."""
    try:
        head = await reader.readuntil(b"\r\n\r\n")
        method, target, headers = parse_head(head[:-4])
        body = await reader.readexactly(int(headers.get("content-length", "0")))
        status, payload, wait_seconds = route(method, target, body)
        if wait_seconds:
            await asyncio.sleep(wait_seconds)  # the loop serves other clients meanwhile
        writer.write(build_response(status, payload))
        await writer.drain()
    except asyncio.IncompleteReadError:
        pass  # the client went away
    finally:
        writer.close()
        with contextlib.suppress(ConnectionError):
            await writer.wait_closed()

Appending to _scratch/phase6/raw_http.py


Part 5: the asyncio server and the command-line entry point.

In [28]:
%%writefile -a _scratch/phase6/raw_http.py


async def serve_asyncio(port: int) -> None:
    """One thread, one event loop, any number of connections."""
    server = await asyncio.start_server(handle_stream, "127.0.0.1", port, backlog=128)
    async with server:
        await server.serve_forever()


if __name__ == "__main__":
    mode, port_text = sys.argv[1], sys.argv[2]
    if mode == "asyncio":
        asyncio.run(serve_asyncio(int(port_text)))
    else:
        {"blocking": serve_blocking, "threaded": serve_threaded}[mode](int(port_text))

Appending to _scratch/phase6/raw_http.py


The pieces are plain functions, so they can be called without any socket. Parse a
head, route a request, serialize a response:

In [29]:
raw_http = fresh_import("raw_http")
print(raw_http.parse_head(b"GET /hello?name=Asha HTTP/1.1\r\nHost: x\r\nAccept: */*"))
print(raw_http.route("GET", "/hello?name=Asha", b""))
print(raw_http.route("DELETE", "/hello", b""))
print(raw_http.build_response(201, {"id": 7}))

('GET', '/hello?name=Asha', {'host': 'x', 'accept': '*/*'})
(200, {'message': 'hello, Asha'}, 0.0)
(404, {'error': 'no route DELETE /hello'}, 0.0)
b'HTTP/1.1 201 Created\r\nContent-Type: application/json\r\nContent-Length: 9\r\nConnection: close\r\n\r\n{"id": 7}'


Now as a real server. `background_server` runs `python raw_http.py blocking PORT`
in a child process, waits until `GET /health` answers, and stops it at the end of
the `with` block. Any HTTP client can talk to it, here the stdlib's `http.client`:

In [30]:
def raw_server_command(mode: str, port: int) -> list[str]:
    """Command line that runs one of the raw servers on port."""
    return [sys.executable, str(PHASE6_DIR / "raw_http.py"), mode, str(port)]


port = free_port()
with background_server(raw_server_command("blocking", port), port) as base_url:
    connection = http.client.HTTPConnection("127.0.0.1", port, timeout=5)
    for method, path, body in (
        ("GET", "/hello?name=Ravi", None),
        ("POST", "/echo", b'{"tea": 2}'),
        ("POST", "/echo", b"{not json"),
        ("GET", "/nowhere", None),
    ):
        connection.request(method, path, body=body)
        response = connection.getresponse()
        print(f"{method:<4} {path:<17} -> {response.status} {response.read().decode()}")
        connection.close()  # our server sends Connection: close

GET  /hello?name=Ravi  -> 200 {"message": "hello, Ravi"}
POST /echo             -> 200 {"received": {"tea": 2}}
POST /echo             -> 400 {"error": "body is not valid JSON"}
GET  /nowhere          -> 404 {"error": "no route GET /nowhere"}


#### The same server in Node

For comparison, Node's `http` module with the routes the benchmark uses. Node
parses the request for you and calls the handler; the handler never blocks, because
the only way to wait in Node is a callback or a promise (`setTimeout` here). That is
the asyncio model with the event loop hidden:

| Raw Python server | Node `http` |
| --- | --- |
| `parse_head` + `read_request` | done by Node's C parser (llhttp) before your callback |
| `route` with `match` | `if (method === … && url.pathname === …)` |
| `build_response` | `response.writeHead(status, headers)` + `response.end(body)` |
| `await asyncio.sleep(…)` | `setTimeout(…)` |
| `asyncio.start_server` | `http.createServer(handler).listen(port)` |

In [31]:
%%writefile _scratch/phase6/node_http.mjs
// The same routes with Node's built-in http module, for the benchmark below.
import http from "node:http";

const port = Number(process.argv[2]);

http
  .createServer((request, response) => {
    const url = new URL(request.url, "http://127.0.0.1");
    const send = (status, payload) => {
      const body = JSON.stringify(payload);
      response.writeHead(status, {
        "Content-Type": "application/json",
        "Content-Length": Buffer.byteLength(body),
        Connection: "close",
      });
      response.end(body);
    };
    if (request.method === "GET" && url.pathname === "/health") return send(200, { status: "ok" });
    if (request.method === "GET" && url.pathname === "/slow") {
      const milliseconds = Number(url.searchParams.get("ms") ?? 50);
      return setTimeout(() => send(200, { slept_ms: milliseconds }), milliseconds);
    }
    send(404, { error: `no route ${request.method} ${url.pathname}` });
  })
  .listen(port, "127.0.0.1");

Overwriting _scratch/phase6/node_http.mjs


#### Benchmark: 40 concurrent clients, each waiting 50 ms

Each server gets 40 simultaneous `GET /slow?ms=50` requests from one
`httpx.AsyncClient` (one connection per request, since the servers close them). If
requests are served one after another, that takes 40 × 50 ms = 2 s; if the waits
overlap, about 50 ms plus overhead. Node runs too when `node` is installed. Times
vary by machine; the shape does not:

In [32]:
async def burst(base_url: str, path: str, request_count: int) -> float:
    """Send request_count concurrent GETs; return the wall time in seconds."""
    limits = httpx.Limits(max_connections=request_count)
    async with httpx.AsyncClient(
        base_url=base_url, limits=limits, timeout=30, trust_env=False
    ) as client:
        started = time.perf_counter()
        responses = await asyncio.gather(*(client.get(path) for _ in range(request_count)))
        elapsed = time.perf_counter() - started
    assert all(response.status_code == 200 for response in responses)
    return elapsed


server_commands = {
    mode: lambda port, mode=mode: raw_server_command(mode, port)
    for mode in ("blocking", "threaded", "asyncio")
}
if HAS_NODE:
    server_commands["node http"] = lambda port: [
        "node",
        str(PHASE6_DIR / "node_http.mjs"),
        str(port),
    ]
print(interpreter_label())
for label, make_command in server_commands.items():
    port = free_port()
    with background_server(make_command(port), port) as base_url:
        seconds = await burst(base_url, "/slow?ms=50", request_count=40)
    print(f"{label:<10} 40 requests in {seconds:6.3f} s  ({40 / seconds:7.1f} req/s)")
if not HAS_NODE:
    print("SKIP node http: the node command is not installed")

CPython 3.15.1, default (GIL) build, x86_64


blocking   40 requests in  2.048 s  (   19.5 req/s)
threaded   40 requests in  0.085 s  (  472.1 req/s)


asyncio    40 requests in  0.085 s  (  469.1 req/s)
node http  40 requests in  0.081 s  (  494.1 req/s)


What to notice:

- **blocking** serves one connection at a time: ~2 s. Every other client sits in
  the listen backlog while one request waits.
- **threaded** overlaps the waits: each connection has its own thread, so 40 waits
  happen at once. It scales until thread count, memory and context switches hurt
  (thousands of connections).
- **asyncio** overlaps them on **one thread**: while one request awaits, the loop
  serves the others. This is the model Uvicorn (and Node) use.
- **node** lands in the same range as asyncio: same architecture.

A real server also needs keep-alive, timeouts, request size limits, header
validation, chunked bodies, `Expect: 100-continue`, HTTP pipelining and protection
against slow clients. That is why you run Uvicorn or Gunicorn and only write the
app.

### 6.1.5 WSGI (PEP 3333)

WSGI (2003) split Python web programs into a **server** (parses HTTP, manages
sockets and workers) and an **application** (your code), connected by one
function signature:

```python
def app(environ: dict, start_response: Callable) -> Iterable[bytes]: ...
```

- `environ`: a dict with CGI-style keys (`REQUEST_METHOD`, `PATH_INFO`,
  `QUERY_STRING`, `CONTENT_LENGTH`, `HTTP_<HEADER_NAME>`) plus `wsgi.input`, a
  file-like object for the body.
- `start_response(status, headers)`: call it once with `"200 OK"` and a list of
  `(name, value)` string pairs.
- Return an **iterable of bytes**: the body. The server writes each item.

Flask and Django are WSGI apps; Gunicorn, uWSGI and `wsgiref` are WSGI servers.
The demo module has an app, a middleware that wraps it, and `application`, the
wrapped app a server would load:

Part 1 of `wsgi_demo.py`: imports, type aliases and a JSON helper.

In [33]:
%%writefile _scratch/phase6/wsgi_demo.py
"""A WSGI application and a WSGI middleware written by hand (PEP 3333)."""

import json
import time
import uuid
from collections.abc import Callable, Iterable
from typing import Any
from urllib.parse import parse_qs

type Environ = dict[str, Any]
type StartResponse = Callable[..., Callable[[bytes], object]]
type WSGIApp = Callable[[Environ, StartResponse], Iterable[bytes]]


def json_response(start_response: StartResponse, status: str, payload: object) -> list[bytes]:
    """Call start_response with JSON headers and return the body as a one-item list."""
    body = json.dumps(payload).encode()
    headers = [("Content-Type", "application/json"), ("Content-Length", str(len(body)))]
    start_response(status, headers)
    return [body]

Overwriting _scratch/phase6/wsgi_demo.py


Part 2: the WSGI app; every request detail comes from `environ`.

In [34]:
%%writefile -a _scratch/phase6/wsgi_demo.py


def app(environ: Environ, start_response: StartResponse) -> Iterable[bytes]:
    """Route on REQUEST_METHOD and PATH_INFO; every value comes from the environ dict."""
    method, path = environ["REQUEST_METHOD"], environ.get("PATH_INFO", "/")
    query = parse_qs(environ.get("QUERY_STRING", ""))
    if (method, path) == ("GET", "/health"):
        return json_response(start_response, "200 OK", {"status": "ok"})
    if (method, path) == ("GET", "/hello"):
        name = query.get("name", ["world"])[0]
        request_id = environ.get("app.request_id")
        return json_response(start_response, "200 OK", {"message": f"hello, {name}", "id": request_id})
    if (method, path) == ("GET", "/slow"):
        time.sleep(float(query.get("ms", ["100"])[0]) / 1000)  # blocks this worker thread
        return json_response(start_response, "200 OK", {"slow": True})
    if (method, path) == ("POST", "/echo"):
        length = int(environ.get("CONTENT_LENGTH") or 0)
        raw_body = environ["wsgi.input"].read(length)  # a blocking file-like object
        return json_response(start_response, "200 OK", {"received": json.loads(raw_body or b"null")})
    return json_response(start_response, "404 Not Found", {"error": f"no route {method} {path}"})

Appending to _scratch/phase6/wsgi_demo.py


Part 3: the middleware, and `application`, the app a server loads.

In [35]:
%%writefile -a _scratch/phase6/wsgi_demo.py


class RequestIdMiddleware:
    """Give every request an id (the client's X-Request-ID, or a new one) and echo it back."""

    def __init__(self, app: WSGIApp, make_id: Callable[[], str] = lambda: uuid.uuid4().hex) -> None:
        """Wrap app; make_id creates ids for requests that bring none."""
        self.app = app
        self.make_id = make_id

    def __call__(self, environ: Environ, start_response: StartResponse) -> Iterable[bytes]:
        """Add the id to the environ (for the app) and to the response headers."""
        request_id = environ.get("HTTP_X_REQUEST_ID") or self.make_id()
        environ["app.request_id"] = request_id

        def start_response_with_id(
            status: str, headers: list[tuple[str, str]], exc_info: object = None
        ) -> Callable[[bytes], object]:
            """Call the server's start_response with one more header."""
            return start_response(status, [*headers, ("X-Request-ID", request_id)], exc_info)

        return self.app(environ, start_response_with_id)


application = RequestIdMiddleware(app)  # the object a WSGI server (Gunicorn, uWSGI) loads

Appending to _scratch/phase6/wsgi_demo.py


A WSGI app is just a function, so you can call it directly with an environ and a
`start_response` that records what it was given. `wsgiref.util.setup_testing_defaults`
fills in the required keys:

In [36]:
import wsgiref.util

wsgi_demo = fresh_import("wsgi_demo")


def call_wsgi(
    wsgi_app: Callable[..., Iterable[bytes]],
    path: str,
    query: str = "",
    headers: dict[str, str] | None = None,
) -> tuple[str, list[tuple[str, str]], bytes]:
    """Call a WSGI app in-process; return (status, headers, body)."""
    environ: dict[str, Any] = {"PATH_INFO": path, "QUERY_STRING": query}
    for name, value in (headers or {}).items():
        environ[f"HTTP_{name.upper().replace('-', '_')}"] = value
    wsgiref.util.setup_testing_defaults(environ)
    recorded: dict[str, Any] = {}

    def start_response(
        status: str, response_headers: list[tuple[str, str]], exc_info: object = None
    ) -> Callable[[bytes], None]:
        """Record the status and headers the app sent."""
        recorded.update(status=status, headers=response_headers)
        return lambda data: None  # the legacy write() callable

    body = b"".join(wsgi_app(environ, start_response))
    return recorded["status"], recorded["headers"], body


print(call_wsgi(wsgi_demo.app, "/hello", "name=Meena"))
print(call_wsgi(wsgi_demo.application, "/hello", headers={"X-Request-ID": "req-42"}))
print(call_wsgi(wsgi_demo.app, "/missing")[0])

('200 OK', [('Content-Type', 'application/json'), ('Content-Length', '39')], b'{"message": "hello, Meena", "id": null}')
('200 OK', [('Content-Type', 'application/json'), ('Content-Length', '43'), ('X-Request-ID', 'req-42')], b'{"message": "hello, world", "id": "req-42"}')
404 Not Found


The keys a WSGI app reads most (a request with a header, through the testing
defaults). Note how the `X-Request-ID` header became `HTTP_X_REQUEST_ID`:

In [37]:
sample_environ: dict[str, Any] = {
    "PATH_INFO": "/hello",
    "QUERY_STRING": "name=Meena",
    "HTTP_X_REQUEST_ID": "req-42",
}
wsgiref.util.setup_testing_defaults(sample_environ)
for key in (
    "REQUEST_METHOD",
    "PATH_INFO",
    "QUERY_STRING",
    "SERVER_PROTOCOL",
    "HTTP_HOST",
    "HTTP_X_REQUEST_ID",
    "wsgi.url_scheme",
    "wsgi.input",
):
    value = sample_environ[key]
    print(f"{key:<18} {value if isinstance(value, str) else type(value).__name__ + ' object'!r}")

REQUEST_METHOD     'GET'
PATH_INFO          '/hello'
QUERY_STRING       'name=Meena'
SERVER_PROTOCOL    'HTTP/1.0'
HTTP_HOST          '127.0.0.1'
HTTP_X_REQUEST_ID  'req-42'
wsgi.url_scheme    'http'
wsgi.input         'BytesIO object'


**Middleware** is an object with the same signature that wraps another app.
`RequestIdMiddleware` changes the request (adds `app.request_id` to the environ)
and the response (wraps `start_response` to append a header). Stacking is just
nesting: `GzipMiddleware(RequestIdMiddleware(app))`. Express middleware calls
`next()`; WSGI middleware calls the wrapped app.

Serving it with the stdlib's reference server, `wsgiref.simple_server`, in a
thread (shut down at the end of the cell):

In [38]:
import wsgiref.simple_server


class QuietHandler(wsgiref.simple_server.WSGIRequestHandler):
    """Request handler that does not log every request to stderr."""

    def log_message(self, format: str, *args: object) -> None:
        """Do nothing (the default writes one line per request to stderr)."""


@contextlib.contextmanager
def wsgiref_server(wsgi_app: Callable[..., Iterable[bytes]]) -> Iterator[str]:
    """Serve wsgi_app with wsgiref in a thread; yield the base URL; shut down after."""
    server = wsgiref.simple_server.make_server("127.0.0.1", 0, wsgi_app, handler_class=QuietHandler)
    thread = threading.Thread(target=server.serve_forever, daemon=True)
    thread.start()
    try:
        yield f"http://127.0.0.1:{server.server_port}"
    finally:
        server.shutdown()
        server.server_close()
        thread.join()


with (
    wsgiref_server(wsgi_demo.application) as base_url,
    httpx.Client(base_url=base_url, trust_env=False) as client,
):
    response = client.get("/hello", params={"name": "Asha"}, headers={"X-Request-ID": "abc"})
    print(response.status_code, response.json(), response.headers["x-request-id"])
    print(client.post("/echo", json={"cups": 3}).json())

200 {'message': 'hello, Asha', 'id': 'abc'} abc
{'received': {'cups': 3}}


#### Gunicorn

Gunicorn is the standard production WSGI server on Linux and macOS (not Windows): a
**pre-fork** model with a master process that starts `-w N` worker processes, each
serving requests with its own copy of the app, and restarts workers that die or
hang. `gunicorn -w 4 -b 0.0.0.0:8000 wsgi_demo:application`. With
`-k uvicorn_worker.UvicornWorker` it manages Uvicorn workers for ASGI apps instead.
Gunicorn is not a course dependency; the cell runs it only when it is installed.
Uvicorn can also host a WSGI app (`--interface wsgi`, through a thread-pool
adapter), which is what runs here either way:

In [39]:
if OPTIONAL_PACKAGES["gunicorn"]:
    port = free_port()
    command = [
        sys.executable,
        "-m",
        "gunicorn",
        "-w",
        "2",
        "-b",
        f"127.0.0.1:{port}",
        "--chdir",
        str(PHASE6_DIR),
        "wsgi_demo:application",
    ]
    with background_server(command, port) as base_url:
        print("gunicorn:", httpx.get(f"{base_url}/hello", trust_env=False).json())
else:
    print("SKIP gunicorn: not installed (not a course dependency; `uv add gunicorn` to try it)")
with uvicorn_server("wsgi_demo:application", extra_args=["--interface", "wsgi"]) as base_url:
    print("uvicorn --interface wsgi:", httpx.get(f"{base_url}/hello", trust_env=False).json())

SKIP gunicorn: not installed (not a course dependency; `uv add gunicorn` to try it)
uvicorn --interface wsgi: {'message': 'hello, world', 'id': '915fc237d34944b7b5a327b7c9c5c316'}


#### Limitations of WSGI

- **One request = one blocked worker for its whole life.** The call returns only
  when the response is ready, so a request waiting 100 ms on a database holds a
  thread (or process) for 100 ms. `wsgiref` is single-threaded: five concurrent
  slow requests take five times as long.
- **No WebSockets or long-lived connections.** A chat with 10 000 open sockets would
  need 10 000 threads; WSGI has no way to say "a message arrived".
- **No async.** You cannot `await` inside a WSGI app; `wsgi.input` is a blocking
  file.
- **Request/response only.** No lifespan (startup/shutdown) events; servers invent
  their own hooks.

ASGI (next section) fixes all four. The cell measures the first limitation with
five concurrent `GET /slow?ms=100` requests:

In [40]:
async def concurrent_gets(base_url: str, path: str, request_count: int) -> float:
    """Wall time for request_count concurrent GETs on fresh connections."""
    async with httpx.AsyncClient(base_url=base_url, trust_env=False, timeout=30) as client:
        started = time.perf_counter()
        await asyncio.gather(*(client.get(path) for _ in range(request_count)))
        return time.perf_counter() - started


with wsgiref_server(wsgi_demo.app) as base_url:
    seconds = await asyncio.to_thread(asyncio.run, concurrent_gets(base_url, "/slow?ms=100", 5))
print(f"wsgiref (1 thread), 5 x 100 ms requests: {seconds:.2f} s")

wsgiref (1 thread), 5 x 100 ms requests: 0.52 s


> **Note:** the client runs in its own thread with its own loop
> (`asyncio.to_thread(asyncio.run, …)`) only to keep it independent of the kernel's
> loop; any HTTP client would show the same 0.5 s.

### 6.1.6 ASGI

ASGI (Asynchronous Server Gateway Interface) is WSGI redesigned around
coroutines and **messages**:

```python
async def app(scope: dict, receive: Callable[[], Awaitable[dict]],
              send: Callable[[dict], Awaitable[None]]) -> None: ...
```

- `scope`: what kind of connection this is and its details (`type`, `method`,
  `path`, `query_string` as bytes, `headers` as a list of `(bytes, bytes)` pairs).
- `await receive()`: the next event **from** the client or server.
- `await send(message)`: an event **to** the client.

| Scope `type` | Messages the app receives | Messages the app sends |
| --- | --- | --- |
| `http` | `http.request` (`body`, `more_body`), `http.disconnect` | `http.response.start` (`status`, `headers`), `http.response.body` (`body`, `more_body`) |
| `websocket` | `websocket.connect`, `websocket.receive` (`text`/`bytes`), `websocket.disconnect` | `websocket.accept`, `websocket.send`, `websocket.close` |
| `lifespan` | `lifespan.startup`, `lifespan.shutdown` | `lifespan.startup.complete`, `lifespan.shutdown.complete` (or `.failed`) |

Because the app is a coroutine, a request waiting on I/O costs a suspended task,
not a thread; a WebSocket is just a long-running call that loops over `receive()`.
Starlette, FastAPI, Django (3.0+) and Litestar are ASGI apps; Uvicorn, Hypercorn
and Granian are ASGI servers.

```mermaid
sequenceDiagram
    participant Client
    participant Uvicorn
    participant App as app(scope, receive, send)
    Client->>Uvicorn: GET /stream HTTP/1.1
    Uvicorn->>App: call with scope {type: http, path: /stream}
    App->>Uvicorn: send http.response.start (200, headers)
    App->>Uvicorn: send http.response.body (line 1, more_body=True)
    Uvicorn-->>Client: headers + chunk 1
    App->>Uvicorn: send http.response.body (line 2, more_body=True)
    Uvicorn-->>Client: chunk 2
    App->>Uvicorn: send http.response.body (b"", more_body=False)
    Uvicorn-->>Client: final 0-size chunk
```

<details><summary>ASCII version</summary>

```text
Client ──GET /stream──▶ Uvicorn ──scope──▶ app
                        Uvicorn ◀── http.response.start (200)
Client ◀──headers──────
                        Uvicorn ◀── http.response.body "line 1" more_body=True
Client ◀──chunk 1──────
                        Uvicorn ◀── http.response.body b"" more_body=False
Client ◀──0-size chunk─
```

</details>

The demo app handles all three scope types, streams a response with `more_body`,
reads a request body that may arrive in several messages, and has an ASGI version
of the request-id middleware (it wraps `send` instead of `start_response`):

Part 1 of `asgi_demo.py`: type aliases, shared state, JSON and body helpers.

In [41]:
%%writefile _scratch/phase6/asgi_demo.py
"""An ASGI application written by hand: http, websocket and lifespan scopes, plus middleware."""

import asyncio
import json
import time
import uuid
from collections.abc import Awaitable, Callable, MutableMapping
from typing import Any
from urllib.parse import parse_qs

type Scope = MutableMapping[str, Any]
type Message = MutableMapping[str, Any]
type Receive = Callable[[], Awaitable[Message]]
type Send = Callable[[Message], Awaitable[None]]
type ASGIApp = Callable[[Scope, Receive, Send], Awaitable[None]]

state: dict[str, Any] = {"requests": 0}  # filled in by the lifespan startup event


async def send_json(send: Send, status: int, payload: object) -> None:
    """Send a complete JSON response: one start message and one body message."""
    body = json.dumps(payload).encode()
    headers = [(b"content-type", b"application/json"), (b"content-length", str(len(body)).encode())]
    await send({"type": "http.response.start", "status": status, "headers": headers})
    await send({"type": "http.response.body", "body": body})


async def read_body(receive: Receive) -> tuple[bytes, int]:
    """Collect the request body from http.request messages; return it and the message count."""
    body, message_count, more_body = b"", 0, True
    while more_body:
        message = await receive()
        body += message.get("body", b"")
        message_count += 1
        more_body = message.get("more_body", False)
    return body, message_count

Overwriting _scratch/phase6/asgi_demo.py


Part 2: the lifespan protocol and a streamed response (`more_body`).

In [42]:
%%writefile -a _scratch/phase6/asgi_demo.py


async def lifespan(scope: Scope, receive: Receive, send: Send) -> None:
    """Startup and shutdown hooks: open pools here, close them on shutdown."""
    while True:
        message = await receive()
        if message["type"] == "lifespan.startup":
            state["started_at"] = time.time()
            await send({"type": "lifespan.startup.complete"})
        elif message["type"] == "lifespan.shutdown":
            await send({"type": "lifespan.shutdown.complete"})
            return


async def stream_lines(send: Send, count: int, delay: float) -> None:
    """Stream count lines, one body message each (more_body=True), delay seconds apart."""
    headers = [(b"content-type", b"text/plain; charset=utf-8")]  # no content-length: chunked
    await send({"type": "http.response.start", "status": 200, "headers": headers})
    for number in range(1, count + 1):
        line = f"line {number}\n".encode()
        await send({"type": "http.response.body", "body": line, "more_body": True})
        await asyncio.sleep(delay)
    await send({"type": "http.response.body", "body": b"", "more_body": False})

Appending to _scratch/phase6/asgi_demo.py


Part 3: the HTTP router and the WebSocket echo.

In [43]:
%%writefile -a _scratch/phase6/asgi_demo.py


async def http_app(scope: Scope, receive: Receive, send: Send) -> None:
    """Route plain HTTP requests."""
    state["requests"] += 1
    method, path = scope["method"], scope["path"]
    query = {name: values[0] for name, values in parse_qs(scope["query_string"].decode()).items()}
    if (method, path) == ("GET", "/health"):
        await send_json(send, 200, {"status": "ok"})
    elif (method, path) == ("GET", "/hello"):
        await send_json(send, 200, {"message": f"hello, {query.get('name', 'world')}"})
    elif (method, path) == ("POST", "/echo"):
        body, message_count = await read_body(receive)
        await send_json(send, 200, {"bytes": len(body), "messages": message_count})
    elif (method, path) == ("GET", "/stream"):
        await stream_lines(send, int(query.get("count", "3")), float(query.get("delay", "0.1")))
    elif (method, path) == ("GET", "/stats"):
        lifespan_ran = "started_at" in state
        await send_json(send, 200, {"requests": state["requests"], "lifespan_ran": lifespan_ran})
    else:
        await send_json(send, 404, {"error": f"no route {method} {path}"})


async def websocket_app(scope: Scope, receive: Receive, send: Send) -> None:
    """Echo text frames on /ws; refuse any other path."""
    await receive()  # websocket.connect
    if scope["path"] != "/ws":
        await send({"type": "websocket.close", "code": 1008})
        return
    await send({"type": "websocket.accept"})
    while True:
        message = await receive()
        if message["type"] == "websocket.disconnect":
            return
        await send({"type": "websocket.send", "text": f"echo: {message.get('text')}"})

Appending to _scratch/phase6/asgi_demo.py


Part 4: the entry point that dispatches on `scope['type']`, and the middleware.

In [44]:
%%writefile -a _scratch/phase6/asgi_demo.py


async def app(scope: Scope, receive: Receive, send: Send) -> None:
    """The ASGI entry point: one coroutine function for every kind of connection."""
    match scope["type"]:
        case "lifespan":
            await lifespan(scope, receive, send)
        case "http":
            await http_app(scope, receive, send)
        case "websocket":
            await websocket_app(scope, receive, send)


class RequestIdMiddleware:
    """ASGI version of the WSGI middleware: wrap send to add an X-Request-ID header."""

    def __init__(self, app: ASGIApp, make_id: Callable[[], str] = lambda: uuid.uuid4().hex) -> None:
        """Wrap app; make_id creates ids for requests that bring none."""
        self.app = app
        self.make_id = make_id

    async def __call__(self, scope: Scope, receive: Receive, send: Send) -> None:
        """Pass non-HTTP scopes through; add the header to http.response.start."""
        if scope["type"] != "http":
            await self.app(scope, receive, send)
            return
        request_id = dict(scope["headers"]).get(b"x-request-id", b"").decode() or self.make_id()

        async def send_with_id(message: Message) -> None:
            """Forward every message; add the header to http.response.start."""
            if message["type"] == "http.response.start":
                headers = [*message.get("headers", []), (b"x-request-id", request_id.encode())]
                message = {**message, "headers": headers}
            await send(message)

        await self.app(scope, receive, send_with_id)


application = RequestIdMiddleware(app)  # the object Uvicorn loads

Appending to _scratch/phase6/asgi_demo.py


#### Calling an ASGI app by hand

No server needed: build a `scope`, give it a `receive` that returns prepared
messages and a `send` that records what the app sends. This is what Uvicorn does,
minus the sockets (and what Starlette's `TestClient` and `httpx.ASGITransport`
do for tests):

In [45]:
asgi_demo = fresh_import("asgi_demo")


def http_scope(
    method: str, path: str, query: bytes = b"", headers: Iterable[tuple[bytes, bytes]] = ()
) -> dict[str, Any]:
    """A minimal ASGI http scope, as a server would build it."""
    return {
        "type": "http",
        "asgi": {"version": "3.0"},
        "http_version": "1.1",
        "method": method,
        "scheme": "http",
        "path": path,
        "query_string": query,
        "headers": list(headers),
        "client": ("127.0.0.1", 50000),
        "server": ("127.0.0.1", 8000),
    }

`call_asgi` plays the server: its `receive()` hands out prepared messages in order and its
`send()` records everything the app sends back:

In [46]:
async def call_asgi(
    app: Callable[..., Awaitable[None]],
    scope: dict[str, Any],
    incoming: Iterable[dict[str, Any]] = (),
) -> list[dict[str, Any]]:
    """Run app once: receive() returns the incoming messages in order; return what it sent."""
    pending = list(incoming) or [{"type": "http.request", "body": b"", "more_body": False}]
    sent: list[dict[str, Any]] = []

    async def receive() -> dict[str, Any]:
        """The next prepared message, as a server would deliver it."""
        return pending.pop(0)

    async def send(message: dict[str, Any]) -> None:
        """Record one message the app sent."""
        sent.append(message)

    await app(scope, receive, send)
    return sent


for message in await call_asgi(asgi_demo.app, http_scope("GET", "/hello", b"name=Ravi")):
    print(message)

{'type': 'http.response.start', 'status': 200, 'headers': [(b'content-type', b'application/json'), (b'content-length', b'26')]}
{'type': 'http.response.body', 'body': b'{"message": "hello, Ravi"}'}


A request body in two messages (`more_body=True`, then `False`), as a server
produces for a large or streamed upload; the app's `read_body` loop joins them:

In [47]:
upload = [
    {"type": "http.request", "body": b'{"part": ', "more_body": True},
    {"type": "http.request", "body": b'"two"}', "more_body": False},
]
echo_messages = await call_asgi(asgi_demo.app, http_scope("POST", "/echo"), upload)
print(echo_messages[-1]["body"])

b'{"bytes": 15, "messages": 2}'


A streamed response: one `http.response.start`, then several body messages with
`more_body=True`, then an empty final one. The sleeps are set to 0 here:

In [48]:
for message in await call_asgi(asgi_demo.app, http_scope("GET", "/stream", b"count=3&delay=0")):
    print(message)

{'type': 'http.response.start', 'status': 200, 'headers': [(b'content-type', b'text/plain; charset=utf-8')]}
{'type': 'http.response.body', 'body': b'line 1\n', 'more_body': True}
{'type': 'http.response.body', 'body': b'line 2\n', 'more_body': True}
{'type': 'http.response.body', 'body': b'line 3\n', 'more_body': True}
{'type': 'http.response.body', 'body': b'', 'more_body': False}


The **lifespan** scope: the server sends `lifespan.startup` before accepting any
connection and `lifespan.shutdown` when it stops. This is where an app opens its
database pool and HTTP clients (FastAPI's `lifespan=` parameter, Unit 6.3.10):

In [49]:
lifespan_events = [{"type": "lifespan.startup"}, {"type": "lifespan.shutdown"}]
lifespan_scope = {"type": "lifespan", "asgi": {"version": "3.0"}}
print(await call_asgi(asgi_demo.app, lifespan_scope, lifespan_events))
print(f"{'started_at' in asgi_demo.state=}")

[{'type': 'lifespan.startup.complete'}, {'type': 'lifespan.shutdown.complete'}]
'started_at' in asgi_demo.state=True


The **websocket** scope: connect, two text frames, disconnect. The app accepts,
echoes each frame, and returns when the client leaves:

In [50]:
websocket_scope = {**http_scope("GET", "/ws"), "type": "websocket"}
websocket_events = [
    {"type": "websocket.connect"},
    {"type": "websocket.receive", "text": "hello"},
    {"type": "websocket.receive", "text": "chai?"},
    {"type": "websocket.disconnect", "code": 1000},
]
for message in await call_asgi(asgi_demo.app, websocket_scope, websocket_events):
    print(message)

{'type': 'websocket.accept'}
{'type': 'websocket.send', 'text': 'echo: hello'}
{'type': 'websocket.send', 'text': 'echo: chai?'}


**ASGI middleware** wraps the app the same way WSGI middleware does, but it
intercepts *messages*: `RequestIdMiddleware` replaces `send` with `send_with_id`,
which adds a header to `http.response.start`. It passes `websocket` and `lifespan`
scopes through untouched:

In [51]:
scope_with_id = http_scope("GET", "/hello", headers=[(b"x-request-id", b"req-7")])
start_message = (await call_asgi(asgi_demo.application, scope_with_id))[0]
print(start_message["headers"])

[(b'content-type', b'application/json'), (b'content-length', b'27'), (b'x-request-id', b'req-7')]


#### Uvicorn

Uvicorn is the ASGI server: it accepts connections on an asyncio loop (uvloop
when installed), parses HTTP/1.1 with `httptools` (or pure-Python `h11`), handles
WebSocket upgrades (`websockets` / `wsproto`), turns bytes into scopes and messages,
and runs the lifespan protocol.

| Command | Meaning |
| --- | --- |
| `uvicorn asgi_demo:application` | import `application` from `asgi_demo` and serve it on 127.0.0.1:8000 |
| `--reload` | restart on file changes (development only) |
| `--workers 4` | four processes behind one socket |
| `--host 0.0.0.0 --port 80` | listen on every interface |
| `--proxy-headers`, `--forwarded-allow-ips` | trust `X-Forwarded-*` from a proxy |
| `--interface wsgi` | host a WSGI app (6.1.5) |
| `fastapi dev main.py` / `fastapi run` | FastAPI's CLI: Uvicorn with sensible defaults |

The demo app under Uvicorn, for real. Notice in the output: Uvicorn added a `date`
and `server` header; the middleware added `x-request-id`; the streamed response has
**no `content-length`**, so Uvicorn used `transfer-encoding: chunked`, and the
lines arrive ~0.2 s apart as the app sends them; `/stats` shows the lifespan
startup ran:

In [52]:
with (
    uvicorn_server("asgi_demo:application") as base_url,
    httpx.Client(base_url=base_url, trust_env=False, timeout=10) as client,
):
    hello = client.get("/hello", params={"name": "Asha"}, headers={"x-request-id": "r-1"})
    print(hello.json(), {name: hello.headers[name] for name in ("server", "x-request-id")})
    started = time.perf_counter()
    with client.stream("GET", "/stream", params={"count": 3, "delay": 0.2}) as streamed:
        print(f"{streamed.headers.get('transfer-encoding')=}")
        for line in streamed.iter_lines():
            print(f"  +{time.perf_counter() - started:.1f} s  {line!r}")
    print(client.get("/stats").json())

{'message': 'hello, Asha'} {'server': 'uvicorn', 'x-request-id': 'r-1'}
streamed.headers.get('transfer-encoding')='chunked'
  +0.0 s  'line 1'


  +0.2 s  'line 2'


  +0.4 s  'line 3'


{'requests': 4, 'lifespan_ran': True}


The WebSocket route through Uvicorn, with the `websockets` package's synchronous
client (installed with `fastapi[standard]`; the cell skips without it):

In [53]:
if OPTIONAL_PACKAGES["websockets"]:
    from websockets.sync.client import connect as websocket_connect

    with uvicorn_server("asgi_demo:application") as base_url:
        with websocket_connect(base_url.replace("http://", "ws://") + "/ws") as websocket:
            for text in ("hello", "chai?"):
                websocket.send(text)
                print(websocket.recv(timeout=5))
else:
    print("SKIP websocket client: the websockets package is not installed")

echo: hello
echo: chai?


**Under the hood: how Uvicorn knows what you gave it.** `uvicorn.Config.load()`
inspects the object: a **coroutine function**, or an object whose `__call__` is
one, is ASGI 3; anything else is assumed to be the old two-step ASGI 2 style. It
cannot tell a WSGI function from ASGI 2, which is why `--interface wsgi` exists:

In [54]:
import uvicorn

for label, candidate in (
    ("asgi_demo.app", asgi_demo.app),
    ("asgi_demo.application", asgi_demo.application),
    ("wsgi_demo.app", wsgi_demo.app),
):
    config = uvicorn.Config(app=candidate)
    config.load()
    call = candidate if inspect.isfunction(candidate) else type(candidate).__call__
    print(f"{label:<22} coroutine={inspect.iscoroutinefunction(call)!s:<5} -> {config.interface}")
print(f"{inspect.signature(asgi_demo.app)=}")
print(f"{uvicorn.__version__=}")

asgi_demo.app          coroutine=True  -> asgi3


asgi_demo.application  coroutine=True  -> asgi3


wsgi_demo.app          coroutine=False -> asgi2
inspect.signature(asgi_demo.app)=<Signature (scope: Scope, receive: Receive, send: Send) -> None>
uvicorn.__version__='0.54.0'


### 6.1.7 Framework landscape

| Framework | Interface | Style | Validation / docs | Closest JS relative |
| --- | --- | --- | --- | --- |
| **Flask** (2010) | WSGI (async views run in a thread) | micro: routing + request/response, extensions for the rest | extensions (marshmallow, flask-smorest) | **Express** |
| **Django** (2005) | WSGI and ASGI | batteries included: ORM, migrations, admin, auth, templates, forms | Django REST framework, django-ninja | NestJS + Prisma + AdminJS (Rails-like) |
| **Starlette** (2018) | ASGI | lightweight toolkit: routing, middleware, WebSockets, `TestClient` | none built in | **Koa** / Fastify core |
| **FastAPI** (2018) | ASGI (built on Starlette) | type hints drive validation, serialization, DI and docs | **Pydantic** + automatic **OpenAPI** | Fastify with JSON Schema, NestJS DTOs + zod |
| **Litestar** (2021) | ASGI | opinionated: DI, layered config, msgspec or Pydantic, plugins | msgspec / Pydantic + OpenAPI | **NestJS** |

How to choose: an existing Django shop or an admin-heavy product → Django; a JSON
API with typed contracts → FastAPI (or Litestar); a tiny service or a lot of
legacy WSGI → Flask. All of them run behind the same servers and proxies.

**Express → FastAPI mapping**

| Express | FastAPI |
| --- | --- |
| `const app = express()` | `app = FastAPI()` |
| `app.get("/notes/:id", handler)` | `@app.get("/notes/{note_id}")` on a function |
| `req.params.id` (string) | `note_id: int` parameter (converted and validated) |
| `req.query.limit` | `limit: int = 20` parameter |
| `express.json()` + `req.body` + zod/joi | a Pydantic model parameter |
| `res.status(201).json(note)` | `status_code=201`; `return note` |
| `app.use(middleware)` | `app.add_middleware(...)` or `@app.middleware("http")` |
| `express.Router()` | `APIRouter()` (Unit 6.3) |
| `next(err)` + error middleware | `raise HTTPException(...)`, `@app.exception_handler` |
| `swagger-jsdoc` / `@nestjs/swagger` | built in: `/docs`, `/redoc`, `/openapi.json` |
| Supertest | `TestClient` / `httpx.AsyncClient` (Unit 6.6) |

The same endpoint, `GET /hello?name=…`, in each framework. Starlette and FastAPI
are installed (FastAPI depends on Starlette), so they run through Starlette's
`TestClient`, which calls the ASGI app in-process like `call_asgi` above. Flask,
Django and Litestar are not course dependencies; their code is shown and run only if
installed:

In [55]:
from fastapi import FastAPI
from starlette.applications import Starlette
from starlette.requests import Request
from starlette.responses import JSONResponse
from starlette.routing import Route
from starlette.testclient import TestClient


async def starlette_hello(request: Request) -> JSONResponse:
    """Starlette: you read the query string yourself and build the response."""
    return JSONResponse({"message": f"hello, {request.query_params.get('name', 'world')}"})


starlette_app = Starlette(routes=[Route("/hello", starlette_hello)])

fastapi_app = FastAPI()


@fastapi_app.get("/hello")
async def fastapi_hello(name: str = "world") -> dict[str, str]:
    """FastAPI: the parameter declares the query string; the return value becomes JSON."""
    return {"message": f"hello, {name}"}


for label, asgi_app in (("Starlette", starlette_app), ("FastAPI", fastapi_app)):
    with TestClient(asgi_app) as client:
        print(f"{label:<9}", client.get("/hello", params={"name": "Asha"}).json())

Starlette {'message': 'hello, Asha'}
FastAPI   {'message': 'hello, Asha'}


The others, guarded by `find_spec`. The Express version is printed for comparison:

In [56]:
OTHER_FRAMEWORK_SOURCES = {
    "flask": """
from flask import Flask, request
app = Flask(__name__)

@app.get("/hello")
def hello():
    return {"message": f"hello, {request.args.get('name', 'world')}"}
""",
    "litestar": """
from litestar import Litestar, get

@get("/hello")
async def hello(name: str = "world") -> dict[str, str]:
    return {"message": f"hello, {name}"}

app = Litestar([hello])
""",
    "django": """
# views.py (+ urls.py: path("hello", views.hello))
from django.http import JsonResponse

def hello(request):
    return JsonResponse({"message": f"hello, {request.GET.get('name', 'world')}"})
""",
}
EXPRESS_SOURCE = """
app.get("/hello", (req, res) => {
  res.json({ message: `hello, ${req.query.name ?? "world"}` });
});
"""
for framework, source in OTHER_FRAMEWORK_SOURCES.items():
    status = "installed" if OPTIONAL_PACKAGES[framework] else "SKIP (not installed)"
    print(f"--- {framework}: {status}{source}")
print(f"--- express (Node, for comparison){EXPRESS_SOURCE}")

--- flask: SKIP (not installed)
from flask import Flask, request
app = Flask(__name__)

@app.get("/hello")
def hello():
    return {"message": f"hello, {request.args.get('name', 'world')}"}

--- litestar: SKIP (not installed)
from litestar import Litestar, get

@get("/hello")
async def hello(name: str = "world") -> dict[str, str]:
    return {"message": f"hello, {name}"}

app = Litestar([hello])

--- django: SKIP (not installed)
# views.py (+ urls.py: path("hello", views.hello))
from django.http import JsonResponse

def hello(request):
    return JsonResponse({"message": f"hello, {request.GET.get('name', 'world')}"})

--- express (Node, for comparison)
app.get("/hello", (req, res) => {
  res.json({ message: `hello, ${req.query.name ?? "world"}` });
});



The Flask snippet runs too when Flask is present (through its own test client):

In [57]:
if OPTIONAL_PACKAGES["flask"]:
    flask_namespace: dict[str, Any] = {"__name__": "flask_hello"}
    exec(OTHER_FRAMEWORK_SOURCES["flask"], flask_namespace)
    flask_client = flask_namespace["app"].test_client()
    print("Flask    ", flask_client.get("/hello?name=Asha").get_json())
else:
    print("SKIP Flask: not installed (`uv add flask` to run this cell)")

SKIP Flask: not installed (`uv add flask` to run this cell)


### 6.1.8 HTTP clients

| Client | Sync / async | HTTP/2 | Default timeout | Notes |
| --- | --- | --- | --- | --- |
| `urllib.request` | sync | no | **none** (blocks forever) | stdlib; clumsy API; fine for scripts with no dependencies |
| `requests` | sync | no | **none** | the classic; `Session` for reuse; `HTTPAdapter` + `Retry` |
| `httpx` | **both** (`Client`, `AsyncClient`) | yes (`httpx[http2]`) | 5 s | `requests`-like API; ASGI/WSGI transports for tests; FastAPI's test client builds on it |
| `aiohttp` | async | no | 5 min total | client **and** server; very fast; asyncio only |

Four habits matter more than the library:

1. **Reuse a client** (`requests.Session`, `httpx.Client`): it keeps a connection
   pool, so later requests skip the TCP (and TLS) handshake.
2. **Always set a timeout**, split into connect, read, write and pool waits.
3. **Retry only what is safe to retry** (idempotent methods, connect errors, 502/503/
   504/429), with backoff and jitter (Unit 5.4.9).
4. **Stream large bodies** instead of loading them into memory.

The demos use the local shop API (`SHOP_API`): `/rates`, `/flaky/<key>?fail=N`
(503 for the first N calls), `/slow?seconds=S`, `/connections`, `POST /echo`.

**`urllib.request`**: a `Request` object, `urlopen`, and `HTTPError` for 4xx/5xx
(it raises, unlike the others). `trust_env=False` / `ProxyHandler({})` keep proxy
settings from the environment out of these local demos:

In [58]:
rates_request = urllib.request.Request(
    f"{SHOP_API}/rates?base=INR", headers={"Accept": "application/json"}
)
with NO_PROXY_OPENER.open(rates_request, timeout=5) as response:
    print(response.status, json.load(response)["rates"])

post_request = urllib.request.Request(
    f"{SHOP_API}/echo",
    data=json.dumps({"tea": 2}).encode(),
    method="POST",
    headers={"Content-Type": "application/json"},
)
with NO_PROXY_OPENER.open(post_request, timeout=5) as response:
    print(response.status, json.load(response))
try:
    NO_PROXY_OPENER.open(f"{SHOP_API}/status/404", timeout=5)
except urllib.error.HTTPError as error:
    print(f"{type(error).__name__}: {error.code} {error.reason}")

200 {'USD': 0.012, 'EUR': 0.011, 'GBP': 0.0094}
200 {'received': {'tea': 2}}
HTTPError: 404 Not Found


**`requests`**: a `Session` reuses connections (the shop API counts them), and an
`HTTPAdapter` with `urllib3.Retry` retries 503s (`backoff_factor=0` keeps the demo
fast; use a real backoff in production). `requests` has no default timeout, so
every call passes one:

In [59]:
if OPTIONAL_PACKAGES["requests"]:
    import requests
    from requests.adapters import HTTPAdapter
    from urllib3.util.retry import Retry

    with requests.Session() as session:
        session.trust_env = False
        before = shop_connection_count()
        for _ in range(5):
            session.get(f"{SHOP_API}/rates", timeout=5).raise_for_status()
        new_connections = shop_connection_count() - before - 1
        print(f"requests.Session: 5 requests, {new_connections} new connection(s)")

        retry = Retry(total=3, status_forcelist=[503], backoff_factor=0, allowed_methods={"GET"})
        session.mount("http://", HTTPAdapter(max_retries=retry))
        flaky = session.get(f"{SHOP_API}/flaky/requests-demo?fail=2", timeout=5)
        print(f"with Retry: {flaky.status_code} {flaky.json()}")
else:
    print("SKIP requests: not installed")

requests.Session: 5 requests, 1 new connection(s)
with Retry: 200 {'ok': True, 'attempt': 3}


**`httpx`, sync.** `httpx.Client` with a `base_url`, a split `Timeout`, and
connection reuse. A read timeout of 0.3 s against `/slow?seconds=1` raises
`ReadTimeout`; `raise_for_status()` turns 4xx/5xx into `HTTPStatusError`:

In [60]:
shop_timeout = httpx.Timeout(5.0, read=0.3)  # 5 s for connect/write/pool, 0.3 s per read
with httpx.Client(base_url=SHOP_API, timeout=shop_timeout, trust_env=False) as client:
    before = shop_connection_count()
    rates = [client.get("/rates").json()["rates"]["USD"] for _ in range(5)]
    print(f"httpx.Client: {rates=}, {shop_connection_count() - before - 1} new connection(s)")
    print(raised_message(lambda: client.get("/slow", params={"seconds": 1})))
    print(raised_message(lambda: client.get("/status/503").raise_for_status()).splitlines()[0])
    print(f"{client.get('/rates').http_version=}")

httpx.Client: rates=[0.012, 0.012, 0.012, 0.012, 0.012], 1 new connection(s)


ReadTimeout: timed out
HTTPStatusError: Server error '503 Service Unavailable' for url 'http://127.0.0.1:18046/status/503'
client.get('/rates').http_version='HTTP/1.1'


Without reuse, every top-level `httpx.get` opens (and closes) its own connection:

In [61]:
before = shop_connection_count()
for _ in range(5):
    httpx.get(f"{SHOP_API}/rates", trust_env=False)
print(f"5 x httpx.get: {shop_connection_count() - before - 1} new connection(s)")

5 x httpx.get: 5 new connection(s)


**`httpx`, async.** The same API with `await`. Five 0.3 s requests run
concurrently with `asyncio.gather` and take about 0.3 s, not 1.5 s (the shop
API is threaded, so the server side overlaps too):

In [62]:
async with httpx.AsyncClient(base_url=SHOP_API, timeout=5, trust_env=False) as async_client:
    with Timer("5 concurrent /slow?seconds=0.3"):
        slow_replies = await asyncio.gather(
            *(async_client.get("/slow", params={"seconds": 0.3}) for _ in range(5))
        )
print([reply.status_code for reply in slow_replies])

5 concurrent /slow?seconds=0.3 took 312.34 ms
[200, 200, 200, 200, 200]


**Retries.** `httpx.HTTPTransport(retries=N)` retries only **connection** failures
(refused, DNS), never responses. For 503s, retry yourself (or with `tenacity`,
Unit 4.6). Below: a closed port with transport retries, then a small retry loop with
exponential backoff and full jitter around `/flaky`:

In [63]:
import random

closed_port = free_port()  # nothing listens here
with httpx.Client(transport=httpx.HTTPTransport(retries=2), trust_env=False) as client:
    with Timer("3 connection attempts", report=None) as attempts_timer:
        outcome = raised_error_name(lambda: client.get(f"http://127.0.0.1:{closed_port}/"))
print(f"{outcome} after transport retries ({attempts_timer.elapsed_seconds:.2f} s)")

RETRYABLE_STATUS = {429, 502, 503, 504}


def get_with_retries(
    client: httpx.Client, url: str, attempts: int = 4, base_delay: float = 0.01
) -> httpx.Response:
    """GET url, retrying retryable statuses with exponential backoff and full jitter."""
    rng = random.Random(42)
    for attempt in range(1, attempts + 1):
        response = client.get(url)
        if response.status_code not in RETRYABLE_STATUS or attempt == attempts:
            return response
        time.sleep(rng.uniform(0, base_delay * 2**attempt))
    raise AssertionError("unreachable")


with httpx.Client(base_url=SHOP_API, timeout=5, trust_env=False) as client:
    flaky = get_with_retries(client, "/flaky/httpx-demo?fail=2")
print(f"{flaky.status_code=} {flaky.json()=}")

ConnectError after transport retries (0.50 s)
flaky.status_code=200 flaky.json()={'ok': True, 'attempt': 3}


**Streaming.** `client.stream(...)` returns before the body is read; iterate with
`iter_bytes()`, `iter_text()` or `iter_lines()` and memory stays flat however large
the body is. Uploads stream too: pass a generator as `content=` and httpx sends a
chunked request, which arrives at the ASGI app as several `http.request` messages
(the `/echo` route counts them). The ASGI demo app from 6.1.6 serves both, under
Uvicorn:

In [64]:
def upload_parts() -> Iterator[bytes]:
    """A request body produced piece by piece (a file read in chunks, for example)."""
    for part_number in range(4):
        yield f"part {part_number};".encode() * 1000


with (
    uvicorn_server("asgi_demo:application") as base_url,
    httpx.Client(base_url=base_url, timeout=10, trust_env=False) as client,
):
    with client.stream("GET", "/stream", params={"count": 4, "delay": 0}) as streamed:
        chunk_sizes = [len(chunk) for chunk in streamed.iter_raw()]
    print(f"download: {len(chunk_sizes)} chunk(s), sizes {chunk_sizes}")
    upload_reply = client.post("/echo", content=upload_parts())
    print(f"upload:   {upload_reply.json()}")

download: 4 chunk(s), sizes [7, 7, 7, 7]
upload:   {'bytes': 28000, 'messages': 3}


> **Note:** how many chunks arrive depends on timing: the server and the OS may
> merge small writes. The total bytes do not change.

**HTTP/2 and aiohttp.** `httpx.Client(http2=True)` needs the optional `h2`
package (`uv add "httpx[http2]"`) and an HTTPS server that offers `h2` via ALPN;
against the plain-HTTP shop API it would still use HTTP/1.1. `aiohttp` is an
asyncio-only client (and server) that is not a course dependency. Both cells run
only if the package is installed:

In [65]:
if OPTIONAL_PACKAGES["h2"]:
    with httpx.Client(http2=True, trust_env=False) as client:
        print(f"http2=True against plain HTTP: {client.get(f'{SHOP_API}/rates').http_version}")
else:
    print('SKIP HTTP/2: the h2 package is not installed (`uv add "httpx[http2]"`)')

AIOHTTP_SOURCE = """
async with aiohttp.ClientSession(timeout=aiohttp.ClientTimeout(total=5)) as session:
    async with session.get(f"{SHOP_API}/rates") as response:
        print(response.status, await response.json())
"""
if OPTIONAL_PACKAGES["aiohttp"]:
    import aiohttp

    async with aiohttp.ClientSession(timeout=aiohttp.ClientTimeout(total=5)) as session:
        async with session.get(f"{SHOP_API}/rates") as response:
            print("aiohttp:", response.status, await response.json())
else:
    print(f"SKIP aiohttp: not installed (`uv add aiohttp`). The code would be:{AIOHTTP_SOURCE}")

SKIP HTTP/2: the h2 package is not installed (`uv add "httpx[http2]"`)
SKIP aiohttp: not installed (`uv add aiohttp`). The code would be:
async with aiohttp.ClientSession(timeout=aiohttp.ClientTimeout(total=5)) as session:
    async with session.get(f"{SHOP_API}/rates") as response:
        print(response.status, await response.json())



### Predict the output — Unit 6.1

Three short questions on the details that bite.

**Predict the output.**

```python
raw = b"GET / HTTP/1.1\r\nHost: x\r\n\r\n"
print(raw.split(b"\r\n"))
```

<details><summary>Answer</summary>

`[b'GET / HTTP/1.1', b'Host: x', b'', b'']`. The head ends with **two** CRLFs (one
ends the last header line, one is the blank line), so splitting produces two empty
pieces at the end. Parsers look for `\r\n\r\n` instead of splitting naively.

</details>

Run the next cell to check your prediction.

In [66]:
raw = b"GET / HTTP/1.1\r\nHost: x\r\n\r\n"
print(raw.split(b"\r\n"))

[b'GET / HTTP/1.1', b'Host: x', b'', b'']


**Predict the output.**

```python
from http import HTTPStatus
print(HTTPStatus(422).phrase, HTTPStatus(422).is_client_error)
```

<details><summary>Answer</summary>

`Unprocessable Content True` on Python 3.13+. Older versions say "Unprocessable
Entity" (RFC 4918); 3.13 adopted the RFC 9110 name. 422 is a 4xx, so the client must
change the request; FastAPI uses it for every validation error.

</details>

Run the next cell to check your prediction.

In [67]:
print(HTTPStatus(422).phrase, HTTPStatus(422).is_client_error)

Unprocessable Content True


**Predict the output.**

```python
from wsgiref.validate import validator

def app(environ, start_response):
    start_response("200 OK", [("Content-Type", "text/plain")])
    return "hello"   # a str, not a list of bytes

# call validator(app) with a valid environ ...
```

<details><summary>Answer</summary>

`AssertionError: You should not return a string as your application iterator, ...`.
A WSGI app must return an iterable of **bytes**. A `str` is iterable too (one
character at a time), so without the validator a server would send `h`, `e`, `l`,
... as separate writes, or fail on the first `str`. `wsgiref.validate` catches the
mistake at once: use it in tests of hand-written WSGI code.

</details>

Run the next cell to check your prediction.

In [68]:
from wsgiref.validate import validator


def str_returning_app(environ: dict[str, Any], start_response: Callable[..., object]) -> object:
    """Broken on purpose: returns a str instead of an iterable of bytes."""
    start_response("200 OK", [("Content-Type", "text/plain")])
    return "hello"


validated_environ: dict[str, Any] = {"QUERY_STRING": ""}
wsgiref.util.setup_testing_defaults(validated_environ)
print(raised_message(lambda: validator(str_returning_app)(validated_environ, lambda *args: None)))

AssertionError: You should not return a string as your application iterator, instead return a single-item list containing a bytestring.


### Lab 6.1

Five exercises, from bytes on a socket to a micro-framework. Replace each
`raise NotImplementedError`, then run the check cell; until then the checks print
FAIL, which is expected. Three setup cells define `serve_raw(handler)` (a
thread-per-connection socket server that passes each raw request to
`handler(bytes) -> bytes`), `http_call(...)` (one request with `http.client`) and
`ManualClock` (a fake clock for timing tests). `call_asgi` and `http_scope` from
6.1.6 are reused.

In [69]:
def read_raw_request(connection: socket.socket) -> bytes:
    """Read bytes up to the blank line, then Content-Length more bytes."""
    data = b""
    while b"\r\n\r\n" not in data and (chunk := connection.recv(65536)):
        data += chunk
    if b"\r\n\r\n" not in data:
        return data  # the client closed early; let the handler reject it
    match = re.search(rb"(?im)^content-length:\s*(\d+)", data)
    needed = data.index(b"\r\n\r\n") + 4 + (int(match[1]) if match else 0)
    while len(data) < needed and (chunk := connection.recv(65536)):
        data += chunk
    return data


def serve_one(connection: socket.socket, handler: Callable[[bytes], bytes]) -> None:
    """Answer one connection with handler(raw request); a crashing handler gives a plain 500."""
    with connection:
        connection.settimeout(5)
        raw_request = read_raw_request(connection)
        try:
            reply = handler(raw_request)
        except Exception as error:  # a broken handler must not kill the thread noisily
            reason = f"handler raised {type(error).__name__}".encode()  # plain text, no length
            reply = b"HTTP/1.1 500 Internal Server Error\r\nContent-Type: text/plain\r\n"
            reply += b"\r\n" + reason  # the server closes the connection after it
        connection.sendall(reply)

Setup, part 2: the accept loop (one thread per connection) and `serve_raw`, a context manager
that starts it in a thread, yields the port, and stops it at the end of the `with` block:

In [70]:
def accept_loop(
    server: socket.socket, stopping: threading.Event, handler: Callable[[bytes], bytes]
) -> None:
    """Accept connections until stopping is set, serving each in its own thread."""
    while not stopping.is_set():
        try:
            connection, _address = server.accept()
        except TimeoutError:
            continue
        threading.Thread(target=serve_one, args=(connection, handler), daemon=True).start()


@contextlib.contextmanager
def serve_raw(handler: Callable[[bytes], bytes]) -> Iterator[int]:
    """Serve handler(raw request) -> raw response on 127.0.0.1; yield the port; stop after."""
    server = socket.create_server(("127.0.0.1", 0))
    server.settimeout(0.1)  # so the accept loop notices the stop flag
    stopping = threading.Event()
    thread = threading.Thread(target=accept_loop, args=(server, stopping, handler), daemon=True)
    thread.start()
    try:
        yield server.getsockname()[1]
    finally:
        stopping.set()
        thread.join()
        server.close()

Setup, part 3: `http_call` (one request with `http.client`) and `ManualClock` (a fake clock that
returns the readings you give it):

In [71]:
def http_call(
    port: int,
    method: str,
    path: str,
    body: bytes | None = None,
    headers: dict[str, str] | None = None,
) -> tuple[int, dict[str, str], bytes]:
    """One request with http.client; return (status, lower-cased headers, body)."""
    connection = http.client.HTTPConnection("127.0.0.1", port, timeout=5)
    try:
        connection.request(method, path, body=body, headers=headers or {})
        response = connection.getresponse()
        response_headers = {name.lower(): value for name, value in response.getheaders()}
        return response.status, response_headers, response.read()
    finally:
        connection.close()


class ManualClock:
    """A clock that returns the given readings in order (for deterministic timing tests)."""

    def __init__(self, *readings: float) -> None:
        """Store the readings to hand out."""
        self.readings = list(readings)

    def __call__(self) -> float:
        """Return the next reading."""
        return self.readings.pop(0)

#### Exercise 1 — parse a raw HTTP request into a dataclass

Write `parse_http_request(raw: bytes) -> HttpRequest`. Rules:

- The head ends at the first `\r\n\r\n`; without one raise
  `ValueError("incomplete request")`.
- The request line has exactly three parts `METHOD TARGET VERSION`; the method is
  upper-case letters; the version starts with `HTTP/1.`. Otherwise raise
  `ValueError("bad request line")`.
- `path` is the target without the query; `query` maps each name to the **list**
  of its values (`?tag=a&tag=b` → `{"tag": ["a", "b"]}`), percent-decoded.
- Header names are lower-cased; a line without `:` raises
  `ValueError("bad header line")`; a repeated header joins values with `", "`.
- `body` is exactly `Content-Length` bytes (0 without the header); extra bytes
  are ignored; fewer raise `ValueError("body shorter than Content-Length")`.

Your solution:

In [72]:
@dataclasses.dataclass(frozen=True)
class HttpRequest:
    """A parsed HTTP/1.x request."""

    method: str
    path: str
    query: dict[str, list[str]]
    version: str
    headers: dict[str, str]
    body: bytes


def parse_http_request(raw: bytes) -> HttpRequest:
    """Parse raw request bytes (head + body) into an HttpRequest."""
    raise NotImplementedError

Check exercise 1:

In [73]:
sample_get = (
    b"GET /notes/search?tag=work&tag=home&q=green%20tea HTTP/1.1\r\n"
    b"Host: localhost\r\nAccept: application/json\r\nX-Tag: a\r\nx-tag: b\r\n\r\n"
)
sample_post = b'POST /echo HTTP/1.1\r\nContent-Length: 7\r\n\r\n{"a":1}EXTRA'


def request_line_parts(raw: bytes) -> tuple[str, str, str]:
    """Method, path and version of a parsed request."""
    request = parse_http_request(raw)
    return request.method, request.path, request.version


def parse_error(raw: bytes) -> str:
    """What parse_http_request raises for raw, as "ErrorName: message"."""
    return raised_message(lambda: parse_http_request(raw))

The parsing checks:

In [74]:
run_checks(
    [
        (
            "method, path, version",
            lambda: request_line_parts(sample_get),
            ("GET", "/notes/search", "HTTP/1.1"),
        ),
        (
            "query lists, decoded",
            lambda: parse_http_request(sample_get).query,
            {"tag": ["work", "home"], "q": ["green tea"]},
        ),
        (
            "headers lower-cased, repeats joined",
            lambda: parse_http_request(sample_get).headers,
            {"host": "localhost", "accept": "application/json", "x-tag": "a, b"},
        ),
        ("no body without Content-Length", lambda: parse_http_request(sample_get).body, b""),
        (
            "body is exactly Content-Length bytes",
            lambda: parse_http_request(sample_post).body,
            b'{"a":1}',
        ),
    ]
)

FAIL  method, path, version: raised NotImplementedError: 
FAIL  query lists, decoded: raised NotImplementedError: 
FAIL  headers lower-cased, repeats joined: raised NotImplementedError: 
FAIL  no body without Content-Length: raised NotImplementedError: 
FAIL  body is exactly Content-Length bytes: raised NotImplementedError: 
0/5 checks passed


The error checks:

In [75]:
run_checks(
    [
        (
            "incomplete head",
            lambda: parse_error(b"GET / HTTP/1.1\r\nHost: x\r\n"),
            "ValueError: incomplete request",
        ),
        (
            "bad request line",
            lambda: parse_error(b"get / HTTP/1.1\r\n\r\n"),
            "ValueError: bad request line",
        ),
        (
            "bad version",
            lambda: parse_error(b"GET / SPDY/3\r\n\r\n"),
            "ValueError: bad request line",
        ),
        (
            "bad header line",
            lambda: parse_error(b"GET / HTTP/1.1\r\nNoColon\r\n\r\n"),
            "ValueError: bad header line",
        ),
        (
            "short body",
            lambda: parse_error(b"POST / HTTP/1.1\r\nContent-Length: 9\r\n\r\nabc"),
            "ValueError: body shorter than Content-Length",
        ),
    ]
)

FAIL  incomplete head: expected 'ValueError: incomplete request', got 'NotImplementedError: '
FAIL  bad request line: expected 'ValueError: bad request line', got 'NotImplementedError: '
FAIL  bad version: expected 'ValueError: bad request line', got 'NotImplementedError: '
FAIL  bad header line: expected 'ValueError: bad header line', got 'NotImplementedError: '
FAIL  short body: expected 'ValueError: body shorter than Content-Length', got 'NotImplementedError: '
0/5 checks passed


#### Exercise 2 — a raw socket server with routing, JSON and 404

Write `handle_raw_request(raw: bytes) -> bytes`, the whole application of a raw
server: parse (use your `parse_http_request`), route, and return complete response
bytes. Every response is JSON with `Content-Type: application/json`, a correct
`Content-Length` and `Connection: close`, and a status line
`HTTP/1.1 <code> <phrase>`. Routes over the `LAB_NOTES` dict below:

| Request | Response |
| --- | --- |
| `GET /health` | 200 `{"status": "ok"}` |
| `GET /notes` | 200, the list of notes sorted by id |
| `GET /notes/<id>` | 200 the note; 404 `{"error": "note <id> not found"}` (also for a non-numeric id) |
| `POST /echo` | 200 `{"received": <parsed JSON body>}`; 400 `{"error": "invalid JSON"}` |
| known path, other method | 405 `{"error": "method not allowed"}` and an `Allow` header (`GET` or `POST`) |
| anything else | 404 `{"error": "no route <METHOD> <path>"}` |
| unparseable request | 400 `{"error": "<the ValueError message>"}` |

The check cell serves your function with `serve_raw` and talks to it over a real
socket with `http.client`.

In [76]:
LAB_NOTES = {
    1: {"id": 1, "title": "Buy tea", "pinned": True},
    2: {"id": 2, "title": "Call Ravi", "pinned": False},
}

Your solution:

In [77]:
def handle_raw_request(raw: bytes) -> bytes:
    """Parse, route and answer one raw HTTP request with a JSON response."""
    raise NotImplementedError

Check exercise 2:

In [78]:
def json_call(port: int, method: str, path: str, body: bytes | None = None) -> tuple[int, object]:
    """Status and decoded JSON body of one request (the body text if it is not JSON)."""
    status, _headers, raw_body = http_call(port, method, path, body)
    try:
        return status, json.loads(raw_body)
    except json.JSONDecodeError:
        return status, raw_body.decode(errors="replace")


def raw_status_line(raw_request: bytes) -> bytes:
    """First line of handle_raw_request's answer to raw_request."""
    return handle_raw_request(raw_request).split(b"\r\n", 1)[0]


LAB2_REQUESTS = [  # (method, path and a label after the space, body)
    ("GET", "/health", None),
    ("GET", "/notes", None),
    ("GET", "/notes/2", None),
    ("GET", "/notes/9", None),
    ("GET", "/notes/abc", None),
    ("POST", "/echo", b'{"cups": 2}'),
    ("POST", "/echo bad", b"{oops"),
    ("GET", "/nope", None),
]
with serve_raw(handle_raw_request) as lab_port:
    lab2_answers = {
        path: json_call(lab_port, method, path.split(" ")[0], body)
        for method, path, body in LAB2_REQUESTS
    }
    lab2_delete = http_call(lab_port, "DELETE", "/health")
    lab2_health = http_call(lab_port, "GET", "/health")

The routing checks (the requests above went through a real socket):

In [79]:
run_checks(
    [
        ("GET /health", lambda: lab2_answers["/health"], (200, {"status": "ok"})),
        ("GET /notes sorted", lambda: lab2_answers["/notes"], (200, [LAB_NOTES[1], LAB_NOTES[2]])),
        ("GET /notes/2", lambda: lab2_answers["/notes/2"], (200, LAB_NOTES[2])),
        (
            "GET /notes/9 -> 404",
            lambda: lab2_answers["/notes/9"],
            (404, {"error": "note 9 not found"}),
        ),
        ("GET /notes/abc -> 404", lambda: lab2_answers["/notes/abc"][0], 404),
        ("POST /echo", lambda: lab2_answers["/echo"], (200, {"received": {"cups": 2}})),
        (
            "POST /echo bad JSON -> 400",
            lambda: lab2_answers["/echo bad"],
            (400, {"error": "invalid JSON"}),
        ),
        (
            "unknown route -> 404",
            lambda: lab2_answers["/nope"],
            (404, {"error": "no route GET /nope"}),
        ),
    ]
)

FAIL  GET /health: expected (200, {'status': 'ok'}), got (500, 'handler raised NotImplementedError')
FAIL  GET /notes sorted: expected (200, [{'id': 1, 'title': 'Buy tea', 'pinned': True}, {'id': 2, 'title': 'Call Ravi', 'pinned': False}]), got (500, 'handler raised NotImplementedError')
FAIL  GET /notes/2: expected (200, {'id': 2, 'title': 'Call Ravi', 'pinned': False}), got (500, 'handler raised NotImplementedError')
FAIL  GET /notes/9 -> 404: expected (404, {'error': 'note 9 not found'}), got (500, 'handler raised NotImplementedError')
FAIL  GET /notes/abc -> 404: expected 404, got 500
FAIL  POST /echo: expected (200, {'received': {'cups': 2}}), got (500, 'handler raised NotImplementedError')
FAIL  POST /echo bad JSON -> 400: expected (400, {'error': 'invalid JSON'}), got (500, 'handler raised NotImplementedError')
FAIL  unknown route -> 404: expected (404, {'error': 'no route GET /nope'}), got (500, 'handler raised NotImplementedError')
0/8 checks passed


The protocol checks (status line, headers, `Allow`):

In [80]:
run_checks(
    [
        (
            "DELETE /health -> 405 + Allow",
            lambda: (lab2_delete[0], lab2_delete[1].get("allow")),
            (405, "GET"),
        ),
        ("JSON content type", lambda: lab2_health[1].get("content-type"), "application/json"),
        ("Connection: close", lambda: lab2_health[1].get("connection"), "close"),
        (
            "Content-Length matches",
            lambda: int(lab2_health[1]["content-length"]) == len(lab2_health[2]),
            True,
        ),
        (
            "status line has the phrase",
            lambda: raw_status_line(b"GET /nope HTTP/1.1\r\n\r\n"),
            b"HTTP/1.1 404 Not Found",
        ),
        (
            "unparseable -> 400",
            lambda: raw_status_line(b"garbage\r\n\r\n"),
            b"HTTP/1.1 400 Bad Request",
        ),
    ]
)

FAIL  DELETE /health -> 405 + Allow: expected (405, 'GET'), got (500, None)
FAIL  JSON content type: expected 'application/json', got 'text/plain'
FAIL  Connection: close: expected 'close', got None
FAIL  Content-Length matches: raised KeyError: 'content-length'
FAIL  status line has the phrase: raised NotImplementedError: 
FAIL  unparseable -> 400: raised NotImplementedError: 
0/6 checks passed


#### Exercise 3 — a WSGI timing middleware

Write `TimingMiddleware(app, clock=time.perf_counter)`, a WSGI middleware that:

- reads `clock()` when the request starts and again when the wrapped app calls
  `start_response`;
- adds a `Server-Timing` header `app;dur=<milliseconds with one decimal>` (for
  readings 1.0 and 1.25: `app;dur=250.0`), keeping the app's status and headers;
- appends the duration in **seconds** to `self.durations`;
- passes `exc_info` through and returns the app's iterable unchanged (so the
  server still calls its `close()`).

The checks call it through `wsgiref.validate.validator`, which rejects protocol
mistakes.

Your solution:

In [81]:
class TimingMiddleware:
    """WSGI middleware that reports app time in a Server-Timing header."""

    def __init__(
        self, app: Callable[..., Iterable[bytes]], clock: Callable[[], float] = time.perf_counter
    ) -> None:
        """Wrap app; clock returns seconds."""
        self.app = app
        self.clock = clock
        self.durations: list[float] = []

    def __call__(
        self, environ: dict[str, Any], start_response: Callable[..., object]
    ) -> Iterable[bytes]:
        """Time the app up to its start_response call and add the header."""
        raise NotImplementedError

Check exercise 3:

In [82]:
from wsgiref.validate import validator


class ClosingBody:
    """A WSGI body iterable that records whether close() was called."""

    closed = False

    def __iter__(self) -> Iterator[bytes]:
        """Yield the one body chunk."""
        yield b"made it"

    def close(self) -> None:
        """Called by the server when it is done with the body."""
        ClosingBody.closed = True


def closing_app(environ: dict[str, Any], start_response: Callable[..., object]) -> ClosingBody:
    """A WSGI app whose body must be closed by the server."""
    start_response("201 Created", [("Content-Type", "text/plain"), ("X-App", "yes")])
    return ClosingBody()

`timed_call` runs a WSGI app through `wsgiref.validate.validator` and closes the body, as a
server would; then the middleware runs with a fake clock (readings 1.0 and 1.25):

In [83]:
def timed_call(app: Callable[..., Iterable[bytes]]) -> tuple[str, list[tuple[str, str]], bytes]:
    """Call app through wsgiref's validator; close the body like a server would."""
    environ: dict[str, Any] = {"QUERY_STRING": "", "PATH_INFO": "/hello", "SCRIPT_NAME": ""}
    wsgiref.util.setup_testing_defaults(environ)
    recorded: dict[str, Any] = {}

    def start_response(
        status: str, headers: list[tuple[str, str]], exc_info: object = None
    ) -> Callable[[bytes], None]:
        """Record what the app sent."""
        recorded.update(status=status, headers=headers)
        return lambda data: None

    body_iterable = validator(app)(environ, start_response)
    body = b"".join(body_iterable)
    body_iterable.close()
    return recorded["status"], recorded["headers"], body


ClosingBody.closed = False
timed = TimingMiddleware(closing_app, clock=ManualClock(1.0, 1.25))
timed_result: tuple[str, list[tuple[str, str]], bytes] = ("not run", [], b"")
try:
    timed_result = timed_call(timed)
except Exception as error:
    timed_result = ("raised", [], type(error).__name__.encode())

Check exercise 3:

In [84]:
def real_clock_timing() -> bool:
    """True if the middleware around wsgi_demo.app adds a well-formed header."""
    headers = dict(timed_call(TimingMiddleware(wsgi_demo.app))[1])
    return re.fullmatch(r"app;dur=\d+\.\d", headers.get("Server-Timing", "")) is not None


run_checks(
    [
        ("status kept", lambda: timed_result[0], "201 Created"),
        (
            "Server-Timing header",
            lambda: dict(timed_result[1]).get("Server-Timing"),
            "app;dur=250.0",
        ),
        ("app headers kept", lambda: dict(timed_result[1]).get("X-App"), "yes"),
        ("body unchanged", lambda: timed_result[2], b"made it"),
        ("duration recorded in seconds", lambda: timed.durations, [0.25]),
        ("close() reaches the app's iterable", lambda: ClosingBody.closed, True),
        ("works with the real clock", real_clock_timing, True),
    ]
)

FAIL  status kept: expected '201 Created', got 'raised'
FAIL  Server-Timing header: expected 'app;dur=250.0', got None
FAIL  app headers kept: expected 'yes', got None
FAIL  body unchanged: expected b'made it', got b'NotImplementedError'
FAIL  duration recorded in seconds: expected [0.25], got []
FAIL  close() reaches the app's iterable: expected True, got False
FAIL  works with the real clock: raised NotImplementedError: 
0/7 checks passed


#### Exercise 4 — a raw ASGI app with a streaming endpoint

Write `countdown_app(scope, receive, send)`, a raw ASGI app (no framework):

- **lifespan:** answer `lifespan.startup` with `lifespan.startup.complete` and
  `lifespan.shutdown` with `lifespan.shutdown.complete`, then return.
- **`GET /countdown?start=N`** (N an integer from 1 to 10, default 3): status 200,
  header `content-type: text/plain; charset=utf-8`, then **one body message per
  line** with `more_body=True`: `"N\n"`, …, `"1\n"`, `"liftoff\n"`, and finally an
  empty body with `more_body=False`. Await `asyncio.sleep(COUNTDOWN_TICK)` after
  each line.
- Bad `start` (not an integer, or out of range): status 400, the same content type,
  and a single body message `b"start must be an integer from 1 to 10\n"`.
- Anything else: status 404 and a single body `b"not found\n"`.

Your solution:

In [85]:
COUNTDOWN_TICK = 0.0  # seconds between lines; set to 0.5 to watch it under Uvicorn


async def countdown_app(
    scope: dict[str, Any],
    receive: Callable[[], Awaitable[dict[str, Any]]],
    send: Callable[[dict[str, Any]], Awaitable[None]],
) -> None:
    """Raw ASGI app: lifespan support and a streaming GET /countdown."""
    raise NotImplementedError

Check exercise 4:

In [86]:
async def countdown_messages(query: bytes, path: str = "/countdown") -> list[dict[str, Any]]:
    """Messages countdown_app sends for one GET (a one-item error list if it raised)."""
    try:
        return await call_asgi(countdown_app, http_scope("GET", path, query))
    except Exception as error:
        return [{"type": f"raised {type(error).__name__}"}]


text_plain = [(b"content-type", b"text/plain; charset=utf-8")]
three = await countdown_messages(b"start=3")
default = await countdown_messages(b"")
bad = await countdown_messages(b"start=99")
not_a_number = await countdown_messages(b"start=ten")
missing = await countdown_messages(b"", path="/elsewhere")
try:
    lifespan_events = [{"type": "lifespan.startup"}, {"type": "lifespan.shutdown"}]
    lifespan_sent = await call_asgi(countdown_app, {"type": "lifespan"}, lifespan_events)
except Exception as error:
    lifespan_sent = f"raised {type(error).__name__}"
try:
    countdown_transport = httpx.ASGITransport(app=countdown_app)
    async with httpx.AsyncClient(transport=countdown_transport, base_url="http://lab") as client:
        countdown_text = (await client.get("/countdown", params={"start": 2})).text
except Exception as error:
    countdown_text = f"raised {type(error).__name__}"

Check exercise 4:

In [87]:
run_checks(
    [
        (
            "start message",
            lambda: three[0],
            {"type": "http.response.start", "status": 200, "headers": text_plain},
        ),
        (
            "one body message per line",
            lambda: [message.get("body") for message in three[1:]],
            [b"3\n", b"2\n", b"1\n", b"liftoff\n", b""],
        ),
        (
            "more_body flags",
            lambda: [message.get("more_body", False) for message in three[1:]],
            [True, True, True, True, False],
        ),
        ("default start is 3", lambda: len(default), 6),
        (
            "out of range -> 400",
            lambda: (bad[0].get("status"), bad[-1].get("body")),
            (400, b"start must be an integer from 1 to 10\n"),
        ),
        ("not a number -> 400", lambda: not_a_number[0].get("status"), 400),
        (
            "unknown path -> 404",
            lambda: (missing[0].get("status"), missing[-1].get("body")),
            (404, b"not found\n"),
        ),
        (
            "lifespan",
            lambda: lifespan_sent,
            [{"type": "lifespan.startup.complete"}, {"type": "lifespan.shutdown.complete"}],
        ),
        ("through httpx.ASGITransport", lambda: countdown_text, "2\n1\nliftoff\n"),
    ]
)

FAIL  start message: expected {'type': 'http.response.start', 'status': 200, 'headers': [(b'content-type', b'text/plain; charset=utf-8')]}, got {'type': 'raised NotImplementedError'}
FAIL  one body message per line: expected [b'3\n', b'2\n', b'1\n', b'liftoff\n', b''], got []
FAIL  more_body flags: expected [True, True, True, True, False], got []
FAIL  default start is 3: expected 6, got 1
FAIL  out of range -> 400: expected (400, b'start must be an integer from 1 to 10\n'), got (None, None)
FAIL  not a number -> 400: expected 400, got None
FAIL  unknown path -> 404: expected (404, b'not found\n'), got (None, None)
FAIL  lifespan: expected [{'type': 'lifespan.startup.complete'}, {'type': 'lifespan.shutdown.complete'}], got 'raised NotImplementedError'
FAIL  through httpx.ASGITransport: expected '2\n1\nliftoff\n', got 'raised NotImplementedError'
0/9 checks passed


#### Exercise 5 — an ASGI micro-framework, compared with FastAPI

Complete `MicroFramework` (aim for about 150 lines with `Request`) so this works:

```python
app = MicroFramework()

@app.route("/notes/{note_id}", methods=["GET"])
async def get_note(request: Request, note_id: int) -> dict:
    return NOTES[note_id]
```

Rules:

- `route(path, methods=("GET",))` registers the handler and returns it unchanged.
  A `{name}` segment matches one path segment; the value is converted with the
  handler's annotation for `name` (`int` → `int(value)`; anything else stays `str`).
  A failed conversion means "no match".
- The handler is called as `await handler(request, **path_params)`. It returns a
  dict or list (→ 200 JSON) or a `(status, payload)` tuple.
- No route matches the path → 404 `{"detail": "Not Found"}`. The path matches but
  not the method → 405 `{"detail": "Method Not Allowed"}` with an `allow` header
  listing the allowed methods (sorted, comma-separated). The handler raises → 500
  `{"detail": "Internal Server Error"}`.
- Every response has `content-type: application/json` and a `content-length`.
- Lifespan events are answered with `.complete`.

`Request` is given: `method`, `path`, `query` (first value per name), `headers`
(lower-cased str dict), `await request.body()`, `await request.json()`.

In [88]:
class Request:
    """What a handler receives: the scope's data plus the body, read on demand."""

    def __init__(
        self, scope: dict[str, Any], receive: Callable[[], Awaitable[dict[str, Any]]]
    ) -> None:
        """Copy the useful parts of the scope."""
        self.method: str = scope["method"]
        self.path: str = scope["path"]
        parsed = urllib.parse.parse_qs(scope["query_string"].decode())
        self.query = {name: values[0] for name, values in parsed.items()}
        self.headers = {name.decode().lower(): value.decode() for name, value in scope["headers"]}
        self._receive = receive

    async def body(self) -> bytes:
        """Read every http.request message and join the bodies."""
        chunks, more_body = [], True
        while more_body:
            message = await self._receive()
            chunks.append(message.get("body", b""))
            more_body = message.get("more_body", False)
        return b"".join(chunks)

    async def json(self) -> Any:
        """The body parsed as JSON."""
        return json.loads(await self.body() or b"null")

Your solution:

In [89]:
type Handler = Callable[..., Awaitable[object]]


class MicroFramework:
    """A tiny ASGI framework: decorator routing, typed path parameters, JSON responses."""

    def __init__(self) -> None:
        """Start with no routes."""
        self.routes: list[tuple[str, frozenset[str], Handler]] = []

    def route(self, path: str, methods: Iterable[str] = ("GET",)) -> Callable[[Handler], Handler]:
        """Decorator: register handler for path and methods."""
        raise NotImplementedError

    async def __call__(
        self,
        scope: dict[str, Any],
        receive: Callable[[], Awaitable[dict[str, Any]]],
        send: Callable[[dict[str, Any]], Awaitable[None]],
    ) -> None:
        """ASGI entry point."""
        raise NotImplementedError

Check exercise 5. The check builds the same small Notes API twice, once on your
framework and once on FastAPI, sends both the same requests, and compares status
codes and JSON. One difference is expected and printed, not checked: FastAPI answers
`GET /notes/abc` with **422** and a detailed validation error, because it validates
path parameters instead of just failing to match them.

In [90]:
from fastapi import FastAPI, HTTPException


def seed_notes() -> dict[int, dict[str, object]]:
    """Fresh copy of the demo data for each app."""
    return {1: {"id": 1, "title": "Buy tea"}, 2: {"id": 2, "title": "Call Ravi"}}


def build_micro_notes() -> MicroFramework:
    """The Notes demo on MicroFramework."""
    notes, micro = seed_notes(), MicroFramework()

    @micro.route("/notes")
    async def list_notes(request: Request) -> list[dict[str, object]]:
        """GET /notes."""
        return list(notes.values())

    @micro.route("/notes", methods=["POST"])
    async def create_note(request: Request) -> tuple[int, dict[str, object]]:
        """POST /notes: 201 and the new note."""
        payload = await request.json()
        note = {"id": max(notes) + 1, "title": payload["title"]}
        notes[note["id"]] = note
        return 201, note

    @micro.route("/notes/{note_id}")
    async def get_note(request: Request, note_id: int) -> object:
        """GET /notes/{note_id}, or a 404 tuple."""
        if note_id not in notes:
            return 404, {"detail": f"note {note_id} not found"}
        return notes[note_id]

    @micro.route("/boom")
    async def boom(request: Request) -> object:
        """A handler with a bug: the framework must answer 500."""
        raise RuntimeError("bug")

    return micro

The FastAPI equivalent, and the comparison:

In [91]:
def build_fastapi_notes() -> FastAPI:
    """The same Notes demo on FastAPI."""
    notes, api = seed_notes(), FastAPI()

    @api.get("/notes")
    async def list_notes() -> list[dict[str, object]]:
        """GET /notes."""
        return list(notes.values())

    @api.post("/notes", status_code=201)
    async def create_note(payload: dict[str, str]) -> dict[str, object]:
        """POST /notes: 201 and the new note."""
        note = {"id": max(notes) + 1, "title": payload["title"]}
        notes[note["id"]] = note
        return note

    @api.get("/notes/{note_id}")
    async def get_note(note_id: int) -> dict[str, object]:
        """GET /notes/{note_id}, or HTTP 404."""
        if note_id not in notes:
            raise HTTPException(status_code=404, detail=f"note {note_id} not found")
        return notes[note_id]

    return api

Send both apps the same seven requests:

In [92]:
COMPARISON_REQUESTS = [
    ("GET", "/notes", None),
    ("GET", "/notes/2", None),
    ("GET", "/notes/99", None),
    ("POST", "/notes", {"title": "Chai"}),
    ("GET", "/notes/3", None),
    ("DELETE", "/notes", None),
    ("GET", "/nope", None),
]


async def responses_from(build_app: Callable[[], Callable[..., Awaitable[None]]]) -> object:
    """(status, json) for COMPARISON_REQUESTS sent to a fresh app, or the error raised."""
    answers = []
    try:
        transport = httpx.ASGITransport(app=build_app(), raise_app_exceptions=False)
        async with httpx.AsyncClient(transport=transport, base_url="http://lab") as client:
            for method, path, body in COMPARISON_REQUESTS:
                reply = await client.request(method, path, json=body)
                answers.append((reply.status_code, reply.json()))
    except Exception as error:
        return f"raised {type(error).__name__}"
    return answers


fastapi_answers = await responses_from(build_fastapi_notes)
micro_answers = await responses_from(build_micro_notes)

Three behaviours that only the micro-framework is checked for, then the checks:

In [93]:
async def micro_extra() -> dict[str, object]:
    """Allow header, crash handling and a non-int path parameter on the micro app."""
    try:
        transport = httpx.ASGITransport(app=build_micro_notes(), raise_app_exceptions=False)
        async with httpx.AsyncClient(transport=transport, base_url="http://lab") as client:
            not_allowed = await client.delete("/notes/1")
            crashed = await client.get("/boom")
            not_int = await client.get("/notes/abc")
    except Exception as error:
        return {"error": f"raised {type(error).__name__}"}
    return {
        "allow": not_allowed.headers.get("allow"),
        "content-type": not_allowed.headers.get("content-type"),
        "crash": (crashed.status_code, crashed.json()),
        "not int": not_int.status_code,
    }


def route_returns_handler() -> bool:
    """True if route() hands back the decorated function unchanged."""
    return MicroFramework().route("/x")(seed_notes) is seed_notes


extra = await micro_extra()
run_checks(
    [
        ("same answers as FastAPI for 7 requests", lambda: micro_answers, fastapi_answers),
        ("405 lists allowed methods", lambda: extra["allow"], "GET"),
        (
            "handler error -> 500 JSON",
            lambda: extra["crash"],
            (500, {"detail": "Internal Server Error"}),
        ),
        ("non-int path parameter -> 404", lambda: extra["not int"], 404),
        ("JSON content type", lambda: extra["content-type"], "application/json"),
        ("route() returns the handler", route_returns_handler, True),
    ]
)

FAIL  same answers as FastAPI for 7 requests: expected [(200, [{'id': 1, 'title': 'Buy tea'}, {'id': 2, 'title': 'Call Ravi'}]), (200, {'id': 2, 'title': 'Call Ravi'}), (404, {'detail': 'note 99 not found'}), (201, {'id': 3, 'title': 'Chai'}), (200, {'id': 3, 'title': 'Chai'}), (405, {'detail': 'Method Not Allowed'}), (404, {'detail': 'Not Found'})], got 'raised NotImplementedError'
FAIL  405 lists allowed methods: raised KeyError: 'allow'
FAIL  handler error -> 500 JSON: raised KeyError: 'crash'
FAIL  non-int path parameter -> 404: raised KeyError: 'not int'
FAIL  JSON content type: raised KeyError: 'content-type'
FAIL  route() returns the handler: raised NotImplementedError: 
0/6 checks passed


The difference FastAPI makes on a bad path parameter, and the size of your framework:

In [94]:
fastapi_transport = httpx.ASGITransport(app=build_fastapi_notes())
async with httpx.AsyncClient(transport=fastapi_transport, base_url="http://lab") as client:
    print(f"FastAPI answers GET /notes/abc with {(await client.get('/notes/abc')).status_code}")
members = [getattr(member, "__func__", member) for member in vars(MicroFramework).values()]
functions = [member for member in members if inspect.isfunction(member)]
method_lines = sum(len(inspect.getsource(function).splitlines()) for function in functions)
print(f"MicroFramework methods: {method_lines} lines (inspect.getsource needs a file for classes)")

FastAPI answers GET /notes/abc with 422
MicroFramework methods: 14 lines (inspect.getsource needs a file for classes)


What the 150 lines leave out, and FastAPI provides: validation of query, header,
cookie and body parameters with clear 422 errors; response models and
serialization; dependency injection; OpenAPI docs; middleware, CORS, sessions;
background tasks; WebSockets; exception handlers; and years of edge cases. The next
unit uses all of it.

### Recap — Unit 6.1

| Concept | One-line takeaway | JS / Node equivalent |
| --- | --- | --- |
| TCP/IP, ports, DNS | TCP is an ordered **byte stream**; HTTP adds its own framing | `net` module |
| TLS | handshake (SNI, ALPN, certificate check), then encrypted records; often ended at a proxy | `tls` module |
| HTTP/1.1 message | request/status line, headers, blank line (CRLF CRLF), body of `Content-Length` bytes or chunked | what `http` parses for you |
| Methods and status codes | safe vs idempotent; 401 who-are-you vs 403 no; 409 conflict; 422 invalid values; 429 slow down | same semantics |
| REST design | plural nouns + verbs; query-string filters; limit/offset or cursor; `/v1`; idempotency keys | same conventions |
| Raw socket server | blocking (serial) vs a thread per connection vs asyncio streams (one thread, many waits) | `http.createServer` ≈ the asyncio version |
| WSGI | `app(environ, start_response) -> Iterable[bytes]`; one worker per in-flight request; Gunicorn | Express on a blocking server (no real equivalent) |
| ASGI | `async app(scope, receive, send)`; http, websocket, lifespan; `more_body` streams; Uvicorn | Node's event loop with a standard app interface |
| Middleware | WSGI wraps `start_response`; ASGI wraps `send` / `receive` | `app.use((req, res, next) => …)` |
| Frameworks | Flask (WSGI micro), Django (full stack), Starlette (ASGI toolkit), FastAPI (types → validation + docs), Litestar | Express, NestJS, Koa, Fastify |
| HTTP clients | reuse a client, always set timeouts, retry only retryable failures, stream big bodies; httpx does sync and async | `fetch` / undici / axios |

**Further reading**

- [`socket`](https://docs.python.org/3/library/socket.html),
  [`ssl`](https://docs.python.org/3/library/ssl.html),
  [`http.client`](https://docs.python.org/3/library/http.client.html),
  [`http` (`HTTPStatus`, `HTTPMethod`)](https://docs.python.org/3/library/http.html),
  [`asyncio` streams](https://docs.python.org/3/library/asyncio-stream.html),
  [`wsgiref`](https://docs.python.org/3/library/wsgiref.html),
  [`urllib.request`](https://docs.python.org/3/library/urllib.request.html)
- [PEP 3333](https://peps.python.org/pep-3333/) (WSGI),
  [PEP 758](https://peps.python.org/pep-0758/) (`except A, B` without brackets)
- [RFC 9110](https://www.rfc-editor.org/rfc/rfc9110) (HTTP semantics),
  [RFC 9112](https://www.rfc-editor.org/rfc/rfc9112) (HTTP/1.1),
  [RFC 8288](https://www.rfc-editor.org/rfc/rfc8288) (`Link` header)
- [ASGI specification](https://asgi.readthedocs.io/), [Uvicorn](https://www.uvicorn.org/),
  [Starlette](https://www.starlette.io/), [HTTPX](https://www.python-httpx.org/),
  [Requests](https://requests.readthedocs.io/), [Gunicorn](https://gunicorn.org/)

## Unit 6.2 — FastAPI Fundamentals and Pydantic v2

**Objectives**

- Explain what FastAPI adds on top of Starlette and Pydantic, and trace a request
  through it.
- Declare path operations with typed path, query and body parameters, including
  enums, path converters, constrained `Annotated` parameters and query-parameter
  models.
- Model data with Pydantic v2: field types and constraints, validators, computed
  fields, serialization options, configuration, strict mode, `TypeAdapter`, custom
  `Annotated` types, and validation errors.
- Design separate Create / Update / Read schemas and implement PATCH correctly.
- Shape responses: response models, status codes, response classes, headers,
  cookies, `HTTPException`.
- Choose between `async def` and `def` handlers, and prove the difference under
  load.
- Use and customize the generated OpenAPI docs, and generate a TypeScript client.
- Explain how FastAPI turns a function signature into a validated endpoint.
- Start the **Notes API** (`projects/notes_api/`) that the rest of Phase 6 grows.

**JS bridge**

| JavaScript / TypeScript | FastAPI + Pydantic | The difference that bites |
| --- | --- | --- |
| Express route + `req.params` / `req.query` | typed function parameters | values arrive **converted and validated**; a bad `int` is a 422, not `NaN` |
| zod schema + `z.infer<typeof Schema>` | a Pydantic model **is** the type | one class gives validation, the static type, serialization and the JSON Schema |
| `schema.parse(data)` / `safeParse` | `Model.model_validate(data)` raises `ValidationError` | errors are a structured list (`errors()`), not one message |
| NestJS DTO + `class-validator` decorators | Pydantic model + `Field(...)` constraints | constraints live in type hints (`Annotated`) |
| `@nestjs/swagger`, `fastify-swagger` | built in: `/docs`, `/redoc`, `/openapi.json` | the docs come from the same types that validate, so they cannot drift |
| Supertest (`request(app).get(...)`) | `TestClient(app).get(...)` | in-process, no port; `httpx.AsyncClient` + `ASGITransport` for async tests |
| `async` everywhere in Node | `async def` **or** `def` handlers | a blocking call inside `async def` freezes every request; `def` runs in a thread pool |

Run the setup cells first. Part 1: imports, the scratch folder, and the Notes API
folder `projects/notes_api/` with `src/` added to `sys.path`, so the kernel can
import the app the notebook writes:

In [95]:
import asyncio
import contextlib
import dataclasses
import importlib
import importlib.util
import inspect
import json
import os
import platform
import re
import shutil
import socket
import subprocess
import sys
import sysconfig
import threading
import time
import typing
import urllib.request
from collections.abc import AsyncIterator, Awaitable, Callable, Iterable, Iterator
from datetime import UTC, date, datetime
from decimal import Decimal
from enum import StrEnum
from pathlib import Path
from types import ModuleType
from typing import Annotated, Any, Literal
from uuid import UUID

import anyio
import fastapi
import httpx
import pydantic
import starlette
import uvicorn
from fastapi import Body, FastAPI, HTTPException, Query, Response, status
from fastapi import Path as PathParam  # fastapi.Path, renamed: pathlib.Path is used for files
from fastapi.testclient import TestClient
from pydantic import (
    BaseModel,
    ConfigDict,
    EmailStr,
    Field,
    HttpUrl,
    TypeAdapter,
    ValidationError,
    computed_field,
    field_validator,
    model_validator,
)

REPO_ROOT = find_repo_root()
scratch_dir = REPO_ROOT / "notebooks" / "_scratch"  # git-ignored
PHASE6_DIR = scratch_dir / "phase6"
PHASE6_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(REPO_ROOT / "notebooks")  # %%writefile cells use paths relative to notebooks/
if str(PHASE6_DIR) not in sys.path:
    sys.path.insert(0, str(PHASE6_DIR))
CHILD_ENV = {**os.environ, "PYTHON_COLORS": "0", "PYTHONUNBUFFERED": "1", "NO_COLOR": "1"}
CHILD_ENV.pop("FORCE_COLOR", None)
HAS_NODE = shutil.which("node") is not None
HAS_UV = shutil.which("uv") is not None

NOTES_API_DIR = REPO_ROOT / "projects" / "notes_api"  # the Notes API's own uv project
NOTES_SRC = NOTES_API_DIR / "src"
for folder in (NOTES_SRC / "notes_api", NOTES_API_DIR / "tests"):
    folder.mkdir(parents=True, exist_ok=True)  # %%writefile does not create folders
if str(NOTES_SRC) not in sys.path:
    sys.path.insert(0, str(NOTES_SRC))

Part 2: the helpers shared with Unit 6.1 (`raised_error_name`, `raised_message`,
`interpreter_label`, `Timer`, `fresh_import`, `mask_dates`):

In [96]:
def raised_error_name(thunk: Callable[[], object]) -> str:
    """Call thunk and return the name of the exception it raised, or "no error"."""
    try:
        thunk()
    except Exception as error:
        return type(error).__name__
    return "no error"


def raised_message(thunk: Callable[[], object]) -> str:
    """Call thunk and return "ErrorName: message" (memory addresses hidden), or "no error"."""
    try:
        thunk()
    except Exception as error:
        return re.sub(r" at 0x[0-9a-f]+", "", f"{type(error).__name__}: {error}")
    return "no error"


def interpreter_label() -> str:
    """Implementation, version and build flavour: print it next to every benchmark."""
    free_threaded = bool(sysconfig.get_config_var("Py_GIL_DISABLED"))
    build = "free-threaded build" if free_threaded else "default (GIL) build"
    implementation = platform.python_implementation()
    return f"{implementation} {platform.python_version()}, {build}, {platform.machine()}"


class Timer:
    """Timing context manager (shared since Unit 1.14): reports on exit, keeps elapsed_seconds."""

    def __init__(self, label: str, report: Callable[[str], None] | None = print) -> None:
        """Remember the label and where to report the elapsed time."""
        self.label = label
        self.report = report
        self.elapsed_seconds = 0.0

    def __enter__(self) -> Timer:
        """Start the clock."""
        self._start = time.perf_counter()
        return self

    def __exit__(self, *exc_info: object) -> None:
        """Stop the clock and report the elapsed milliseconds."""
        self.elapsed_seconds = time.perf_counter() - self._start
        if self.report is not None:
            self.report(f"{self.label} took {self.elapsed_seconds * 1000:.2f} ms")


def fresh_import(module_name: str) -> ModuleType:
    """Import a module saved by a %%writefile cell, always re-reading the file from disk."""
    for cached in (PHASE6_DIR / "__pycache__").glob(f"{module_name}.*.pyc"):
        cached.unlink()
    importlib.invalidate_caches()
    sys.modules.pop(module_name, None)
    return importlib.import_module(module_name)


def mask_dates(text: str) -> str:
    """Replace HTTP Date header values and ephemeral port numbers with stable placeholders."""
    text = re.sub(r"(?im)^(date: ).*$", r"\1<date>", text)
    return re.sub(r"127\.0\.0\.1:\d+", "127.0.0.1:<port>", text)

Part 3: the server helpers from Unit 6.1 (`free_port`, `wait_for_health`,
`background_server`, `uvicorn_server`: start a server in a child process, wait for
`/health`, always stop it), plus two helpers for the Notes API:
`reload_notes_api()` re-imports the app after a `%%writefile` cell changed it, and
`tidy_test_output()` hides timings in pytest output:

In [97]:
NO_PROXY_OPENER = urllib.request.build_opener(urllib.request.ProxyHandler({}))


def free_port() -> int:
    """Ask the OS for a free TCP port on 127.0.0.1 (bind to port 0, read it, release it)."""
    with socket.socket() as probe:
        probe.bind(("127.0.0.1", 0))
        return probe.getsockname()[1]


def wait_for_health(
    health_url: str, process: subprocess.Popen[bytes] | None = None, timeout: float = 15.0
) -> None:
    """Poll health_url until it answers 200; fail fast if the server process exits."""
    deadline = time.monotonic() + timeout
    while time.monotonic() < deadline:
        if process is not None and process.poll() is not None:
            raise RuntimeError(f"server exited early with code {process.returncode}")
        try:
            with NO_PROXY_OPENER.open(health_url, timeout=1) as response:
                if response.status == 200:
                    return
        except OSError:
            time.sleep(0.05)
    raise TimeoutError(f"{health_url} did not become healthy within {timeout} s")


@contextlib.contextmanager
def background_server(
    command: list[str],
    port: int,
    *,
    health_path: str = "/health",
    cwd: Path = PHASE6_DIR,
    extra_env: dict[str, str] | None = None,
    timeout: float = 15.0,
) -> Iterator[str]:
    """Start a server process, wait for its health check, yield its base URL, always stop it."""
    log_path = PHASE6_DIR / f"server_{port}.log"
    with log_path.open("w", encoding="utf-8") as log_file:
        process = subprocess.Popen(
            command,
            stdout=log_file,
            stderr=subprocess.STDOUT,
            cwd=cwd,
            env={**CHILD_ENV, **(extra_env or {})},
        )
    base_url = f"http://127.0.0.1:{port}"
    started = False
    try:
        try:
            wait_for_health(base_url + health_path, process, timeout)
        except RuntimeError, TimeoutError:
            print(log_path.read_text(encoding="utf-8")[-2000:])  # show why it did not start
            raise
        started = True
        yield base_url
    finally:
        process.terminate()
        try:
            process.wait(timeout=5)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait()
        if started:
            log_path.unlink(missing_ok=True)  # keep the log only when startup failed


def uvicorn_server(
    app_spec: str, *, app_dir: Path = PHASE6_DIR, extra_args: Iterable[str] = (), **options: Any
) -> contextlib.AbstractContextManager[str]:
    """Serve "module:attribute" with Uvicorn in a child process (see background_server)."""
    port = free_port()
    host_and_port = ["--host", "127.0.0.1", "--port", str(port)]
    command = [sys.executable, "-m", "uvicorn", app_spec, "--app-dir", str(app_dir), *host_and_port]
    command += ["--log-level", "warning", *extra_args]
    return background_server(command, port, **options)


def reload_notes_api() -> ModuleType:
    """Forget every notes_api module and import notes_api.main again from the files on disk."""
    for cache_folder in NOTES_SRC.rglob("__pycache__"):
        shutil.rmtree(cache_folder)  # a .pyc written in the same second could hide an edit
    for module_name in [name for name in sys.modules if name.split(".")[0] == "notes_api"]:
        del sys.modules[module_name]
    importlib.invalidate_caches()
    return importlib.import_module("notes_api.main")


def validation_problems(thunk: Callable[[], object]) -> list[tuple[str, str]] | str:
    """Run thunk; list (dotted location, error type) for each validation problem, or "no error"."""
    try:
        thunk()
    except ValidationError as error:
        return [(".".join(map(str, problem["loc"])), problem["type"]) for problem in error.errors()]
    return "no error"


def tidy_test_output(text: str) -> str:
    """Hide timings and machine paths in pytest / uv output."""
    text = re.sub(r"\d+(\.\d+)?m?s\b", "<time>", text)
    return text.replace(str(REPO_ROOT), "<repo>").strip()


print(interpreter_label())
print(f"{fastapi.__version__=}  {starlette.__version__=}")
print(f"{pydantic.VERSION=}  {uvicorn.__version__=}")
print(f"{HAS_NODE=}  {HAS_UV=}  Notes API folder: {NOTES_API_DIR.relative_to(REPO_ROOT)}")

CPython 3.15.1, default (GIL) build, x86_64
fastapi.__version__='0.163.1'  starlette.__version__='1.10.0'
pydantic.VERSION='2.14.3'  uvicorn.__version__='0.54.0'
HAS_NODE=True  HAS_UV=True  Notes API folder: projects/notes_api


### 6.2.1 Architecture

FastAPI is a thin layer that joins three things:

| Part | Provides | You saw it in |
| --- | --- | --- |
| **Starlette** | the ASGI app, routing, `Request`/`Response`, middleware, WebSockets, `TestClient` | 6.1.6–6.1.7 |
| **Pydantic v2** | parsing and validating input, serializing output, JSON Schema (its core is written in Rust) | Unit 4.6, 6.2.5 |
| **OpenAPI** | a machine-readable description of every route, generated from the same type hints; served at `/openapi.json` with Swagger UI at `/docs` and ReDoc at `/redoc` | 6.2.9 |

What FastAPI itself adds: reading your function signature to decide where each
parameter comes from (path, query, header, cookie, body), dependency injection
(Unit 6.3), and wiring validation errors to 422 responses.

**Request lifecycle**

```mermaid
flowchart TD
    A["Uvicorn: bytes → ASGI scope + messages"] --> B["Middleware stack<br/>(ServerErrorMiddleware → user middleware → ExceptionMiddleware)"]
    B --> C["Router: match method + path → APIRoute"]
    C --> D["Solve parameters and dependencies<br/>path, query, headers, cookies, body"]
    D -->|"invalid"| E["RequestValidationError → 422 JSON"]
    D -->|"valid"| F["Call your endpoint<br/>async def: on the loop · def: in the thread pool"]
    F -->|"raises HTTPException"| G["exception handler → error JSON"]
    F -->|"returns a value"| H["Validate/serialize with response_model"]
    H --> I["JSONResponse → ASGI messages → Uvicorn → bytes"]
```

<details><summary>ASCII version</summary>

```text
Uvicorn (bytes → scope/messages)
  → middleware stack (ServerErrorMiddleware → yours → ExceptionMiddleware)
    → router: match method + path → APIRoute
      → solve params/dependencies (path, query, header, cookie, body)
          invalid → RequestValidationError → 422
      → call endpoint (async def on the loop; def in the thread pool)
          HTTPException → handler → error JSON
      → response_model validation + serialization
    → JSONResponse → ASGI messages → Uvicorn → bytes
```

</details>

`FastAPI` **is** a Starlette application (a subclass), so everything from 6.1.6
applies: it is an ASGI callable with `(scope, receive, send)`. A new app already
has three routes, the docs:

In [98]:
from starlette.applications import Starlette

hello_app = FastAPI()
print(f"{FastAPI.__mro__[1]=}")
print(f"{issubclass(FastAPI, Starlette)=}  {inspect.iscoroutinefunction(FastAPI.__call__)=}")
for route in hello_app.routes:
    print(f"{type(route).__name__:<6} {route.path}")

FastAPI.__mro__[1]=<class 'starlette.applications.Starlette'>
issubclass(FastAPI, Starlette)=True  inspect.iscoroutinefunction(FastAPI.__call__)=True
Route  /openapi.json
Route  /docs
Route  /docs/oauth2-redirect
Route  /redoc


The middleware stack is built on the first request. Each layer wraps the next,
exactly like the request-id middleware in 6.1.6. Walking the `.app` attributes shows
the onion, from the outside in: Starlette's `ServerErrorMiddleware` (turns crashes
into 500s), FastAPI's `ExceptionTelemetryMiddleware` (built-in OpenTelemetry hooks,
inactive until a tracer or meter provider is configured, Unit 6.7), the user
middleware (`BaseHTTPMiddleware` wraps the decorated function), `ExceptionMiddleware`
(runs exception handlers such as the 422 one), `AsyncExitStackMiddleware` (cleans up
`yield` dependencies, Unit 6.3), and finally the router. The exact layers vary by
version:

In [99]:
from starlette.routing import Router


@hello_app.middleware("http")
async def add_timing_header(
    request: fastapi.Request, call_next: Callable[..., Awaitable[Response]]
) -> Response:
    """A user middleware: runs before and after every request."""
    response = await call_next(request)
    response.headers["x-handled-by"] = "add_timing_header"
    return response


layer: Any = hello_app.build_middleware_stack()
while not isinstance(layer, Router):
    print(type(layer).__name__)
    layer = layer.app
print(type(layer).__name__, "(the router: matches the path, calls the endpoint)")

ServerErrorMiddleware
ExceptionTelemetryMiddleware
BaseHTTPMiddleware
ExceptionMiddleware
AsyncExitStackMiddleware
APIRouter (the router: matches the path, calls the endpoint)


The lifecycle, observed. The endpoint prints when it runs; a valid request goes
through the middleware and the endpoint; an invalid one is rejected with 422
**before** the endpoint is called (no "endpoint called" line):

In [100]:
@hello_app.get("/square/{number}")
async def square(number: int) -> dict[str, int]:
    """Square a number taken from the path."""
    print(f"  endpoint called with {number=} ({type(number).__name__})")
    return {"number": number, "square": number * number}


with TestClient(hello_app) as client:
    print("GET /square/12")
    ok = client.get("/square/12")
    print(f"  -> {ok.status_code} {ok.json()} x-handled-by={ok.headers['x-handled-by']}")
    print("GET /square/twelve")
    bad = client.get("/square/twelve")
    print(f"  -> {bad.status_code} {bad.json()['detail'][0]['msg']}")

GET /square/12
  endpoint called with number=12 (int)
  -> 200 {'number': 12, 'square': 144} x-handled-by=add_timing_header
GET /square/twelve
  -> 422 Input should be a valid integer, unable to parse string as an integer


### 6.2.2 Path operations

A **path operation** is a path plus an HTTP method plus the function that handles
it. There is one decorator per method, and `api_route` for several at once:

| Decorator | Typical use |
| --- | --- |
| `@app.get(path)` | read |
| `@app.post(path)` | create |
| `@app.put(path)` | replace |
| `@app.patch(path)` | partial update |
| `@app.delete(path)` | delete |
| `@app.head(path)`, `@app.options(path)`, `@app.trace(path)` | rarely written by hand (CORS middleware answers OPTIONS preflights, Unit 6.3) |
| `@app.api_route(path, methods=[...])` | one function for several methods |

A small catalog API shows each decorator. `TestClient` calls the ASGI app
in-process (like Supertest), so no server is needed:

In [101]:
catalog_app = FastAPI(title="Catalog demo")
CATALOG: dict[int, str] = {1: "Assam tea", 2: "Clay mug"}


@catalog_app.get("/items")
async def list_items() -> dict[int, str]:
    """GET: read the collection."""
    return CATALOG


@catalog_app.post("/items", status_code=201)
async def add_item(name: str) -> dict[str, object]:
    """POST: create (name comes from the query string here; bodies are 6.2.4)."""
    item_id = max(CATALOG) + 1
    CATALOG[item_id] = name
    return {"id": item_id, "name": name}


@catalog_app.put("/items/{item_id}")
async def replace_item(item_id: int, name: str) -> dict[str, object]:
    """PUT: replace the item at a known URL."""
    CATALOG[item_id] = name
    return {"id": item_id, "name": name}


@catalog_app.delete("/items/{item_id}", status_code=204)
async def delete_item(item_id: int) -> None:
    """DELETE: no body in the response (204)."""
    CATALOG.pop(item_id, None)


@catalog_app.api_route("/ping", methods=["GET", "POST"])
async def ping(request: fastapi.Request) -> dict[str, str]:
    """One function, two methods."""
    return {"method": request.method}

Exercise every route. Two details: JSON object keys are always strings, so the
`int` keys of `CATALOG` arrive as `"2"`, `"3"`; and only declared methods are routed,
so `HEAD` and `PATCH` get **405 Method Not Allowed** (plain Starlette routes answer
`HEAD` for every `GET` route; FastAPI's do not, unless you add `"HEAD"` with
`api_route`):

In [102]:
with TestClient(catalog_app) as client:
    print(client.post("/items", params={"name": "Steel kettle"}).json())
    print(client.put("/items/2", params={"name": "Glass mug"}).json())
    print(client.delete("/items/1").status_code, client.get("/items").json())
    print(client.get("/ping").json(), client.post("/ping").json())
    print(f"{client.head('/items').status_code=}  {client.patch('/items/2').status_code=}")

{'id': 3, 'name': 'Steel kettle'}
{'id': 2, 'name': 'Glass mug'}
204 {'2': 'Glass mug', '3': 'Steel kettle'}
{'method': 'GET'} {'method': 'POST'}
client.head('/items').status_code=405  client.patch('/items/2').status_code=405


#### Typed path parameters

A `{name}` in the path becomes the function parameter with the same name. Its
annotation decides the conversion: `/items/7` gives `item_id == 7` as an `int`;
`/items/seven` is rejected with **422** and a structured error, before your code
runs. Constraints (`ge`, `le`, …) go in `fastapi.Path` (imported here as
`PathParam`, because `pathlib.Path` is also in use):

In [103]:
orders_app = FastAPI()


@orders_app.get("/orders/{order_id}")
async def get_order(order_id: Annotated[int, PathParam(ge=1, le=9999)]) -> dict[str, object]:
    """A path parameter that must be an int from 1 to 9999."""
    return {"order_id": order_id, "type": type(order_id).__name__}


with TestClient(orders_app) as client:
    for path in ("/orders/42", "/orders/forty-two", "/orders/0"):
        response = client.get(path)
        print(f"{path:<20} {response.status_code} {response.json()}")

/orders/42           200 {'order_id': 42, 'type': 'int'}
/orders/forty-two    422 {'detail': [{'type': 'int_parsing', 'loc': ['path', 'order_id'], 'msg': 'Input should be a valid integer, unable to parse string as an integer', 'input': 'forty-two'}]}
/orders/0            422 {'detail': [{'type': 'greater_than_equal', 'loc': ['path', 'order_id'], 'msg': 'Input should be greater than or equal to 1', 'input': '0', 'ctx': {'ge': 1}}]}


#### Enum path parameters

A `StrEnum` (*new in 3.11*) restricts a parameter to fixed values, and the allowed
values appear in the docs as a drop-down. An unknown value is a 422 listing the
choices:

In [104]:
class ReportPeriod(StrEnum):
    """Allowed values for the {period} path parameter."""

    DAY = "day"
    WEEK = "week"
    MONTH = "month"


reports_app = FastAPI()


@reports_app.get("/reports/{period}")
async def get_report(period: ReportPeriod) -> dict[str, object]:
    """period arrives as a ReportPeriod member, not a plain string."""
    return {
        "period": period,
        "is_member": isinstance(period, ReportPeriod),
        "days": {"day": 1, "week": 7, "month": 30}[period],
    }


with TestClient(reports_app) as client:
    print(client.get("/reports/week").json())
    print(client.get("/reports/year").json()["detail"][0]["msg"])

{'period': 'week', 'is_member': True, 'days': 7}
Input should be 'day', 'week' or 'month'


#### Path converters

A plain `{name}` matches one segment (no `/`). Starlette's converters change that:
`{file_path:path}` matches the rest of the path including slashes, and `int`,
`float`, `uuid` converters match only those shapes (FastAPI still validates with
the annotation):

In [105]:
files_app = FastAPI()


@files_app.get("/files/{file_path:path}")
async def read_file_info(file_path: str) -> dict[str, str]:
    """file_path may contain slashes."""
    return {"file_path": file_path}


@files_app.get("/users/{user_id:uuid}")
async def get_user(user_id: UUID) -> dict[str, str]:
    """Only UUID-shaped segments match this route."""
    return {"user_id": str(user_id), "version": f"v{user_id.version}"}


with TestClient(files_app) as client:
    print(client.get("/files/reports/2026/november.csv").json())
    print(client.get("/users/12345678-1234-4678-9234-567812345678").json())
    print(client.get("/users/not-a-uuid").status_code, "(no route matched)")

{'file_path': 'reports/2026/november.csv'}
{'user_id': '12345678-1234-4678-9234-567812345678', 'version': 'v4'}
404 (no route matched)


#### Route ordering

Routes are tried **in the order they were declared**; the first match wins (like
Express). A fixed path such as `/users/me` must therefore come before
`/users/{username}`, or the parameterized route captures it:

In [106]:
def build_users_app(me_first: bool) -> FastAPI:
    """Declare /users/me before or after /users/{username}."""
    users_app = FastAPI()

    async def read_me() -> dict[str, str]:
        """The fixed route."""
        return {"route": "current user"}

    async def read_user(username: str) -> dict[str, str]:
        """The parameterized route."""
        return {"route": f"user {username!r}"}

    routes = [("/users/me", read_me), ("/users/{username}", read_user)]
    for path, endpoint in routes if me_first else reversed(routes):
        users_app.get(path)(endpoint)  # the decorator, called directly
    return users_app


for me_first in (True, False):
    with TestClient(build_users_app(me_first)) as client:
        print(f"{me_first=}: GET /users/me -> {client.get('/users/me').json()}")

me_first=True: GET /users/me -> {'route': 'current user'}
me_first=False: GET /users/me -> {'route': "user 'me'"}


### 6.2.3 Query parameters

A function parameter that is **not** in the path and is a simple type (str, int,
bool, list of those, …) is read from the query string:

| Declaration | Meaning |
| --- | --- |
| `page: int = 1` | optional, default 1, converted to int |
| `q: str \| None = None` | optional, `None` when absent |
| `term: str` | **required** (no default): missing → 422 |
| `archived: bool = False` | `true/1/yes/on` and `false/0/no/off` are accepted |
| `tag: Annotated[list[str], Query()] = []` | repeated keys: `?tag=a&tag=b` → `["a", "b"]` (`Query()` is needed for lists) |
| `limit: Annotated[int, Query(ge=1, le=100)] = 20` | validated range |
| `Query(alias="page-size")` | the URL name differs from the Python name |
| `Query(deprecated=True)`, `Query(include_in_schema=False)` | documentation flags |

One endpoint with all of them:

In [107]:
search_app = FastAPI()


@search_app.get("/search")
async def search(
    term: str,
    q: str | None = None,
    page: int = 1,
    archived: bool = False,
    tag: Annotated[list[str], Query()] = [],  # noqa: B006 (FastAPI copies defaults)
    limit: Annotated[int, Query(ge=1, le=100)] = 20,
    page_size: Annotated[int, Query(alias="page-size", ge=1)] = 10,
) -> dict[str, object]:
    """Echo the parsed query parameters with their Python types."""
    values = {
        "term": term,
        "q": q,
        "page": page,
        "archived": archived,
        "tag": tag,
        "limit": limit,
        "page_size": page_size,
    }
    return {name: f"{value!r}" for name, value in values.items()}

Call it with different query strings. Note `"archived": "True"` from `yes`, the
list from repeated keys, and the 422 errors pointing at the exact parameter (`loc`):

In [108]:
with TestClient(search_app) as client:
    print(client.get("/search", params={"term": "tea"}).json())
    print(client.get("/search?term=tea&archived=yes&tag=green&tag=loose&page-size=50").json())
    for bad_query in ("", "term=tea&limit=500", "term=tea&page=two"):
        detail = client.get(f"/search?{bad_query}").json()["detail"][0]
        print(f"{bad_query!r:<22} -> 422 {detail['loc']} {detail['msg']}")

{'term': "'tea'", 'q': 'None', 'page': '1', 'archived': 'False', 'tag': '[]', 'limit': '20', 'page_size': '10'}
{'term': "'tea'", 'q': 'None', 'page': '1', 'archived': 'True', 'tag': "['green', 'loose']", 'limit': '20', 'page_size': '50'}
''                     -> 422 ['query', 'term'] Field required
'term=tea&limit=500'   -> 422 ['query', 'limit'] Input should be less than or equal to 100
'term=tea&page=two'    -> 422 ['query', 'page'] Input should be a valid integer, unable to parse string as an integer


> **Note:** a mutable default like `= []` is normally a bug in Python (Unit 1.10),
> but FastAPI copies parameter defaults for every request, so it is safe here.
> `Annotated[list[str], Query(default_factory=list)]` avoids the lint warning.

#### Query-parameter models

When an endpoint takes many filters, group them in a Pydantic model and declare it
as `Annotated[Model, Query()]`. Each field becomes a query parameter, the model's
validators run, and `extra="forbid"` rejects unknown parameters (catching typos
like `?limt=5`):

In [109]:
class ProductFilters(BaseModel):
    """Every filter for GET /products, validated together."""

    model_config = ConfigDict(extra="forbid")

    category: str | None = None
    min_price: Annotated[Decimal, Field(ge=0)] = Decimal(0)
    max_price: Annotated[Decimal | None, Field(ge=0)] = None
    in_stock: bool | None = None
    sort: Literal["price", "-price", "name"] = "name"
    limit: Annotated[int, Field(ge=1, le=100)] = 20


products_app = FastAPI()


@products_app.get("/products")
async def list_products(filters: Annotated[ProductFilters, Query()]) -> ProductFilters:
    """Return the parsed filters (a real handler would query with them)."""
    return filters


with TestClient(products_app) as client:
    print(client.get("/products?category=tea&max_price=500&sort=-price").json())
    for bad_query in ("limt=5", "sort=cheapest"):
        detail = client.get(f"/products?{bad_query}").json()["detail"][0]
        print(f"{bad_query!r:<16} -> {detail['type']}: {detail['msg']}")

{'category': 'tea', 'min_price': '0', 'max_price': '500', 'in_stock': None, 'sort': '-price', 'limit': 20}
'limt=5'         -> extra_forbidden: Extra inputs are not permitted
'sort=cheapest'  -> literal_error: Input should be 'price', '-price' or 'name'


### 6.2.4 Request bodies

A parameter annotated with a **Pydantic model** is read from the JSON body. FastAPI
decides each parameter's source with these rules:

| Parameter | Source |
| --- | --- |
| name appears in the path template | path |
| simple type (int, str, bool, list of them, …) | query |
| Pydantic model (or dataclass, `TypedDict`) | body |
| `Annotated[..., Body()]` / `Query()` / `Header()` / `Cookie()` / `PathParam()` | explicit |

So one endpoint can mix all three. `NewOrder` is the body; `shop_id` comes from
the path; `dry_run` from the query:

In [110]:
class NewOrder(BaseModel):
    """The JSON body of POST /shops/{shop_id}/orders."""

    sku: str
    quantity: Annotated[int, Field(gt=0, le=50)]
    gift_note: str | None = None


orders_body_app = FastAPI()


@orders_body_app.post("/shops/{shop_id}/orders", status_code=201)
async def place_order(shop_id: int, order: NewOrder, dry_run: bool = False) -> dict[str, object]:
    """Mix path, body and query parameters."""
    return {
        "shop_id": shop_id,
        "dry_run": dry_run,
        "order": order,
        "order_type": type(order).__name__,
    }


with TestClient(orders_body_app) as client:
    response = client.post("/shops/3/orders?dry_run=true", json={"sku": "TEA-500", "quantity": 2})
    print(response.status_code, response.json())
    invalid = client.post("/shops/3/orders", json={"sku": "TEA-500", "quantity": 0, "extra": 1})
    print(invalid.status_code, [(error["loc"], error["msg"]) for error in invalid.json()["detail"]])
    print(
        client.post(
            "/shops/3/orders", content=b"{not json", headers={"content-type": "application/json"}
        ).json()["detail"][0]["type"]
    )

201 {'shop_id': 3, 'dry_run': True, 'order': {'sku': 'TEA-500', 'quantity': 2, 'gift_note': None}, 'order_type': 'NewOrder'}
422 [(['body', 'quantity'], 'Input should be greater than 0')]
json_invalid


Errors point into the body (`["body", "quantity"]`); malformed JSON gives
`json_invalid`. The unknown `extra` key was **ignored**: models ignore extra keys by
default (6.2.5 shows `extra="forbid"`).

#### Several bodies, and `Body(embed=True)`

With **one** model parameter, the body *is* that model's JSON. With **two or more**
body parameters, FastAPI expects an object keyed by parameter name. A single
scalar can join the body with `Body()`. And `Body(embed=True)` makes a single model
expect `{"order": {...}}` instead of `{...}`, useful when a client wraps payloads:

In [111]:
class Customer(BaseModel):
    """A second body model."""

    name: str
    email: EmailStr


embed_app = FastAPI()


@embed_app.post("/checkout")
async def checkout(
    order: NewOrder, customer: Customer, priority: Annotated[int, Body(ge=1, le=3)] = 1
) -> dict[str, object]:
    """Two models and a scalar: the body is {"order": ..., "customer": ..., "priority": ...}."""
    return {"order": order.sku, "customer": customer.email, "priority": priority}


@embed_app.post("/orders-embedded")
async def place_embedded(order: Annotated[NewOrder, Body(embed=True)]) -> NewOrder:
    """embed=True: the body must be {"order": {...}}."""
    return order


with TestClient(embed_app) as client:
    order_json = {"sku": "MUG-02", "quantity": 1}
    print(
        client.post(
            "/checkout",
            json={
                "order": order_json,
                "customer": {"name": "Asha", "email": "asha@example.com"},
                "priority": 2,
            },
        ).json()
    )
    print(client.post("/orders-embedded", json={"order": order_json}).json())
    print(client.post("/orders-embedded", json=order_json).json()["detail"][0]["loc"])

{'order': 'MUG-02', 'customer': 'asha@example.com', 'priority': 2}
{'sku': 'MUG-02', 'quantity': 1, 'gift_note': None}
['body', 'order']


### 6.2.5 Pydantic v2 models

A Pydantic model is a class whose annotations are a schema. Creating an instance
**parses** the input (converting where it is safe) and **validates** it; the result
is a typed object. The same class serializes back to JSON and describes itself as
JSON Schema. In zod terms: the class is both the schema and `z.infer<…>`.

#### Field types

| Type | Accepts | Becomes |
| --- | --- | --- |
| `EmailStr` | a valid address (needs `email-validator`, installed with FastAPI) | `str`, domain lower-cased |
| `HttpUrl` | `http`/`https` URLs only | `HttpUrl` object (`.host`, `.path`); trailing `/` added to bare hosts |
| `UUID` | UUID strings or objects | `uuid.UUID` |
| `Decimal` | strings, ints, floats | `decimal.Decimal`, exact; serialized to JSON as a **string** |
| `datetime` / `date` | ISO 8601 strings, Unix timestamps | `datetime` (time zone kept) |
| `Literal["a", "b"]` | exactly those values | the value |
| another `BaseModel` | a dict (or model) | a nested model instance |
| `list[T]`, `dict[str, T]`, `T \| None` | containers / optional | validated element by element |

In [112]:
class Address(BaseModel):
    """A nested model."""

    city: str
    pin_code: str


class Supplier(BaseModel):
    """One field of each common type."""

    supplier_id: UUID
    contact_email: EmailStr
    website: HttpUrl
    credit_limit: Decimal
    onboarded_at: datetime
    tier: Literal["gold", "silver", "bronze"]
    address: Address
    tags: list[str] = []


supplier = Supplier.model_validate(
    {
        "supplier_id": "6f1c1a5e-8f3a-4c55-9b1e-1f2d3c4b5a69",
        "contact_email": "Orders@Chai-Traders.IN",
        "website": "https://chai-traders.in",
        "credit_limit": "250000.50",
        "onboarded_at": "2026-11-17T09:30:00+05:30",
        "tier": "gold",
        "address": {"city": "Pune", "pin_code": "411001"},
    }
)
for field_name in Supplier.model_fields:
    value = getattr(supplier, field_name)
    print(f"{field_name:<14} {type(value).__name__:<9} {value!r}")

supplier_id    UUID      UUID('6f1c1a5e-8f3a-4c55-9b1e-1f2d3c4b5a69')
contact_email  str       'Orders@chai-traders.in'
website        HttpUrl   HttpUrl('https://chai-traders.in/')
credit_limit   Decimal   Decimal('250000.50')
onboarded_at   datetime  datetime.datetime(2026, 11, 17, 9, 30, tzinfo=TzInfo(19800))
tier           str       'gold'
address        Address   Address(city='Pune', pin_code='411001')
tags           list      []


Invalid input is reported field by field, all at once (zod's `safeParse` error
list, not just the first problem):

In [113]:
try:
    Supplier.model_validate(
        {
            "supplier_id": "123",
            "contact_email": "not-an-email",
            "website": "ftp://files.example",
            "credit_limit": "lots",
            "onboarded_at": "yesterday",
            "tier": "platinum",
            "address": {"city": "Pune"},
        }
    )
except ValidationError as error:
    print(f"{error.error_count()} errors")
    for problem in error.errors():
        print(f"  {'.'.join(map(str, problem['loc'])):<17} {problem['type']}")

7 errors
  supplier_id       uuid_parsing
  contact_email     value_error
  website           url_scheme
  credit_limit      decimal_parsing
  onboarded_at      datetime_from_date_parsing
  tier              literal_error
  address.pin_code  missing


#### `Field` constraints

`Field(...)` adds constraints and metadata to one field; in `Annotated[...]` form the
constraint travels with the type and can be reused:

| Argument | For | Example |
| --- | --- | --- |
| `min_length`, `max_length` | str, list, dict | `Field(min_length=1, max_length=200)` |
| `gt`, `ge`, `lt`, `le`, `multiple_of` | numbers | `Field(gt=0, le=50)` |
| `max_digits`, `decimal_places` | `Decimal` | `Field(max_digits=10, decimal_places=2)` |
| `pattern` | str (regex) | `Field(pattern=r"^[A-Z]{3}-\d{3}$")` |
| `default`, `default_factory` | any | `Field(default_factory=list)` |
| `alias`, `validation_alias`, `serialization_alias` | names on the wire | `Field(alias="sku-code")` |
| `title`, `description`, `examples`, `deprecated` | docs (6.2.9) | `Field(description="Price in rupees")` |
| `strict`, `frozen`, `exclude`, `repr` | behaviour per field | `Field(exclude=True)` |

In [114]:
type Quantity = Annotated[int, Field(gt=0, le=50)]  # a reusable constrained type


class LineItem(BaseModel):
    """Constraints from Field and from a reusable Annotated alias."""

    sku: Annotated[str, Field(pattern=r"^[A-Z]{3}-\d{3}$", examples=["TEA-500"])]
    quantity: Quantity
    unit_price: Annotated[Decimal, Field(gt=0, max_digits=8, decimal_places=2)]
    notes: Annotated[list[str], Field(default_factory=list, max_length=3)]
    internal_code: Annotated[str, Field(exclude=True)] = "n/a"


print(LineItem(sku="TEA-500", quantity=3, unit_price="240.00").model_dump())
for bad in (
    {"sku": "tea-500"},
    {"quantity": 51},
    {"unit_price": "1.999"},
    {"notes": ["a", "b", "c", "d"]},
):
    data = {"sku": "TEA-500", "quantity": 1, "unit_price": "1.00", **bad}
    message = raised_message(lambda data=data: LineItem(**data)).splitlines()[2].strip()
    print(f"{bad!s:<32} -> {message}")

{'sku': 'TEA-500', 'quantity': 3, 'unit_price': Decimal('240.00'), 'notes': []}
{'sku': 'tea-500'}               -> String should match pattern '^[A-Z]{3}-\d{3}$' [type=string_pattern_mismatch, input_value='tea-500', input_type=str]
{'quantity': 51}                 -> Input should be less than or equal to 50 [type=less_than_equal, input_value=51, input_type=int]
{'unit_price': '1.999'}          -> Decimal input should have no more than 2 decimal places [type=decimal_max_places, input_value='1.999', input_type=str]
{'notes': ['a', 'b', 'c', 'd']}  -> List should have at most 3 items after validation, not 4 [type=too_long, input_value=['a', 'b', 'c', 'd'], input_type=list]


> **Note:** `type Quantity = Annotated[...]` is a PEP 695 type alias (*new in
> 3.12*); Pydantic understands it. `internal_code` has `exclude=True`, so it never
> appears in `model_dump()` output.

#### `field_validator`, `model_validator`, `computed_field`

- `@field_validator("name")` runs for one or more fields. `mode="after"` (default)
  receives the already-parsed value; `mode="before"` receives the raw input (to
  clean it up before parsing). Return the value to keep; raise `ValueError` to
  reject.
- `@model_validator(mode="after")` sees the whole model: **cross-field** rules
  (end after start, passwords match). `mode="before"` gets the raw dict.
- `@computed_field` on a property adds a derived value to serialization and the
  schema (but it is never accepted as input).

In [115]:
class Booking(BaseModel):
    """Validators on fields and on the whole model, plus a computed field."""

    guest_name: str
    room_type: Literal["single", "double", "suite"]
    check_in: date
    check_out: date
    nightly_rate: Decimal

    @field_validator("guest_name")
    @classmethod
    def title_case_name(cls, value: str) -> str:
        """After parsing: normalize spacing and capitalization."""
        return " ".join(part.capitalize() for part in value.split())

    @field_validator("room_type", mode="before")
    @classmethod
    def lowercase_room(cls, value: object) -> object:
        """Before parsing: accept "Double" by lower-casing raw strings first."""
        return value.lower() if isinstance(value, str) else value

    @model_validator(mode="after")
    def check_dates(self) -> Booking:
        """Cross-field rule: at least one night."""
        if self.check_out <= self.check_in:
            raise ValueError("check_out must be after check_in")
        return self

    @computed_field
    @property
    def total(self) -> Decimal:
        """Derived, serialized, never accepted as input."""
        return self.nightly_rate * (self.check_out - self.check_in).days

Valid and invalid bookings; the computed `total` appears in the dump:

In [116]:
booking = Booking(
    guest_name="  asha   rao ",
    room_type="Double",
    check_in="2026-12-01",
    check_out="2026-12-04",
    nightly_rate="3500",
)
print(booking.model_dump())
print(
    raised_message(
        lambda: Booking(
            guest_name="Ravi",
            room_type="suite",
            check_in="2026-12-04",
            check_out="2026-12-01",
            nightly_rate="9000",
        )
    ).splitlines()[1]
)
print(f"{'total' in Booking.model_json_schema(mode='serialization')['properties']=}")

{'guest_name': 'Asha Rao', 'room_type': 'double', 'check_in': datetime.date(2026, 12, 1), 'check_out': datetime.date(2026, 12, 4), 'nightly_rate': Decimal('3500'), 'total': Decimal('10500')}
  Value error, check_out must be after check_in [type=value_error, input_value={'guest_name': 'Ravi', 'r... 'nightly_rate': '9000'}, input_type=dict]
'total' in Booking.model_json_schema(mode='serialization')['properties']=True


#### `model_dump` options, `model_validate` / `model_validate_json`

| Call | Result |
| --- | --- |
| `model_dump()` | dict of Python objects (`Decimal`, `datetime` stay objects) |
| `model_dump(mode="json")` | dict of JSON-safe values (strings for `Decimal`, ISO dates) |
| `model_dump(include={...})` / `exclude={...}` | pick or drop fields (nested sets allowed) |
| `model_dump(exclude_unset=True)` | only fields the input actually set (PATCH, 6.2.6) |
| `model_dump(exclude_defaults=True)` / `exclude_none=True` | drop fields equal to their default / `None` |
| `model_dump(by_alias=True)` | use aliases as keys |
| `model_dump_json(indent=2)` | a JSON **string**, produced in Rust (faster than `json.dumps(model_dump())`) |
| `Model.model_validate(obj)` | validate a dict (or an object with `from_attributes`) |
| `Model.model_validate_json(text)` | parse and validate JSON text in one step (faster than `json.loads` first) |

In [117]:
print(f"{booking.model_dump(include={'guest_name', 'total'})=}")
print(f"{booking.model_dump(mode='json', exclude={'check_in', 'check_out'})=}")
print(booking.model_dump_json(indent=2, include={"guest_name", "nightly_rate", "total"}))

booking.model_dump(include={'guest_name', 'total'})={'guest_name': 'Asha Rao', 'total': Decimal('10500')}
booking.model_dump(mode='json', exclude={'check_in', 'check_out'})={'guest_name': 'Asha Rao', 'room_type': 'double', 'nightly_rate': '3500', 'total': '10500'}
{
  "guest_name": "Asha Rao",
  "nightly_rate": "3500",
  "total": "10500"
}


`exclude_unset`, `exclude_defaults` and `exclude_none` on a model with defaults:

In [118]:
class Preferences(BaseModel):
    """Every field has a default."""

    theme: Literal["light", "dark"] = "light"
    page_size: int = 20
    nickname: str | None = None


sent = Preferences.model_validate({"theme": "light", "nickname": None})
print(f"{sent.model_fields_set=}")
print(f"{sent.model_dump()=}")
print(f"{sent.model_dump(exclude_unset=True)=}")
print(f"{sent.model_dump(exclude_defaults=True)=}")
print(f"{sent.model_dump(exclude_none=True)=}")

sent.model_fields_set={'nickname', 'theme'}
sent.model_dump()={'theme': 'light', 'page_size': 20, 'nickname': None}
sent.model_dump(exclude_unset=True)={'theme': 'light', 'nickname': None}
sent.model_dump(exclude_defaults=True)={}
sent.model_dump(exclude_none=True)={'theme': 'light', 'page_size': 20}


`model_validate_json` parses bytes or text directly (one pass, in Rust), and
`model_validate` with `from_attributes=True` reads attributes of any object (a
dataclass, an ORM row in Unit 6.4):

In [119]:
print(Preferences.model_validate_json(b'{"theme": "dark", "page_size": "50"}'))


@dataclasses.dataclass
class PreferencesRow:
    """Stands in for a database row object."""

    theme: str
    page_size: int
    nickname: str | None


row = PreferencesRow(theme="dark", page_size=10, nickname="chai-lover")
print(Preferences.model_validate(row, from_attributes=True))
print(raised_error_name(lambda: Preferences.model_validate(row)))

theme='dark' page_size=50 nickname=None
theme='dark' page_size=10 nickname='chai-lover'
ValidationError


#### `ConfigDict`

`model_config = ConfigDict(...)` sets behaviour for the whole model:

| Option | Effect |
| --- | --- |
| `extra="forbid"` / `"ignore"` (default) / `"allow"` | reject, drop or keep unknown keys |
| `frozen=True` | instances are immutable **and hashable** (value objects, dict keys) |
| `from_attributes=True` | `model_validate(obj)` reads attributes (ORM mode) |
| `str_strip_whitespace=True` | strip every `str` field |
| `alias_generator=to_camel` + `validate_by_name=True` | camelCase on the wire, snake_case in Python |
| `strict=True` | no type coercion anywhere in the model |
| `validate_assignment=True` | re-validate on attribute assignment |
| `json_schema_extra={...}` | extra JSON Schema keys (examples, 6.2.9) |

In [120]:
class StrictSignup(BaseModel):
    """extra="forbid" and str_strip_whitespace=True."""

    model_config = ConfigDict(extra="forbid", str_strip_whitespace=True)

    username: Annotated[str, Field(min_length=3)]


class Currency(BaseModel):
    """frozen=True: immutable and hashable, a value object."""

    model_config = ConfigDict(frozen=True)

    code: str
    minor_units: int = 2


print(StrictSignup(username="   asha  "))
print(raised_message(lambda: StrictSignup(username="asha", is_admin=True)).splitlines()[2].strip())
inr = Currency(code="INR")
print(f"{hash(inr) == hash(Currency(code='INR'))=}  {({inr: 'rupee'})[Currency(code='INR')]=}")
print(raised_message(lambda: setattr(inr, "code", "USD")).splitlines()[2].strip())

username='asha'
Extra inputs are not permitted [type=extra_forbidden, input_value=True, input_type=bool]
hash(inr) == hash(Currency(code='INR'))=True  ({inr: 'rupee'})[Currency(code='INR')]='rupee'
Instance is frozen [type=frozen_instance, input_value='USD', input_type=str]


**camelCase on the wire.** JavaScript clients expect `firstName`; Python code wants
`first_name`. An alias generator converts every field name; `validate_by_name=True`
also accepts the Python name; `by_alias=True` when dumping produces camelCase.
(FastAPI dumps response models by alias automatically.)

In [121]:
from pydantic.alias_generators import to_camel


class CamelModel(BaseModel):
    """Base class: camelCase JSON, snake_case Python."""

    model_config = ConfigDict(
        alias_generator=to_camel, validate_by_name=True, validate_by_alias=True
    )


class UserProfile(CamelModel):
    """Inherits the camelCase config."""

    first_name: str
    last_login_at: datetime | None = None
    is_email_verified: bool = False


profile = UserProfile.model_validate({"firstName": "Meena", "isEmailVerified": True})
print(profile)
print(profile.model_dump(by_alias=True, mode="json"))
print(UserProfile(first_name="Ravi").first_name)

first_name='Meena' last_login_at=None is_email_verified=True
{'firstName': 'Meena', 'lastLoginAt': None, 'isEmailVerified': True}
Ravi


#### Strict vs lax mode

By default Pydantic is **lax**: it converts input when no information is lost
(`"42"` → `42`, `42.0` → `42`, even `True` → `1`), and refuses lossy conversions
(`4.5` → int is an error). That suits query strings and forms, which are always
text. **Strict** mode accepts only the exact type: per field (`Field(strict=True)`
or `StrictInt`), per model (`ConfigDict(strict=True)`), or per call
(`model_validate(data, strict=True)`). In strict JSON mode, a few unambiguous
conversions remain (an ISO string is still accepted for `datetime`).

In [122]:
class Counter(BaseModel):
    """One lax field, one strict field."""

    lax_count: int = 0
    strict_count: Annotated[int, Field(strict=True)] = 0


def parsed_or_error(field_name: str, raw_value: object) -> str:
    """The parsed value of one Counter field, or the error type."""
    try:
        return repr(getattr(Counter(**{field_name: raw_value}), field_name))
    except ValidationError as error:
        return error.errors()[0]["type"]


for raw_value in ("42", 42.0, True, 4.5):
    lax_result = parsed_or_error("lax_count", raw_value)
    strict_result = parsed_or_error("strict_count", raw_value)
    print(f"{raw_value!r:<6} lax -> {lax_result:<15} strict -> {strict_result}")
print(validation_problems(lambda: Counter.model_validate({"lax_count": "42"}, strict=True)))

'42'   lax -> 42              strict -> int_type
42.0   lax -> 42              strict -> int_type
True   lax -> 1               strict -> int_type
4.5    lax -> int_from_float  strict -> int_type
[('lax_count', 'int_type')]


#### `TypeAdapter`

Not everything is a model. `TypeAdapter(T)` gives any type the model methods:
validate a `list[int]`, a `dict[str, Decimal]`, a `TypedDict` or a union; dump it to
JSON; produce its JSON Schema. Build adapters once (module level): construction
compiles a validator.

In [123]:
price_list = TypeAdapter(dict[str, Annotated[Decimal, Field(gt=0)]])
prices = price_list.validate_json('{"TEA-500": "240.00", "MUG-02": 199}')
print(f"{prices=}")
print(f"{price_list.dump_json(prices)=}")
print(f"{price_list.json_schema()=}")
print(raised_message(lambda: price_list.validate_python({"TEA-500": -1})).splitlines()[1:3])

prices={'TEA-500': Decimal('240.00'), 'MUG-02': Decimal('199')}
price_list.dump_json(prices)=b'{"TEA-500":"240.00","MUG-02":"199"}'
price_list.json_schema()={'additionalProperties': {'anyOf': [{'exclusiveMinimum': 0.0, 'type': 'number'}, {'type': 'string'}]}, 'type': 'object'}
['TEA-500', '  Input should be greater than 0 [type=greater_than, input_value=-1, input_type=int]']


#### `ValidationError.errors()`

Every problem is a dict with the same keys, which is what FastAPI returns in a
422's `detail` list. Clients (and your tests) should use `type` and `loc`, which are
stable, not `msg`, which is for humans:

| Key | Meaning |
| --- | --- |
| `type` | machine-readable error code (`missing`, `int_parsing`, `greater_than`, …) |
| `loc` | path to the bad value: `("items", 2, "quantity")` |
| `msg` | human-readable message |
| `input` | the value that failed |
| `ctx` | parameters of the constraint (`{"gt": 0}`) |
| `url` | documentation link for the error type |

`errors(include_url=False, include_context=False)` trims it; `error.json()` gives
JSON text:

In [124]:
class Cart(BaseModel):
    """A nested list, so errors have multi-part locations."""

    items: list[LineItem]


try:
    Cart.model_validate(
        {
            "items": [
                {"sku": "TEA-500", "quantity": 1, "unit_price": "10"},
                {"sku": "TEA-500", "quantity": 0, "unit_price": "10"},
            ]
        }
    )
except ValidationError as error:
    problem = error.errors()[0]
    for key in ("type", "loc", "msg", "input", "ctx", "url"):
        print(f"{key:<6} {problem[key]!r}")
    print(error.errors(include_url=False, include_context=False))

type   'greater_than'
loc    ('items', 1, 'quantity')
msg    'Input should be greater than 0'
input  0
ctx    {'gt': 0}
url    'https://errors.pydantic.dev/2.14/v/greater_than'
[{'type': 'greater_than', 'loc': ('items', 1, 'quantity'), 'msg': 'Input should be greater than 0', 'input': 0}]


#### `Annotated` custom types

`Annotated[base, validator, serializer, ...]` builds a reusable type with its own
parsing and output rules, without subclassing:

| Marker | Runs |
| --- | --- |
| `BeforeValidator(fn)` | before the base type's validation, on raw input |
| `AfterValidator(fn)` | after it, on the typed value |
| `PlainValidator(fn)` | instead of it |
| `PlainSerializer(fn, when_used="json")` | when dumping |
| `StringConstraints(strip_whitespace=True, to_lower=True, ...)` | str clean-up and limits |

A tag type for the Notes API, and a rupee amount that is a `Decimal` in Python and a
formatted string in JSON. Order matters: `StringConstraints(pattern=...)` checks
the **raw** string, before `strip_whitespace` and `to_lower`, so the pattern check
goes in an `AfterValidator`:

In [125]:
from pydantic import AfterValidator, BeforeValidator, PlainSerializer, StringConstraints


def require_slug(value: str) -> str:
    """Allow lowercase letters, digits and dashes only."""
    if not re.fullmatch(r"[a-z0-9-]+", value):
        raise ValueError("use lowercase letters, digits and dashes")
    return value


type Tag = Annotated[
    str,
    StringConstraints(strip_whitespace=True, to_lower=True, min_length=1, max_length=30),
    AfterValidator(require_slug),
]
type Rupees = Annotated[
    Decimal,
    BeforeValidator(lambda raw: str(raw).replace(",", "").removeprefix("₹")),
    PlainSerializer(lambda amount: f"₹{amount:,.2f}", return_type=str, when_used="json"),
]


class Expense(BaseModel):
    """Uses both custom types."""

    amount: Rupees
    tags: list[Tag]


expense = Expense(amount="₹1,25,000.5", tags=["  Travel ", "Q4"])
print(expense)
print(expense.model_dump_json())
print(raised_message(lambda: Expense(amount=1, tags=["team lunch"])).splitlines()[2].strip())

amount=Decimal('125000.5') tags=['travel', 'q4']
{"amount":"₹125,000.50","tags":["travel","q4"]}
Value error, use lowercase letters, digits and dashes [type=value_error, input_value='team lunch', input_type=str]


**Under the hood.** A model class is built by `ModelMetaclass` (Unit 2.5's
metaclass machinery): it reads the annotations, builds a **core schema** (a tree of
plain dicts), and hands it to `pydantic-core`, which compiles a `SchemaValidator`
and `SchemaSerializer` in Rust. Instances keep their data in `__dict__` plus a set
of the fields that were set:

In [126]:
print(f"{type(Expense).__name__=}  {type(Expense.__pydantic_validator__).__name__=}")
print(f"{Expense.__pydantic_core_schema__['type']=}  {list(Expense.model_fields)=}")
print(f"{Expense.model_fields['tags']=}")
print(f"{expense.__dict__=}")
print(f"{expense.model_fields_set=}")
print(f"{Expense.__pydantic_validator__.validate_python({'amount': '10', 'tags': ['x']})=}")

type(Expense).__name__='ModelMetaclass'  type(Expense.__pydantic_validator__).__name__='SchemaValidator'
Expense.__pydantic_core_schema__['type']='model'  list(Expense.model_fields)=['amount', 'tags']
Expense.model_fields['tags']=FieldInfo(annotation=list[Tag], required=True)
expense.__dict__={'amount': Decimal('125000.5'), 'tags': ['travel', 'q4']}
expense.model_fields_set={'amount', 'tags'}
Expense.__pydantic_validator__.validate_python({'amount': '10', 'tags': ['x']})=Expense(amount=Decimal('10'), tags=['x'])


### 6.2.6 Schema design

One resource usually needs **several schemas**, because the client and the server
own different fields at different moments:

| Schema | Used for | Fields |
| --- | --- | --- |
| `UserCreate` | `POST` body | what the client provides, including secrets (`password`) |
| `UserUpdate` | `PATCH` body | the same fields, **all optional** |
| `UserRead` | responses | server-owned fields too (`id`, `created_at`), and **never** secrets |
| (internal) | storage | everything, e.g. `password_hash` |

A shared base class avoids repeating the common fields. `extra="forbid"` on input
schemas turns client typos into 422s instead of silently ignored keys:

In [127]:
class UserBase(BaseModel):
    """Fields every user schema shares."""

    username: Annotated[str, Field(min_length=3, max_length=30, pattern=r"^[a-z0-9_]+$")]
    email: EmailStr
    display_name: Annotated[str, Field(max_length=60)] = ""


class UserCreate(UserBase):
    """POST body: the client sends a password."""

    model_config = ConfigDict(extra="forbid")

    password: Annotated[str, Field(min_length=12)]


class UserUpdate(BaseModel):
    """PATCH body: every field optional, nothing required."""

    model_config = ConfigDict(extra="forbid")

    email: EmailStr | None = None
    display_name: Annotated[str | None, Field(max_length=60)] = None


class UserRead(UserBase):
    """Response: server-owned fields, no password."""

    model_config = ConfigDict(from_attributes=True)

    id: int
    created_at: datetime

The same data through each schema. The stored record has a `password_hash`;
validating it as `UserRead` drops it, so it can never leak into a response:

In [128]:
new_user = UserCreate(username="asha_r", email="asha@example.com", password="correct horse battery")
stored = {
    **new_user.model_dump(exclude={"password"}),
    "id": 1,
    "password_hash": "argon2:...",
    "created_at": datetime(2026, 11, 17, 9, 0, tzinfo=UTC),
}
print(UserRead.model_validate(stored).model_dump(mode="json"))
print(
    validation_problems(
        lambda: UserCreate(
            username="ravi", email="ravi@example.com", password="short", is_admin=True
        )
    )
)
print(f"{sorted(UserRead.model_fields)=}")

{'username': 'asha_r', 'email': 'asha@example.com', 'display_name': '', 'id': 1, 'created_at': '2026-11-17T09:00:00Z'}
[('password', 'string_too_short'), ('is_admin', 'extra_forbidden')]
sorted(UserRead.model_fields)=['created_at', 'display_name', 'email', 'id', 'username']


#### PATCH with `exclude_unset`

A PATCH body says "change these fields". The Update schema makes every field
optional, which creates a problem: `None` now means both "not sent" and "set it to
null". `model_dump(exclude_unset=True)` resolves it: it contains only the keys the
client actually sent. Merge that into the stored record:

| Client sends | `exclude_unset=True` | `exclude_none=True` (wrong) |
| --- | --- | --- |
| `{"display_name": "Asha R"}` | `{"display_name": "Asha R"}` | same |
| `{"display_name": null}` (clear it) | `{"display_name": None}` | `{}`: the clear is lost |
| `{}` | `{}` | `{}` |

In [129]:
def apply_patch(record: dict[str, object], changes: UserUpdate) -> dict[str, object]:
    """Merge only the fields the client sent into a copy of record."""
    return {**record, **changes.model_dump(exclude_unset=True)}


record = {"username": "asha_r", "email": "asha@example.com", "display_name": "Asha"}
for body in ({"display_name": "Asha R"}, {"display_name": None}, {}):
    changes = UserUpdate.model_validate(body)
    print(
        f"{body!s:<28} unset-> {apply_patch(record, changes)['display_name']!r:<9}"
        f" none-> {({**record, **changes.model_dump(exclude_none=True)})['display_name']!r}"
    )

{'display_name': 'Asha R'}   unset-> 'Asha R'  none-> 'Asha R'
{'display_name': None}       unset-> None      none-> 'Asha'
{}                           unset-> 'Asha'    none-> 'Asha'


> **Warning:** with `exclude_unset`, a client can also send `{"email": null}` for a
> field that must never be null. The Notes API's Update schema (after Lab 6.2)
> rejects explicit `null` for non-nullable fields with a `model_validator`.

### 6.2.7 Responses

#### `response_model`

The response model is the contract for what leaves the server. FastAPI validates
the returned value against it and serializes **only its fields**, so extra data
(a password hash, internal flags) is dropped even if the handler returns it. Set it
with `response_model=...` in the decorator, or with the return annotation (FastAPI
uses the annotation when no `response_model` is given). Options:
`response_model_exclude_unset`, `response_model_exclude_none`,
`response_model_include/exclude`; `response_model=None` turns it off.

In [130]:
users_api = FastAPI()
USER_ROWS = {
    1: {
        "id": 1,
        "username": "asha_r",
        "email": "asha@example.com",
        "display_name": "",
        "password_hash": "argon2:secret",
        "created_at": datetime(2026, 11, 17, tzinfo=UTC),
    }
}


@users_api.get("/users/{user_id}", response_model=UserRead)
async def read_user_row(user_id: int) -> dict[str, object]:
    """Returns the raw row; response_model filters it to UserRead's fields."""
    return USER_ROWS[user_id]


@users_api.get(
    "/users/{user_id}/compact", response_model=UserRead, response_model_exclude_defaults=True
)
async def read_user_compact(user_id: int) -> dict[str, object]:
    """Same row; fields still equal to their defaults (display_name="") are left out."""
    return USER_ROWS[user_id]


with TestClient(users_api) as client:
    print(client.get("/users/1").json())
    print(client.get("/users/1/compact").json())

{'username': 'asha_r', 'email': 'asha@example.com', 'display_name': '', 'id': 1, 'created_at': '2026-11-17T00:00:00Z'}


{'username': 'asha_r', 'email': 'asha@example.com', 'id': 1, 'created_at': '2026-11-17T00:00:00Z'}


If the handler returns something that does not fit the response model, that is a
**server** bug, so FastAPI raises `ResponseValidationError` (a 500), never sending
invalid data:

In [131]:
@users_api.get("/broken")
async def broken_user() -> UserRead:
    """The return annotation is the response model; this dict lacks required fields."""
    return {"username": "x"}  # type: ignore[return-value]


with TestClient(users_api, raise_server_exceptions=False) as client:
    print(client.get("/broken").status_code)
with TestClient(users_api) as client:
    print(raised_error_name(lambda: client.get("/broken")))

500
ResponseValidationError


#### Status codes

`status_code=` in the decorator sets the success code and documents it; the
`fastapi.status` module has named constants (`status.HTTP_201_CREATED`). A `204`
response has no body. To choose the code at run time, declare a `response: Response`
parameter and set `response.status_code` (an upsert `PUT`: 201 when created, 200
when replaced):

In [132]:
status_app = FastAPI()
SETTINGS: dict[str, str] = {}


@status_app.put("/settings/{key}", status_code=status.HTTP_200_OK)
async def upsert_setting(
    key: str, value: Annotated[str, Body(embed=True)], response: Response
) -> dict[str, str]:
    """201 when the key is new, 200 when it is replaced."""
    if key not in SETTINGS:
        response.status_code = status.HTTP_201_CREATED
    SETTINGS[key] = value
    return {key: value}


@status_app.delete("/settings/{key}", status_code=status.HTTP_204_NO_CONTENT)
async def delete_setting(key: str) -> None:
    """204: success, nothing to say."""
    SETTINGS.pop(key, None)


with TestClient(status_app) as client:
    themes = ("dark", "light")
    print([client.put("/settings/theme", json={"value": theme}).status_code for theme in themes])
    deleted = client.delete("/settings/theme")
    print(f"{deleted.status_code=} {deleted.content=}  {status.HTTP_422_UNPROCESSABLE_CONTENT=}")

[201, 200]
deleted.status_code=204 deleted.content=b''  status.HTTP_422_UNPROCESSABLE_CONTENT=422


#### Response classes

Returning a dict, list or model gives a `JSONResponse`. Return a `Response` subclass
yourself for anything else; it is sent as is (**no response-model filtering**).
`response_class=` sets the default class for a route and documents the media type.

| Class | Use |
| --- | --- |
| `JSONResponse(content, status_code, headers)` | JSON with a custom status or headers |
| `PlainTextResponse`, `HTMLResponse` | text / HTML |
| `StreamingResponse(iterator, media_type)` | a body produced piece by piece (NDJSON, CSV exports, server-sent events) |
| `FileResponse(path, filename=...)` | a file from disk, with `Content-Length`, `ETag`, `Last-Modified`, `Content-Disposition` |
| `RedirectResponse(url, status_code=307)` | redirects (307/308 keep the method and body; 301/302/303 may turn POST into GET) |

In [133]:
from fastapi.responses import (
    FileResponse,
    HTMLResponse,
    JSONResponse,
    PlainTextResponse,
    RedirectResponse,
    StreamingResponse,
)

responses_app = FastAPI()


@responses_app.get("/accepted")
async def accepted_job() -> JSONResponse:
    """A JSONResponse with its own status code and header."""
    return JSONResponse(
        {"job_id": "job-42", "state": "queued"},
        status_code=202,
        headers={"Location": "/jobs/job-42"},
    )


@responses_app.get("/robots.txt", response_class=PlainTextResponse)
async def robots() -> str:
    """response_class turns the returned str into text/plain."""
    return "User-agent: *\nDisallow: /admin/\n"


@responses_app.get("/hello.html", response_class=HTMLResponse)
async def hello_page() -> str:
    """An HTML page from a string."""
    return "<h1>Hello from FastAPI</h1>"


with TestClient(responses_app) as client:
    job = client.get("/accepted")
    print(job.status_code, job.headers["location"], job.json())
    for path in ("/robots.txt", "/hello.html"):
        reply = client.get(path)
        print(f"{path:<12} {reply.headers['content-type']:<26} {reply.text!r}")

202 /jobs/job-42 {'job_id': 'job-42', 'state': 'queued'}
/robots.txt  text/plain; charset=utf-8  'User-agent: *\nDisallow: /admin/\n'


/hello.html  text/html; charset=utf-8   '<h1>Hello from FastAPI</h1>'


`StreamingResponse` takes a sync or async iterator. Here an export streams one JSON
object per line (NDJSON) without building the whole body in memory; the client reads
it line by line:

In [134]:
async def order_lines(count: int) -> AsyncIterator[str]:
    """Yield one JSON document per line, as a database cursor would."""
    for order_id in range(1, count + 1):
        yield json.dumps({"order_id": order_id, "total": order_id * 120}) + "\n"
        await asyncio.sleep(0)  # let other tasks run between rows


@responses_app.get("/orders/export")
async def export_orders(count: Annotated[int, Query(ge=1, le=1000)] = 3) -> StreamingResponse:
    """Stream NDJSON."""
    return StreamingResponse(order_lines(count), media_type="application/x-ndjson")


with (
    TestClient(responses_app) as client,
    client.stream("GET", "/orders/export?count=3") as streamed,
):
    print(streamed.headers["content-type"], streamed.headers.get("content-length"))
    for line in streamed.iter_lines():
        print(" ", json.loads(line))

application/x-ndjson None
  {'order_id': 1, 'total': 120}
  {'order_id': 2, 'total': 240}
  {'order_id': 3, 'total': 360}


`FileResponse` serves a file efficiently and fills in the caching headers;
`filename=` adds `Content-Disposition: attachment` so browsers download it.
`RedirectResponse` defaults to **307**; use 308 for a permanent move that keeps the
method:

In [135]:
report_path = PHASE6_DIR / "sales_report.csv"
report_path.write_text("month,revenue\n2026-10,125000\n2026-11,142000\n", encoding="utf-8")


@responses_app.get("/reports/sales.csv")
async def download_report() -> FileResponse:
    """Send a file from disk as a download."""
    return FileResponse(report_path, media_type="text/csv", filename="sales-2026.csv")


@responses_app.get("/old-reports/sales.csv")
async def moved_report() -> RedirectResponse:
    """Permanent move that keeps the method."""
    return RedirectResponse("/reports/sales.csv", status_code=status.HTTP_308_PERMANENT_REDIRECT)


with TestClient(responses_app) as client:
    download = client.get("/reports/sales.csv")
    for header in (
        "content-type",
        "content-length",
        "content-disposition",
        "etag",
        "last-modified",
    ):
        value = "<varies>" if header in ("etag", "last-modified") else download.headers[header]
        print(f"{header:<20} {value}")
    moved = client.get("/old-reports/sales.csv", follow_redirects=False)
    print(
        moved.status_code,
        moved.headers["location"],
        "| followed:",
        client.get("/old-reports/sales.csv").status_code,
    )

content-type         text/csv; charset=utf-8
content-length       44
content-disposition  attachment; filename="sales-2026.csv"
etag                 <varies>
last-modified        <varies>


308 /reports/sales.csv | followed: 200


#### Headers and cookies

To add headers or cookies **and** keep returning plain data (so the response model
still applies), declare a `response: Response` parameter and modify it. Request
headers are read with `Header()` (the parameter `user_agent` reads `User-Agent`:
underscores become dashes, and matching ignores case); cookies with `Cookie()`.
Cookie flags matter for security: `httponly` (no JavaScript access), `secure`
(HTTPS only), `samesite` (CSRF protection), `max_age`.

In [136]:
from fastapi import Cookie, Header

session_app = FastAPI()


@session_app.post("/login")
async def login(response: Response) -> dict[str, str]:
    """Set a session cookie and a custom header on a normal JSON response."""
    response.set_cookie(
        "session_id", "s-123", httponly=True, secure=True, samesite="lax", max_age=3600
    )
    response.headers["X-Login-Count"] = "1"
    return {"status": "logged in"}


@session_app.get("/whoami")
async def whoami(
    session_id: Annotated[str | None, Cookie()] = None,
    user_agent: Annotated[str | None, Header()] = None,
) -> dict[str, str | None]:
    """Read a cookie and a request header."""
    return {"session_id": session_id, "user_agent": user_agent}


@session_app.post("/logout")
async def logout(response: Response) -> dict[str, str]:
    """Delete the cookie (an expired Set-Cookie)."""
    response.delete_cookie("session_id")
    return {"status": "logged out"}


with TestClient(session_app, base_url="https://testserver") as client:
    logged_in = client.post("/login")
    print(logged_in.headers["set-cookie"])
    print(
        logged_in.headers["x-login-count"],
        client.get("/whoami", headers={"User-Agent": "notes-cli/1.0"}).json(),
    )
    print(re.sub(r"expires=[^;]+", "expires=<now>", client.post("/logout").headers["set-cookie"]))

session_id=s-123; HttpOnly; Max-Age=3600; Path=/; SameSite=lax; Secure
1 {'session_id': 's-123', 'user_agent': 'notes-cli/1.0'}


session_id=""; expires=<now>; Max-Age=0; Path=/; SameSite=lax


#### `HTTPException`

Raise `HTTPException(status_code, detail, headers)` anywhere in a handler (or in a
function it calls) to stop and send an error. FastAPI's version accepts any
JSON-serializable `detail` (a string, or a dict with a machine-readable code) and
extra headers. The body is always `{"detail": ...}`. In Express terms: `throw
createError(404, ...)` instead of `return res.status(404).json(...)`.

In [137]:
errors_app = FastAPI()
TAKEN_USERNAMES = {"asha_r"}


@errors_app.get("/notes/{note_id}")
async def find_note(
    note_id: int, authorization: Annotated[str | None, Header()] = None
) -> dict[str, int]:
    """401 without credentials, 404 for unknown notes."""
    if authorization is None:
        raise HTTPException(
            status.HTTP_401_UNAUTHORIZED,
            detail="not authenticated",
            headers={"WWW-Authenticate": "Bearer"},
        )
    if note_id != 1:
        raise HTTPException(status.HTTP_404_NOT_FOUND, detail=f"note {note_id} not found")
    return {"id": note_id}


@errors_app.post("/users", status_code=201)
async def register(username: Annotated[str, Body(embed=True)]) -> dict[str, str]:
    """409 with a structured detail when the name is taken."""
    if username in TAKEN_USERNAMES:
        raise HTTPException(
            status.HTTP_409_CONFLICT, detail={"code": "username_taken", "field": "username"}
        )
    return {"username": username}


with TestClient(errors_app) as client:
    unauthenticated = client.get("/notes/1")
    print(
        unauthenticated.status_code,
        unauthenticated.json(),
        unauthenticated.headers["www-authenticate"],
    )
    print(client.get("/notes/7", headers={"Authorization": "Bearer t"}).json())
    taken = client.post("/users", json={"username": "asha_r"})
    print(taken.status_code, taken.json())

401 {'detail': 'not authenticated'} Bearer
{'detail': 'note 7 not found'}
409 {'detail': {'code': 'username_taken', 'field': 'username'}}


### 6.2.8 `async def` vs `def` handlers

FastAPI accepts both, and runs them differently:

| Handler | Runs | A blocking call inside it (`time.sleep`, `requests.get`, a sync DB driver) |
| --- | --- | --- |
| `async def` | **on the event loop**, in the server's only thread | **blocks every request** on that worker until it returns |
| `def` | in AnyIO's **thread pool** (`run_in_threadpool`), default 40 threads | blocks only its own thread; at most 40 such calls at once per process |
| `async def` + `await run_in_threadpool(fn, ...)` | loop, with one call pushed to the pool | fine: only that call uses a thread |

Rules of thumb: use `async def` when everything you await is async (httpx
`AsyncClient`, async database drivers, `asyncio.sleep`); use plain `def` when you call
blocking libraries; never call blocking code directly inside `async def`. CPU-heavy
work blocks either way (the GIL, Unit 5.1): send it to a process pool or a job queue
(Unit 6.6).

Which thread runs each kind of handler:

In [138]:
threads_app = FastAPI()


@threads_app.get("/sync")
def sync_handler() -> dict[str, str]:
    """def: called in a worker thread."""
    return {"thread": threading.current_thread().name}


@threads_app.get("/async")
async def async_handler() -> dict[str, object]:
    """async def: called on the event loop's thread."""
    limiter = anyio.to_thread.current_default_thread_limiter()
    return {"thread": threading.current_thread().name, "pool_size": limiter.total_tokens}


with TestClient(threads_app) as client:
    print("def       ->", client.get("/sync").json())
    async_reply = client.get("/async").json()
    print(
        "async def ->",
        async_reply | {"thread": re.sub(r"-[0-9a-f]{6,}$", "-<id>", async_reply["thread"])},
    )

def       -> {'thread': 'AnyIO worker thread'}
async def -> {'thread': 'asyncio-portal-<id>', 'pool_size': 40}


> **Note:** `TestClient` runs the app's event loop in its own thread, whose name you
> see for `async def`; under Uvicorn it is the main thread. `def` handlers always run
> in "AnyIO worker thread"s.

#### A blocking demo under load

The app below has the same 0.2 s wait written four ways. It runs under Uvicorn in a
child process (one worker, like a single container), and each endpoint gets 10
concurrent requests. A fifth measurement sends one `/health` request **while** 10
`/async-blocking` requests are in flight:

Part 1 of `blocking_api.py`: the health check and the **wrong** way to wait.

In [139]:
%%writefile _scratch/phase6/blocking_api.py
"""Four ways to wait 0.2 s in a FastAPI handler, for the 6.2.8 load test."""

import asyncio
import threading
import time

import anyio.to_thread
from fastapi import FastAPI
from fastapi.concurrency import run_in_threadpool

DELAY_SECONDS = 0.2
app = FastAPI()


@app.get("/health")
async def health() -> dict[str, str]:
    """Answers at once, unless the event loop is blocked."""
    return {"status": "ok"}


@app.get("/async-blocking")
async def async_blocking() -> dict[str, str]:
    """WRONG: a blocking call inside async def freezes the event loop."""
    time.sleep(DELAY_SECONDS)
    return {"thread": threading.current_thread().name}

Overwriting _scratch/phase6/blocking_api.py


Part 2: three correct ways, and the thread-pool size.

In [140]:
%%writefile -a _scratch/phase6/blocking_api.py


@app.get("/sync-blocking")
def sync_blocking() -> dict[str, str]:
    """A plain def handler: FastAPI runs it in the thread pool, so blocking is fine."""
    time.sleep(DELAY_SECONDS)
    return {"thread": threading.current_thread().name}


@app.get("/async-sleep")
async def async_sleep() -> dict[str, str]:
    """async def with a non-blocking wait: the loop serves others meanwhile."""
    await asyncio.sleep(DELAY_SECONDS)
    return {"thread": threading.current_thread().name}


@app.get("/async-offloaded")
async def async_offloaded() -> dict[str, str]:
    """async def that pushes one blocking call to the thread pool explicitly."""
    await run_in_threadpool(time.sleep, DELAY_SECONDS)
    return {"thread": threading.current_thread().name}


@app.get("/thread-pool")
async def thread_pool() -> dict[str, int]:
    """The size of the thread pool that def handlers share (AnyIO's default limiter)."""
    return {"total_tokens": int(anyio.to_thread.current_default_thread_limiter().total_tokens)}

Appending to _scratch/phase6/blocking_api.py


Run the load test (times vary by machine; the pattern does not):

In [141]:
async def timed_burst(client: httpx.AsyncClient, path: str, request_count: int) -> float:
    """Wall time for request_count concurrent GETs of path."""
    started = time.perf_counter()
    await asyncio.gather(*(client.get(path) for _ in range(request_count)))
    return time.perf_counter() - started


async def health_latency_during(client: httpx.AsyncClient, path: str, request_count: int) -> float:
    """How long /health takes while request_count requests to path are running."""
    burst = asyncio.gather(*(client.get(path) for _ in range(request_count)))
    await asyncio.sleep(0.05)  # let the burst reach the server first
    started = time.perf_counter()
    await client.get("/health")
    latency = time.perf_counter() - started
    await burst
    return latency


limits = httpx.Limits(max_connections=100, max_keepalive_connections=100)
with uvicorn_server("blocking_api:app") as base_url:
    async with httpx.AsyncClient(
        base_url=base_url, limits=limits, timeout=60, trust_env=False
    ) as client:
        for path in ("/async-blocking", "/sync-blocking", "/async-sleep", "/async-offloaded"):
            seconds = await timed_burst(client, path, 10)
            print(f"{path:<17} 10 requests x 0.2 s: {seconds:5.2f} s")
        latency = await health_latency_during(client, "/async-blocking", 10)
        print(f"/health while /async-blocking is busy: {latency:5.2f} s (normally ~0.001 s)")
        pool = (await client.get("/thread-pool")).json()["total_tokens"]
        seconds = await timed_burst(client, "/sync-blocking", 60)
        print(f"/sync-blocking with 60 requests and a pool of {pool}: {seconds:5.2f} s")

/async-blocking   10 requests x 0.2 s:  2.01 s


/sync-blocking    10 requests x 0.2 s:  0.23 s


/async-sleep      10 requests x 0.2 s:  0.21 s


/async-offloaded  10 requests x 0.2 s:  0.21 s


/health while /async-blocking is busy:  1.96 s (normally ~0.001 s)


/sync-blocking with 60 requests and a pool of 40:  0.43 s


What to notice:

- **`/async-blocking`** serializes: 10 × 0.2 s ≈ 2 s, and even `/health` waits for
  the queue, because the event loop is stuck in `time.sleep`. This is the single
  most common FastAPI performance bug.
- **`/sync-blocking`** overlaps the waits in the thread pool: ≈ 0.2 s.
- **`/async-sleep`** and **`/async-offloaded`** overlap too.
- **60 sync requests** need two rounds (40 threads, then 20): ≈ 0.4 s. The pool
  size can be raised (`anyio.to_thread.current_default_thread_limiter().total_tokens
  = 100` in a lifespan handler), but more threads cost memory and GIL contention;
  async drivers scale further.

### 6.2.9 OpenAPI

FastAPI generates an **OpenAPI 3.1** document from your routes and models and
serves it with two UIs:

| URL | What | Turn off / move |
| --- | --- | --- |
| `/openapi.json` | the machine-readable description | `FastAPI(openapi_url=None)` disables all three |
| `/docs` | Swagger UI: try requests in the browser | `docs_url=None` or another path |
| `/redoc` | ReDoc: readable reference docs | `redoc_url=None` |

What you can add, and where:

| Level | Options |
| --- | --- |
| app | `title`, `version`, `summary`, `description` (Markdown), `contact`, `license_info`, `openapi_tags`, `servers`, `generate_unique_id_function` |
| route | `tags`, `summary`, `description` (or the docstring), `response_description`, `responses={404: {...}}`, `deprecated=True`, `include_in_schema=False`, `operation_id` |
| model / field | `Field(description=..., examples=[...])`, `model_config = ConfigDict(json_schema_extra={"examples": [...]})` |
| parameter | `Body(openapi_examples={...})`, `Query(description=..., deprecated=True)` |

A documented demo API (served from the scratch folder):

Part 1 of `products_api.py`: app metadata and clean operation ids.

In [142]:
%%writefile _scratch/phase6/products_api.py
"""A documented API: OpenAPI metadata, tags, examples and clean operation ids."""

from decimal import Decimal
from typing import Annotated

from fastapi import Body, FastAPI, HTTPException, Query, status
from fastapi.routing import APIRoute
from pydantic import BaseModel, ConfigDict, Field


def operation_id(route: APIRoute) -> str:
    """Use the function name as the operationId: generated clients get listProducts, not
    list_products_products_get."""
    return route.name


app = FastAPI(
    title="Products API",
    version="1.2.0",
    summary="A small catalog, documented for humans and code generators.",
    description="Prices are in **rupees**. Every list endpoint is paginated.",
    contact={"name": "Catalog team", "email": "catalog@example.com"},
    license_info={"name": "MIT", "identifier": "MIT"},
    openapi_tags=[
        {"name": "products", "description": "Browse and manage products."},
        {"name": "health", "description": "Liveness probe."},
    ],
    generate_unique_id_function=operation_id,
)

Overwriting _scratch/phase6/products_api.py


Part 2: schemas with descriptions and examples, and the data.

In [143]:
%%writefile -a _scratch/phase6/products_api.py


class ProductCreate(BaseModel):
    """What a client sends to create a product."""

    model_config = ConfigDict(
        extra="forbid",
        json_schema_extra={"examples": [{"name": "Assam tea", "price": "240.00", "tags": ["tea"]}]},
    )

    name: Annotated[str, Field(min_length=1, max_length=80, description="Display name.")]
    price: Annotated[Decimal, Field(gt=0, decimal_places=2, description="Price in rupees.")]
    tags: Annotated[list[str], Field(default_factory=list, examples=[["tea", "loose-leaf"]])]


class ProductRead(ProductCreate):
    """What the API returns."""

    id: int


class ErrorMessage(BaseModel):
    """The body of an error response."""

    detail: str


PRODUCTS: dict[int, ProductRead] = {
    1: ProductRead(id=1, name="Assam tea", price=Decimal("240.00"), tags=["tea"]),
}

Appending to _scratch/phase6/products_api.py


Part 3: read endpoints, one with a documented 404 response.

In [144]:
%%writefile -a _scratch/phase6/products_api.py


@app.get("/health", tags=["health"])
async def health() -> dict[str, str]:
    """Liveness probe."""
    return {"status": "ok"}


@app.get("/products", tags=["products"], summary="List products")
async def list_products(limit: Annotated[int, Query(ge=1, le=100)] = 20) -> list[ProductRead]:
    """Return up to `limit` products, oldest first."""
    return list(PRODUCTS.values())[:limit]


@app.get(
    "/products/{product_id}",
    tags=["products"],
    responses={404: {"model": ErrorMessage, "description": "No product with this id."}},
)
async def get_product(product_id: int) -> ProductRead:
    """Fetch one product."""
    if product_id not in PRODUCTS:
        raise HTTPException(status.HTTP_404_NOT_FOUND, detail=f"product {product_id} not found")
    return PRODUCTS[product_id]

Appending to _scratch/phase6/products_api.py


Part 4: create (named request examples), a deprecated route and a hidden one.

In [145]:
%%writefile -a _scratch/phase6/products_api.py


@app.post(
    "/products",
    tags=["products"],
    status_code=status.HTTP_201_CREATED,
    response_description="The created product, with its id.",
)
async def create_product(
    product: Annotated[
        ProductCreate,
        Body(
            openapi_examples={
                "tea": {"summary": "A tea", "value": {"name": "Darjeeling", "price": "320.00"}},
                "mug": {"summary": "A mug", "value": {"name": "Clay mug", "price": "199.00"}},
            }
        ),
    ],
) -> ProductRead:
    """Create a product."""
    created = ProductRead(id=max(PRODUCTS) + 1, **product.model_dump())
    PRODUCTS[created.id] = created
    return created


@app.get("/legacy/products-export", tags=["products"], deprecated=True)
async def legacy_export() -> list[ProductRead]:
    """Old endpoint kept for one more release (shown struck through in /docs)."""
    return list(PRODUCTS.values())


@app.get("/internal/metrics", include_in_schema=False)
async def internal_metrics() -> dict[str, int]:
    """Hidden from the docs."""
    return {"products": len(PRODUCTS)}

Appending to _scratch/phase6/products_api.py


The three documentation URLs, and the parts of the generated document. Notice the
operation ids (function names, thanks to `generate_unique_id_function`), the hidden
`/internal/metrics` route, the deprecated flag, and the `HTTPValidationError`
schema FastAPI adds for 422 responses:

In [146]:
products_api = fresh_import("products_api")
with TestClient(products_api.app) as client:
    for path in ("/docs", "/redoc"):
        page = client.get(path)
        title = re.search(r"<title>(.*?)</title>", page.text)[1]
        print(f"{path:<7} {page.status_code} {page.headers['content-type']:<24} {title}")
    products_openapi = client.get("/openapi.json").json()
info = products_openapi["info"]
print(f"OpenAPI {products_openapi['openapi']}: {info['title']} {info['version']}")
for path, operations in products_openapi["paths"].items():
    for method, operation in operations.items():
        flag = " (deprecated)" if operation.get("deprecated") else ""
        name, tags = operation["operationId"], operation["tags"]
        print(f"  {method.upper():<5} {path:<26} {name:<15} tags={tags}{flag}")
print(f"{sorted(products_openapi['components']['schemas'])=}")

/docs   200 text/html; charset=utf-8 Products API - Swagger UI
/redoc  200 text/html; charset=utf-8 Products API - ReDoc
OpenAPI 3.1.0: Products API 1.2.0
  GET   /health                    health          tags=['health']
  GET   /products                  list_products   tags=['products']
  POST  /products                  create_product  tags=['products']
  GET   /products/{product_id}     get_product     tags=['products']
  GET   /legacy/products-export    legacy_export   tags=['products'] (deprecated)
sorted(products_openapi['components']['schemas'])=['ErrorMessage', 'HTTPValidationError', 'ProductCreate', 'ProductRead', 'ValidationError']


The request-body schema with its examples, and the documented responses of
`GET /products/{product_id}`:

In [147]:
print(json.dumps(products_openapi["components"]["schemas"]["ProductCreate"], indent=2)[:900])
print(
    json.dumps(products_openapi["paths"]["/products/{product_id}"]["get"]["responses"], indent=1)[
        :700
    ]
)

{
  "properties": {
    "name": {
      "type": "string",
      "maxLength": 80,
      "minLength": 1,
      "title": "Name",
      "description": "Display name."
    },
    "price": {
      "anyOf": [
        {
          "type": "number",
          "exclusiveMinimum": 0.0
        },
        {
          "type": "string"
        }
      ],
      "title": "Price",
      "description": "Price in rupees."
    },
    "tags": {
      "items": {
        "type": "string"
      },
      "type": "array",
      "title": "Tags",
      "examples": [
        [
          "tea",
          "loose-leaf"
        ]
      ]
    }
  },
  "additionalProperties": false,
  "type": "object",
  "required": [
    "name",
    "price"
  ],
  "title": "ProductCreate",
  "description": "What a client sends to create a product.",
  "examples": [
    {
      "name": "Assam tea",
      "price": "240.00",
      "tags": [
 
{
 "200": {
  "description": "Successful Response",
  "content": {
   "application/json": {
    "sc

#### Generating a TypeScript client

Because the contract is machine-readable, the frontend never has to hand-write
request code or types. Real projects use a generator:

| Tool | Produces |
| --- | --- |
| `openapi-typescript` (+ `openapi-fetch`) | `.d.ts` types for every path and schema, plus a typed `fetch` wrapper |
| `@hey-api/openapi-ts`, `orval` | typed SDK functions, optionally TanStack Query hooks |
| `openapi-generator` | clients for many languages |

```bash
npx openapi-typescript http://127.0.0.1:8000/openapi.json -o src/api/schema.d.ts
```

That needs the npm registry, so the notebook does not run it. Instead, a small
generator shows what such tools do. Part 1 maps JSON Schema to TypeScript types and
writes one `interface` per component schema:

In [148]:
def ts_type(schema: dict[str, Any]) -> str:
    """TypeScript type for a JSON Schema fragment (the common cases)."""
    if "$ref" in schema:
        return schema["$ref"].rsplit("/", 1)[1]
    if "anyOf" in schema:
        return " | ".join(ts_type(option) for option in schema["anyOf"])
    if "enum" in schema:
        return " | ".join(json.dumps(value) for value in schema["enum"])
    if "const" in schema:
        return json.dumps(schema["const"])
    match schema.get("type"):
        case "string":
            return "string"
        case "integer" | "number":
            return "number"
        case "boolean":
            return "boolean"
        case "null":
            return "null"
        case "array":
            item = ts_type(schema.get("items", {}))
            return f"({item})[]" if " " in item else f"{item}[]"
        case "object":
            return f"Record<string, {ts_type(schema.get('additionalProperties') or {})}>"
    return "unknown"

Then one exported `interface` per component schema (`?` marks optional properties):

In [149]:
def typescript_declarations(openapi: dict[str, Any]) -> str:
    """One exported interface (or type) per component schema."""
    lines = [
        f"// Generated from {openapi['info']['title']} {openapi['info']['version']}. Do not edit."
    ]
    for name, schema in openapi["components"]["schemas"].items():
        if "properties" not in schema:
            lines.append(f"export type {name} = {ts_type(schema)};")
            continue
        required = set(schema.get("required", []))
        lines.append(f"export interface {name} {{")
        for prop, prop_schema in schema["properties"].items():
            lines.append(f"  {prop}{'' if prop in required else '?'}: {ts_type(prop_schema)};")
        lines.append("}")
    return "\n".join(lines)

Part 2 writes one `async` function per operation (named after its `operationId`, in
camelCase), as an ES module that runs on Node's built-in `fetch`. JSDoc comments
point at the `.d.ts` types, so editors type-check calls even in plain JavaScript:

In [150]:
def operation_function(
    path: str, method: str, operation: dict[str, Any], types_file: str
) -> list[str]:
    """JavaScript lines for one operation: an async function named after its operationId."""
    name = re.sub(r"_(\w)", lambda match: match[1].upper(), operation["operationId"])
    parameters = operation.get("parameters", [])
    path_names = [parameter["name"] for parameter in parameters if parameter["in"] == "path"]
    has_query = any(parameter["in"] == "query" for parameter in parameters)
    has_body = "requestBody" in operation
    arguments = [
        "baseUrl",
        *path_names,
        *(["body"] if has_body else []),
        *(["query = {}"] if has_query else []),
    ]
    success = next(
        (reply for code, reply in operation["responses"].items() if code.startswith("2")), {}
    )
    result_schema = success.get("content", {}).get("application/json", {}).get("schema", {})
    result_type = ts_type(result_schema).replace("Record<string, unknown>", "object")
    if "$ref" in json.dumps(result_schema):
        result_type = f'import("./{types_file}").{result_type}'
    url_path = re.sub(r"\{(\w+)\}", r"${encodeURIComponent(\1)}", path)
    lines = [
        f"/** {operation.get('summary', name)}. @returns {{Promise<{result_type}>}} */",
        f"export async function {name}({', '.join(arguments)}) {{",
        f"  const url = new URL(`${{baseUrl}}{url_path}`);",
    ]
    if has_query:
        lines += [
            "  for (const [key, value] of Object.entries(query)) {",
            "    for (const item of [value].flat()) url.searchParams.append(key, item);",
            "  }",
        ]
    body_argument = "body" if has_body else "undefined"
    return [*lines, f"  return request({method.upper()!r}, url, {body_argument});", "}"]

The module: a shared `request` helper, then one function per operation:

In [151]:
def javascript_client(openapi: dict[str, Any], types_file: str) -> str:
    """An ES module with one fetch-based function per operation."""
    lines = [
        f"// Generated client for {openapi['info']['title']}. Do not edit.",
        "async function request(method, url, body) {",
        '  const headers = { "content-type": "application/json" };',
        "  const payload = body === undefined ? undefined : JSON.stringify(body);",
        "  const response = await fetch(url, { method, headers, body: payload });",
        "  if (!response.ok) throw new Error(`${response.status} ${await response.text()}`);",
        "  return response.status === 204 ? null : response.json();",
        "}",
    ]
    for path, operations in openapi["paths"].items():
        for method, operation in operations.items():
            lines += operation_function(path, method, operation, types_file)
    return "\n".join(lines)

Generate both files for the Products API, print the types, and run a small Node
script that imports the client and calls the API served by Uvicorn (skipped without
`node`):

In [152]:
CLIENT_DEMO_SOURCE = """
import { createProduct, getProduct, listProducts } from "./products_client.mjs";
const baseUrl = process.argv[2];
const created = await createProduct(baseUrl, { name: "Clay mug", price: "199.00", tags: ["mug"] });
console.log("created:", created);
console.log("fetched:", (await getProduct(baseUrl, created.id)).name);
console.log("listed:", (await listProducts(baseUrl, { limit: 5 })).map((product) => product.name));
try { await getProduct(baseUrl, 999); } catch (error) { console.log("error:", error.message); }
"""
(PHASE6_DIR / "products_types.d.ts").write_text(
    typescript_declarations(products_openapi) + "\n", encoding="utf-8"
)
(PHASE6_DIR / "products_client.mjs").write_text(
    javascript_client(products_openapi, "products_types.d.ts") + "\n", encoding="utf-8"
)
(PHASE6_DIR / "products_demo.mjs").write_text(CLIENT_DEMO_SOURCE.strip() + "\n", encoding="utf-8")
print((PHASE6_DIR / "products_types.d.ts").read_text(encoding="utf-8"))
if HAS_NODE:
    with uvicorn_server("products_api:app") as base_url:
        node_run = subprocess.run(
            ["node", "products_demo.mjs", base_url],
            cwd=PHASE6_DIR,
            capture_output=True,
            text=True,
            timeout=60,
        )
    print(mask_dates(node_run.stdout + node_run.stderr))
else:
    print("SKIP the Node demo: the node command is not installed")

// Generated from Products API 1.2.0. Do not edit.
export interface ErrorMessage {
  detail: string;
}
export interface HTTPValidationError {
  detail?: ValidationError[];
}
export interface ProductCreate {
  name: string;
  price: number | string;
  tags?: string[];
}
export interface ProductRead {
  name: string;
  price: string;
  tags?: string[];
  id: number;
}
export interface ValidationError {
  loc: (string | number)[];
  msg: string;
  type: string;
  input?: unknown;
  ctx?: Record<string, unknown>;
}



created: { name: 'Clay mug', price: '199.00', tags: [ 'mug' ], id: 2 }
fetched: Clay mug
listed: [ 'Assam tea', 'Clay mug' ]
error: 404 {"detail":"product 999 not found"}



The generated client file, for reference (in a real project it lives in the
frontend repository and is regenerated in CI whenever `/openapi.json` changes):

In [153]:
print((PHASE6_DIR / "products_client.mjs").read_text(encoding="utf-8")[:1500])

// Generated client for Products API. Do not edit.
async function request(method, url, body) {
  const headers = { "content-type": "application/json" };
  const payload = body === undefined ? undefined : JSON.stringify(body);
  const response = await fetch(url, { method, headers, body: payload });
  if (!response.ok) throw new Error(`${response.status} ${await response.text()}`);
  return response.status === 204 ? null : response.json();
}
/** Health. @returns {Promise<Record<string, string>>} */
export async function health(baseUrl) {
  const url = new URL(`${baseUrl}/health`);
  return request('GET', url, undefined);
}
/** List products. @returns {Promise<import("./products_types.d.ts").ProductRead[]>} */
export async function listProducts(baseUrl, query = {}) {
  const url = new URL(`${baseUrl}/products`);
  for (const [key, value] of Object.entries(query)) {
    for (const item of [value].flat()) url.searchParams.append(key, item);
  }
  return request('GET', url, undefined);
}
/**

### 6.2.10 How FastAPI uses Python

Nothing in FastAPI is magic; it is the introspection tools from Phases 1–3:

1. **Decorators** (Unit 1.12): `app.get(path)` returns a decorator that creates an
   `APIRoute` for your function, stores it, and returns the function **unchanged**.
2. **`inspect.signature`** (Unit 3.x, 4.2): FastAPI reads every parameter's name,
   default and annotation.
3. **`Annotated`** (PEP 593): extra metadata (`Query(ge=1)`, `Body(embed=True)`)
   rides along in the annotation without changing the type that type checkers see.
4. **Classification**: in the path template → path; a model → body; a simple type →
   query; explicit markers win. The result is a `Dependant` object, solved per
   request.
5. **Pydantic** builds a validator for each parameter and for the response model,
   once, when the route is created; each request only runs them.

Steps 1 and 2. The decorator gave back the same function object, and the route keeps
a reference to it:

In [154]:
introspect_app = FastAPI()


async def place_order_v2(
    shop_id: int,
    order: NewOrder,
    dry_run: bool = False,
    limit: Annotated[int, Query(ge=1, le=100)] = 20,
    user_agent: Annotated[str | None, fastapi.Header()] = None,
) -> dict[str, object]:
    """An endpoint with every kind of parameter."""
    return {}


decorated = introspect_app.post("/shops/{shop_id}/orders")(place_order_v2)
route = introspect_app.routes[-1]
print(
    f"{decorated is place_order_v2=}  {type(route).__name__=}  {route.endpoint is place_order_v2=}"
)
for name, parameter in inspect.signature(place_order_v2).parameters.items():
    default = (
        "(required)" if parameter.default is inspect.Parameter.empty else repr(parameter.default)
    )
    print(f"  {name:<10} {parameter.annotation!s:<60.60} default={default}")

decorated is place_order_v2=True  type(route).__name__='APIRoute'  route.endpoint is place_order_v2=True
  shop_id    <class 'int'>                                                default=(required)
  order      <class '__main__.NewOrder'>                                  default=(required)
  dry_run    <class 'bool'>                                               default=False
  limit      typing.Annotated[int, Query(PydanticUndefined)]              default=20
  user_agent typing.Annotated[str | None, Header(PydanticUndefined)]      default=None


Step 3: `Annotated[int, Query(ge=1, le=100)]` is still `int` to a type checker;
`typing.get_args` unpacks the metadata FastAPI reads. *New in 3.14* (PEP 649/749):
annotations are evaluated lazily, and `annotationlib.get_annotations` can read them
even when they mention names that do not exist yet (`Format.FORWARDREF`); FastAPI
and Pydantic use the same machinery:

In [155]:
import annotationlib

limit_annotation = typing.get_type_hints(place_order_v2, include_extras=True)["limit"]
base_type, *metadata = typing.get_args(limit_annotation)
print(f"{typing.get_origin(limit_annotation) is Annotated=}  {base_type=}  {metadata=}")


def handler_with_future_type(note: NoteNotDefinedYet) -> None:  # noqa: F821
    """Its annotation names a class that does not exist (yet)."""


print(raised_error_name(lambda: annotationlib.get_annotations(handler_with_future_type)))
forward = annotationlib.get_annotations(
    handler_with_future_type, format=annotationlib.Format.FORWARDREF
)
print(re.sub(r" at 0x[0-9a-f]+", "", repr(forward)))

typing.get_origin(limit_annotation) is Annotated=True  base_type=<class 'int'>  metadata=[Query(PydanticUndefined)]
NameError
{'note': ForwardRef('NoteNotDefinedYet', owner=<function handler_with_future_type>), 'return': None}


Step 4: FastAPI's classification, read from the route's `Dependant` (an internal
object: the attribute names can change between versions):

In [156]:
dependant = route.dependant
for kind in ("path_params", "query_params", "header_params", "cookie_params", "body_params"):
    print(f"{kind:<14} {[param.name for param in getattr(dependant, kind)]}")
print(f"{route.response_model=}  {route.status_code=}  {sorted(route.methods)=}")

path_params    ['shop_id']
query_params   ['dry_run', 'limit']
header_params  ['user_agent']
cookie_params  []
body_params    ['order']
route.response_model=dict[str, object]  route.status_code=None  sorted(route.methods)=['POST']


The same rules in a dozen lines, the annotation-driven injector idea from Unit 1.12.11
grown up. It gets the same answer for this endpoint:

In [157]:
def classify_parameters(path: str, endpoint: Callable[..., object]) -> dict[str, list[str]]:
    """Decide where each parameter comes from, roughly as FastAPI does."""
    path_names = set(re.findall(r"\{(\w+)", path))
    sources: dict[str, list[str]] = {"path": [], "query": [], "header": [], "body": []}
    for name, annotation in typing.get_type_hints(endpoint, include_extras=True).items():
        if name == "return":
            continue
        markers = (
            typing.get_args(annotation)[1:] if typing.get_origin(annotation) is Annotated else ()
        )
        base = typing.get_args(annotation)[0] if markers else annotation
        if name in path_names:
            sources["path"].append(name)
        elif any(isinstance(marker, fastapi.params.Header) for marker in markers):
            sources["header"].append(name)
        elif inspect.isclass(base) and issubclass(base, BaseModel):
            sources["body"].append(name)
        else:
            sources["query"].append(name)
    return sources


print(classify_parameters("/shops/{shop_id}/orders", place_order_v2))

{'path': ['shop_id'], 'query': ['dry_run', 'limit'], 'header': ['user_agent'], 'body': ['order']}


Step 5: the validators are Pydantic objects built at startup. The body parameter's
field carries a `TypeAdapter`-like validator for `NewOrder`; the response model is
validated the same way. Each request just calls them:

In [158]:
body_field = dependant.body_params[0]
print(f"{body_field.name=}  {body_field.field_info.annotation=}")
print(f"{type(NewOrder.__pydantic_validator__).__name__=}")
print(NewOrder.__pydantic_validator__.validate_json(b'{"sku": "TEA-500", "quantity": 2}'))
print(f"{route.dependant.call is place_order_v2=}")

body_field.name='order'  body_field.field_info.annotation=<class '__main__.NewOrder'>
type(NewOrder.__pydantic_validator__).__name__='SchemaValidator'
sku='TEA-500' quantity=2 gift_note=None
route.dependant.call is place_order_v2=True


### Predict the output — Unit 6.2

Three questions on routing, PATCH semantics and lax parsing.

**Predict the output.**

```python
app = FastAPI()

@app.get("/users/{user_id}")
async def read_user(user_id: int) -> dict[str, int]:
    return {"user_id": user_id}

@app.get("/users/me")
async def read_me() -> dict[str, str]:
    return {"user": "me"}

print(TestClient(app).get("/users/me").status_code)
```

<details><summary>Answer</summary>

`422`. Routes are tried in declaration order, `/users/{user_id}` matches the path
`/users/me` first, and `"me"` fails the `int` conversion. Declare fixed paths before
parameterized ones (6.2.2).

</details>

Run the next cell to check your prediction.

In [159]:
predict_app = FastAPI()


@predict_app.get("/users/{user_id}")
async def predict_read_user(user_id: int) -> dict[str, int]:
    """Declared first, so it wins."""
    return {"user_id": user_id}


@predict_app.get("/users/me")
async def predict_read_me() -> dict[str, str]:
    """Never reached for /users/me."""
    return {"user": "me"}


print(TestClient(predict_app).get("/users/me").status_code)

422


**Predict the output.**

```python
changes = Preferences.model_validate({"nickname": None})
print(changes.model_dump(exclude_unset=True))
print(changes.model_dump(exclude_none=True))
```

<details><summary>Answer</summary>

`{'nickname': None}` and then `{'theme': 'light', 'page_size': 20}`.
`exclude_unset` keeps exactly what the client sent, including an explicit `null`
(so a PATCH can clear a field); `exclude_none` drops every `None`, losing the
client's intent, and keeps defaults the client never sent.

</details>

Run the next cell to check your prediction.

In [160]:
changes = Preferences.model_validate({"nickname": None})
print(changes.model_dump(exclude_unset=True))
print(changes.model_dump(exclude_none=True))

{'nickname': None}
{'theme': 'light', 'page_size': 20}


**Predict the output.**

```python
class Price(BaseModel):
    amount: Decimal
    count: int

price = Price(amount="12.50", count=True)
print(price.count, price.model_dump_json())
```

<details><summary>Answer</summary>

`1 {"amount":"12.50","count":1}`. Lax mode converts `True` to `1` (a bool is a
lossless int), and JSON mode writes `Decimal` as a **string** so no precision is
lost; JavaScript clients must not parse it as a float. Use `Field(strict=True)` to
refuse booleans.

</details>

Run the next cell to check your prediction.

In [161]:
class Price(BaseModel):
    """Lax int, Decimal amount."""

    amount: Decimal
    count: int


price = Price(amount="12.50", count=True)
print(price.count, price.model_dump_json())

1 {"amount":"12.50","count":1}


### Lab 6.2

Five exercises: the first version of a notes API, its filters, a registration
model, PATCH, and a benchmark. Every check uses `TestClient` or
`httpx.ASGITransport` (no server, no network). Replace each `raise
NotImplementedError` (or `pass`), then run the check cell; until then the checks
print FAIL, which is expected. After the lab, *Notes API v1* writes the reference
version into `projects/notes_api/`; do the lab first.

#### Exercise 1 — Notes API v1: CRUD with Create / Update / Read schemas

Define three schemas and `create_notes_app()`:

- `NoteCreate`: `title` (stripped, 1–100 characters), `body` (at most 1000
  characters, default `""`), `pinned` (default `False`); unknown keys → 422.
- `NoteUpdate`: the same three fields, **all optional** (default `None`); unknown
  keys → 422. (Exercise 4 uses the idea.)
- `NoteRead`: `id`, `title`, `body`, `pinned`, `created_at` (a `datetime`).
- `create_notes_app()` returns a **new** `FastAPI` app with its own empty in-memory
  store on every call (ids start at 1):

| Route | Behaviour |
| --- | --- |
| `POST /notes` | `NoteCreate` body → 201, `NoteRead` JSON, `Location: /notes/<id>` |
| `GET /notes` | 200, every note as `NoteRead`, in id order |
| `GET /notes/{note_id}` | 200 `NoteRead`; unknown id → 404 `{"detail": "note <id> not found"}` |
| `PUT /notes/{note_id}` | `NoteCreate` body replaces every field (omitted ones get their defaults) → 200 |
| `DELETE /notes/{note_id}` | 204 with an empty body; unknown id → 404 |

Your solution:

In [162]:
class NoteCreate(BaseModel):
    """POST/PUT body: title (stripped, 1-100), body (max 1000, default ""), pinned."""


class NoteUpdate(BaseModel):
    """PATCH body: the same fields, all optional; unknown keys rejected."""


class NoteRead(BaseModel):
    """Response: id, title, body, pinned, created_at."""


def create_notes_app() -> FastAPI:
    """Notes API v1: a fresh app with its own empty in-memory store."""
    raise NotImplementedError

Check exercise 1:

In [163]:
def notes_v1_reads(client: TestClient, results: dict[str, object]) -> None:
    """Create two notes, then read them back (fills results)."""
    first = client.post("/notes", json={"title": "  Buy tea  ", "pinned": True})
    first_note = {key: value for key, value in first.json().items() if key != "created_at"}
    results["create"] = (first.status_code, first.headers.get("location"), first_note)
    results["created_at"] = type(datetime.fromisoformat(first.json()["created_at"])).__name__
    client.post("/notes", json={"title": "Call Ravi", "body": "about the trip"})
    results["list"] = [note["title"] for note in client.get("/notes").json()]
    results["get"] = client.get("/notes/2").json()["body"]
    results["missing"] = (client.get("/notes/99").status_code, client.get("/notes/99").json())
    results["bad id"] = client.get("/notes/abc").status_code
    bad_bodies = [
        {"title": "   "},
        {"title": "x" * 101},
        {"title": "ok", "color": "red"},
        {"body": "x"},
    ]
    results["invalid"] = [client.post("/notes", json=body).status_code for body in bad_bodies]


def notes_v1_writes(client: TestClient, results: dict[str, object]) -> None:
    """Replace and delete note 1 (fills results)."""
    replaced = client.put("/notes/1", json={"title": "Buy green tea"})
    results["put"] = (replaced.status_code, replaced.json()["title"], replaced.json()["pinned"])
    results["put missing"] = client.put("/notes/99", json={"title": "x"}).status_code
    deleted = client.delete("/notes/1")
    gone = client.get("/notes/1").status_code
    results["delete"] = (
        deleted.status_code,
        deleted.content,
        gone,
        client.delete("/notes/1").status_code,
    )

Run the scenario on a fresh app (and check that a second app starts empty):

In [164]:
notes_v1: dict[str, object] = {}
try:
    with TestClient(create_notes_app()) as client:
        notes_v1_reads(client, notes_v1)
        notes_v1_writes(client, notes_v1)
    with TestClient(create_notes_app()) as fresh_client:
        notes_v1["fresh store"] = fresh_client.get("/notes").json()
except Exception as error:
    notes_v1["error"] = f"{type(error).__name__}: {error}"
print(notes_v1.get("error", "scenario ran"))

NotImplementedError: 


Check the endpoints:

In [165]:
run_checks(
    [
        (
            "POST: 201, Location, stripped title",
            lambda: notes_v1["create"],
            (201, "/notes/1", {"id": 1, "title": "Buy tea", "body": "", "pinned": True}),
        ),
        ("created_at parses as a datetime", lambda: notes_v1["created_at"], "datetime"),
        ("GET /notes in id order", lambda: notes_v1["list"], ["Buy tea", "Call Ravi"]),
        ("GET /notes/2", lambda: notes_v1["get"], "about the trip"),
        (
            "unknown id -> 404 detail",
            lambda: notes_v1["missing"],
            (404, {"detail": "note 99 not found"}),
        ),
        ("non-integer id -> 422", lambda: notes_v1["bad id"], 422),
        ("invalid bodies -> 422", lambda: notes_v1["invalid"], [422, 422, 422, 422]),
        ("PUT replaces every field", lambda: notes_v1["put"], (200, "Buy green tea", False)),
        ("PUT unknown id -> 404", lambda: notes_v1["put missing"], 404),
        ("DELETE: 204 empty, then 404", lambda: notes_v1["delete"], (204, b"", 404, 404)),
        ("each app has its own store", lambda: notes_v1["fresh store"], []),
    ]
)

FAIL  POST: 201, Location, stripped title: raised KeyError: 'create'
FAIL  created_at parses as a datetime: raised KeyError: 'created_at'
FAIL  GET /notes in id order: raised KeyError: 'list'
FAIL  GET /notes/2: raised KeyError: 'get'
FAIL  unknown id -> 404 detail: raised KeyError: 'missing'
FAIL  non-integer id -> 422: raised KeyError: 'bad id'
FAIL  invalid bodies -> 422: raised KeyError: 'invalid'
FAIL  PUT replaces every field: raised KeyError: 'put'
FAIL  PUT unknown id -> 404: raised KeyError: 'put missing'
FAIL  DELETE: 204 empty, then 404: raised KeyError: 'delete'
FAIL  each app has its own store: raised KeyError: 'fresh store'
0/11 checks passed


Check the schemas:

In [166]:
run_checks(
    [
        (
            "NoteUpdate: three optional fields",
            lambda: (NoteUpdate().model_dump(exclude_unset=True), sorted(NoteUpdate.model_fields)),
            ({}, ["body", "pinned", "title"]),
        ),
        (
            "NoteUpdate forbids extras",
            lambda: validation_problems(lambda: NoteUpdate(color="red")),
            [("color", "extra_forbidden")],
        ),
        (
            "NoteRead fields",
            lambda: sorted(NoteRead.model_fields),
            ["body", "created_at", "id", "pinned", "title"],
        ),
    ]
)

FAIL  NoteUpdate: three optional fields: expected ({}, ['body', 'pinned', 'title']), got ({}, [])
FAIL  NoteUpdate forbids extras: expected [('color', 'extra_forbidden')], got 'no error'
FAIL  NoteRead fields: expected ['body', 'created_at', 'id', 'pinned', 'title'], got []
0/3 checks passed


#### Exercise 2 — validated filtering and pagination parameters

Write `NoteFilters`, a query-parameter model, and `filter_notes(notes, filters)`:

- `q`: optional text, at most 50 characters; matches notes whose title **or** body
  contains it, ignoring case.
- `pinned`: optional bool; when given, only notes with that value.
- `sort`: one of `"id"`, `"-id"`, `"title"`, `"-title"` (default `"id"`); titles
  compare case-insensitively; `-` means descending.
- `limit`: 1–50, default 10. `offset`: ≥ 0, default 0.
- Unknown query parameters → 422 (`extra="forbid"`).
- `filter_notes(notes, filters)` returns `{"items": [...], "total": N, "limit": ...,
  "offset": ...}`, where `total` counts **all** matches before pagination.

The check cell serves `SAMPLE_NOTES` through a given endpoint,
`GET /notes` with `filters: Annotated[NoteFilters, Query()]`.

In [167]:
SAMPLE_NOTES = [
    {"id": 1, "title": "Buy tea", "body": "Assam and Darjeeling", "pinned": True},
    {"id": 2, "title": "call Ravi", "body": "about the trip", "pinned": False},
    {"id": 3, "title": "Book tickets", "body": "Pune to Goa, December", "pinned": True},
    {"id": 4, "title": "Water plants", "body": "", "pinned": False},
    {"id": 5, "title": "Tea tasting notes", "body": "Nilgiri was brighter", "pinned": False},
    {"id": 6, "title": "Pay electricity bill", "body": "due on the 20th", "pinned": True},
    {"id": 7, "title": "Trip budget", "body": "tickets, hotel, tea estates", "pinned": False},
    {"id": 8, "title": "Read FastAPI docs", "body": "dependencies chapter", "pinned": False},
]

Your solution:

In [168]:
class NoteFilters(BaseModel):
    """Query parameters for GET /notes: q, pinned, sort, limit, offset; extras rejected."""


def filter_notes(notes: list[dict[str, Any]], filters: NoteFilters) -> dict[str, object]:
    """Filter, sort and paginate notes; total counts matches before pagination."""
    raise NotImplementedError

Check exercise 2:

In [169]:
filters_app = FastAPI()


@filters_app.get("/notes")
async def search_sample_notes(filters: Annotated[NoteFilters, Query()]) -> dict[str, object]:
    """Given endpoint: apply the learner's model and function to SAMPLE_NOTES."""
    return filter_notes(SAMPLE_NOTES, filters)


def filters_page(query: str) -> tuple[int, dict[str, Any]]:
    """Status code and JSON body of GET /notes?query."""
    with TestClient(filters_app, raise_server_exceptions=False) as client:
        response = client.get(f"/notes?{query}")
    return response.status_code, response.json() if response.status_code != 500 else {}


def page_ids(query: str) -> object:
    """(ids on the page, total), or the status code if the request failed."""
    status_code, page = filters_page(query)
    if status_code != 200:
        return status_code
    return [note["id"] for note in page["items"]], page["total"]

Check filtering, sorting and paging:

In [170]:
run_checks(
    [
        ("defaults: first 8 by id", lambda: page_ids(""), ([1, 2, 3, 4, 5, 6, 7, 8], 8)),
        ("limit and offset", lambda: page_ids("limit=3&offset=3"), ([4, 5, 6], 8)),
        ("q matches title or body, any case", lambda: page_ids("q=TEA"), ([1, 5, 7], 3)),
        ("pinned=true", lambda: page_ids("pinned=true"), ([1, 3, 6], 3)),
        ("sort=-id with limit", lambda: page_ids("sort=-id&limit=2"), ([8, 7], 8)),
        ("sort=title ignores case", lambda: page_ids("sort=title&limit=3"), ([3, 1, 2], 8)),
        ("total counts before paging", lambda: page_ids("q=tea&limit=1&offset=1"), ([5], 3)),
        (
            "page echoes limit/offset",
            lambda: filters_page("limit=2&offset=4")[1] | {"items": None},
            {"items": None, "total": 8, "limit": 2, "offset": 4},
        ),
    ]
)

FAIL  defaults: first 8 by id: expected ([1, 2, 3, 4, 5, 6, 7, 8], 8), got 500
FAIL  limit and offset: expected ([4, 5, 6], 8), got 500
FAIL  q matches title or body, any case: expected ([1, 5, 7], 3), got 500
FAIL  pinned=true: expected ([1, 3, 6], 3), got 500
FAIL  sort=-id with limit: expected ([8, 7], 8), got 500
FAIL  sort=title ignores case: expected ([3, 1, 2], 8), got 500


FAIL  total counts before paging: expected ([5], 3), got 500
FAIL  page echoes limit/offset: expected {'items': None, 'total': 8, 'limit': 2, 'offset': 4}, got {'items': None}
0/8 checks passed


Check the validation of the parameters:

In [171]:
run_checks(
    [
        ("limit=0 -> 422", lambda: page_ids("limit=0"), 422),
        ("limit=51 -> 422", lambda: page_ids("limit=51"), 422),
        ("offset=-1 -> 422", lambda: page_ids("offset=-1"), 422),
        ("sort=newest -> 422", lambda: page_ids("sort=newest"), 422),
        ("q longer than 50 -> 422", lambda: page_ids("q=" + "x" * 51), 422),
        ("unknown parameter -> 422", lambda: page_ids("colour=red"), 422),
    ]
)

FAIL  limit=0 -> 422: expected 422, got 500
FAIL  limit=51 -> 422: expected 422, got 500
FAIL  offset=-1 -> 422: expected 422, got 500


FAIL  sort=newest -> 422: expected 422, got 500
FAIL  q longer than 50 -> 422: expected 422, got 500
FAIL  unknown parameter -> 422: expected 422, got 500
0/6 checks passed


#### Exercise 3 — a `UserRegistration` model with cross-field validation

Write `UserRegistration` (unknown keys rejected):

- `username`: stripped and lower-cased, then 3–20 characters of `a-z`, `0-9`, `_`.
- `email`: an `EmailStr`.
- `password`: at least 12 characters, with at least one letter **and** one digit.
- `password_confirm`: must equal `password` (error message contains
  `passwords do not match`); **excluded** from `model_dump()`.
- `birth_date` and `signup_date` (both `date`): the user must be at least 13 years
  old on `signup_date` (message contains `at least 13`).
- `accept_terms`: must be `True`.

The checks look at error **locations** (where the problem is), not the exact error
types, so any correct validator style passes.

Your solution:

In [172]:
class UserRegistration(BaseModel):
    """Sign-up form with field rules and cross-field rules (see the exercise text)."""

    @model_validator(mode="before")
    @classmethod
    def not_written_yet(cls, data: object) -> object:
        """Placeholder: delete it when you add the fields and validators."""
        raise NotImplementedError

Check exercise 3:

In [173]:
VALID_SIGNUP = {
    "username": "  Asha_R ",
    "email": "asha@Example.com",
    "password": "chai4ever-and-more",
    "password_confirm": "chai4ever-and-more",
    "birth_date": "2000-05-17",
    "signup_date": "2026-11-17",
    "accept_terms": True,
}


def signup_error_locations(**changes: object) -> object:
    """Locations of the validation errors for VALID_SIGNUP with changes applied."""
    problems = validation_problems(
        lambda: UserRegistration.model_validate({**VALID_SIGNUP, **changes})
    )
    return problems if isinstance(problems, str) else [location for location, _type in problems]


def signup_messages(**changes: object) -> str:
    """All error messages for VALID_SIGNUP with changes applied, joined."""
    try:
        UserRegistration.model_validate({**VALID_SIGNUP, **changes})
    except ValidationError as error:
        return " | ".join(problem["msg"] for problem in error.errors())
    return "no error"


def signup_dump(**changes: object) -> dict[str, Any]:
    """model_dump(mode="json") of VALID_SIGNUP with changes applied."""
    return UserRegistration.model_validate({**VALID_SIGNUP, **changes}).model_dump(mode="json")

Check the field rules:

In [174]:
run_checks(
    [
        (
            "valid registration dumps cleanly",
            signup_dump,
            {
                "username": "asha_r",
                "email": "asha@example.com",
                "password": "chai4ever-and-more",
                "birth_date": "2000-05-17",
                "signup_date": "2026-11-17",
                "accept_terms": True,
            },
        ),
        (
            "username with a space",
            lambda: signup_error_locations(username="asha rao"),
            ["username"],
        ),
        ("username too short", lambda: signup_error_locations(username="as"), ["username"]),
        ("bad email", lambda: signup_error_locations(email="asha-at-example"), ["email"]),
        (
            "password without a digit",
            lambda: signup_error_locations(
                password="only-letters-here", password_confirm="only-letters-here"
            ),
            ["password"],
        ),
        (
            "password too short",
            lambda: signup_error_locations(password="abc123", password_confirm="abc123"),
            ["password"],
        ),
    ]
)

FAIL  valid registration dumps cleanly: raised NotImplementedError: 
FAIL  username with a space: raised NotImplementedError: 
FAIL  username too short: raised NotImplementedError: 
FAIL  bad email: raised NotImplementedError: 
FAIL  password without a digit: raised NotImplementedError: 
FAIL  password too short: raised NotImplementedError: 
0/6 checks passed


Check the remaining rules, including the cross-field ones:

In [175]:
run_checks(
    [
        (
            "terms must be accepted",
            lambda: signup_error_locations(accept_terms=False),
            ["accept_terms"],
        ),
        ("unknown key rejected", lambda: signup_error_locations(is_admin=True), ["is_admin"]),
        (
            "passwords differ",
            lambda: (
                "passwords do not match" in signup_messages(password_confirm="chai4ever-and-less")
            ),
            True,
        ),
        (
            "13th birthday is the cutoff",
            lambda: signup_dump(birth_date="2013-11-17")["birth_date"],
            "2013-11-17",
        ),
        (
            "one day too young",
            lambda: "at least 13" in signup_messages(birth_date="2013-11-18"),
            True,
        ),
    ]
)

FAIL  terms must be accepted: raised NotImplementedError: 
FAIL  unknown key rejected: raised NotImplementedError: 
FAIL  passwords differ: raised NotImplementedError: 
FAIL  13th birthday is the cutoff: raised NotImplementedError: 
FAIL  one day too young: raised NotImplementedError: 
0/5 checks passed


#### Exercise 4 — a PATCH endpoint

Write `NotePatch` and `create_patch_app(seed)`. The app keeps a **copy** of `seed`
(a dict of id → note dict with `id`, `title`, `body`, `pinned`, `version`) and has:

- `GET /notes/{note_id}` → the note (404 `{"detail": "note <id> not found"}` if
  unknown);
- `PATCH /notes/{note_id}` with a `NotePatch` body → the updated note. Only the
  fields the client **sent** change (`exclude_unset`). `version` goes up by one only
  when a value actually changed (`{}` or the same values leave it alone).

`NotePatch` has optional `title` (stripped, 1–100), `body` (≤ 1000) and `pinned`;
unknown keys → 422; and an explicit `null` for any field → 422 (none of them is
nullable).

Your solution:

In [176]:
class NotePatch(BaseModel):
    """PATCH body: optional title/body/pinned; explicit null and unknown keys rejected."""


def create_patch_app(seed: dict[int, dict[str, Any]]) -> FastAPI:
    """GET and PATCH /notes/{note_id} over a copy of seed."""
    raise NotImplementedError

Check exercise 4:

In [177]:
PATCH_SEED = {1: {"id": 1, "title": "Buy tea", "body": "Assam", "pinned": False, "version": 1}}


def patch_scenario() -> dict[str, Any]:
    """A fixed sequence of PATCH requests against a fresh app."""
    results: dict[str, Any] = {}
    with TestClient(create_patch_app(PATCH_SEED)) as client:
        results["pin"] = client.patch("/notes/1", json={"pinned": True}).json()
        results["retitle"] = client.patch("/notes/1", json={"title": "  Buy green tea "}).json()
        results["empty"] = client.patch("/notes/1", json={}).json()["version"]
        results["same value"] = client.patch("/notes/1", json={"pinned": True}).json()["version"]
        bad_bodies = [{"title": None}, {"pinned": None}, {"title": ""}, {"color": "red"}]
        results["errors"] = [client.patch("/notes/1", json=body).status_code for body in bad_bodies]
        missing = client.patch("/notes/99", json={"pinned": True}).status_code
        results["missing"] = (missing, client.get("/notes/99").json())
        results["get"] = client.get("/notes/1").json()
    return results


try:
    patched = patch_scenario()
except Exception as error:
    patched = {"error": f"{type(error).__name__}: {error}"}
print(patched.get("error", "scenario ran"))

NotImplementedError: 


Check exercise 4:

In [178]:
run_checks(
    [
        (
            "only pinned changes, version 2",
            lambda: patched["pin"],
            {"id": 1, "title": "Buy tea", "body": "Assam", "pinned": True, "version": 2},
        ),
        (
            "title stripped, version 3",
            lambda: tuple(patched["retitle"][key] for key in ("title", "body", "version")),
            ("Buy green tea", "Assam", 3),
        ),
        ("{} changes nothing", lambda: patched["empty"], 3),
        ("same value changes nothing", lambda: patched["same value"], 3),
        ("null, empty title, unknown key -> 422", lambda: patched["errors"], [422, 422, 422, 422]),
        ("unknown id -> 404", lambda: patched["missing"], (404, {"detail": "note 99 not found"})),
        ("GET sees the changes", lambda: patched["get"]["title"], "Buy green tea"),
        (
            "seed was not modified",
            lambda: (patched["get"]["version"], PATCH_SEED[1]["version"]),
            (3, 1),
        ),
    ]
)

FAIL  only pinned changes, version 2: raised KeyError: 'pin'
FAIL  title stripped, version 3: raised KeyError: 'retitle'
FAIL  {} changes nothing: raised KeyError: 'empty'
FAIL  same value changes nothing: raised KeyError: 'same value'
FAIL  null, empty title, unknown key -> 422: raised KeyError: 'errors'
FAIL  unknown id -> 404: raised KeyError: 'missing'
FAIL  GET sees the changes: raised KeyError: 'get'
FAIL  seed was not modified: raised KeyError: 'get'
0/8 checks passed


#### Exercise 5 — a blocking-vs-async benchmark

Write two functions:

- `async benchmark_endpoints(app, paths, request_count=8)`: for each path, send
  `request_count` **concurrent** GET requests to `app` through
  `httpx.AsyncClient(transport=httpx.ASGITransport(app=app), base_url="http://bench")`
  and measure the wall time; return `{path: seconds}` in the order of `paths`.
- `diagnose(timings, delay, request_count)`: for each path, `"serialized"` if its
  time is at least `0.75 * delay * request_count` (the requests ran one after
  another), else `"concurrent"`.

The check runs them on `bench_app`, whose three endpoints each wait `BENCH_DELAY`
seconds in a different way, then checks `diagnose` on its own.

In [179]:
BENCH_DELAY = 0.05
bench_app = FastAPI()


@bench_app.get("/async-blocking")
async def bench_async_blocking() -> dict[str, bool]:
    """Blocks the event loop."""
    time.sleep(BENCH_DELAY)
    return {"ok": True}


@bench_app.get("/sync-blocking")
def bench_sync_blocking() -> dict[str, bool]:
    """Blocks a thread-pool thread."""
    time.sleep(BENCH_DELAY)
    return {"ok": True}


@bench_app.get("/async-sleep")
async def bench_async_sleep() -> dict[str, bool]:
    """Waits without blocking."""
    await asyncio.sleep(BENCH_DELAY)
    return {"ok": True}

Your solution:

In [180]:
async def benchmark_endpoints(
    app: FastAPI, paths: list[str], request_count: int = 8
) -> dict[str, float]:
    """Wall seconds for request_count concurrent GETs of each path, in order."""
    raise NotImplementedError


def diagnose(timings: dict[str, float], delay: float, request_count: int) -> dict[str, str]:
    """ "serialized" if a path took at least 0.75 * delay * request_count, else "concurrent"."""
    raise NotImplementedError

Check exercise 5:

In [181]:
bench_paths = ["/async-blocking", "/sync-blocking", "/async-sleep"]
try:
    bench_timings = await benchmark_endpoints(bench_app, bench_paths, request_count=8)
except Exception as error:
    bench_timings = {"error": f"{type(error).__name__}"}
if all(isinstance(seconds, float) for seconds in bench_timings.values()):
    print({path: f"{seconds:.3f} s" for path, seconds in bench_timings.items()})

Check exercise 5:

In [182]:
run_checks(
    [
        ("one timing per path, in order", lambda: list(bench_timings), bench_paths),
        (
            "timings are positive floats",
            lambda: all(
                isinstance(seconds, float) and seconds > 0 for seconds in bench_timings.values()
            ),
            True,
        ),
        (
            "async-blocking serializes (>= 0.3 s)",
            lambda: bench_timings["/async-blocking"] >= 0.3,
            True,
        ),
        (
            "the others overlap (< 0.2 s)",
            lambda: bench_timings["/sync-blocking"] < 0.2 and bench_timings["/async-sleep"] < 0.2,
            True,
        ),
        (
            "diagnose the measurements",
            lambda: diagnose(bench_timings, BENCH_DELAY, 8),
            {
                "/async-blocking": "serialized",
                "/sync-blocking": "concurrent",
                "/async-sleep": "concurrent",
            },
        ),
        (
            "diagnose: either side of 0.3 s",
            lambda: diagnose({"/a": 0.31, "/b": 0.29}, 0.05, 8),
            {"/a": "serialized", "/b": "concurrent"},
        ),
    ]
)

FAIL  one timing per path, in order: expected ['/async-blocking', '/sync-blocking', '/async-sleep'], got ['error']
FAIL  timings are positive floats: expected True, got False
FAIL  async-blocking serializes (>= 0.3 s): raised KeyError: '/async-blocking'
FAIL  the others overlap (< 0.2 s): raised KeyError: '/sync-blocking'
FAIL  diagnose the measurements: raised NotImplementedError: 
FAIL  diagnose: either side of 0.3 s: raised NotImplementedError: 
0/6 checks passed


### Notes API v1 — the project files

Unit 6.2's techniques, assembled into the application the rest of Phase 6 grows.
The cells below **write the files** of `projects/notes_api/` (compare them with your
Lab 6.2 answers). It is a uv project of its own from the start, so it can get its
own lock file now and a container image in Unit 6.8:

```text
projects/notes_api/
├── pyproject.toml        # dependencies: fastapi[standard]; dev: pytest
├── uv.lock               # created by `uv lock` below
├── ruff.toml             # extends the course's ruff settings
├── README.md
├── src/notes_api/
│   ├── __init__.py       # __version__
│   ├── schemas.py        # NoteCreate / NoteUpdate / NoteRead / NoteFilters / NotePage
│   ├── store.py          # NoteStore: in-memory, replaced by SQLAlchemy in 6.4
│   └── main.py           # the FastAPI app and its path operations
└── tests/
    ├── conftest.py       # `client` fixture: TestClient on an empty store
    └── test_notes.py
```

The course's own environment (the notebook kernel) has the same packages, so the
notebook can import `notes_api` from `src/` directly.

First the project metadata. `pythonpath = ["src"]` lets pytest import the package
even without installing it:

In [183]:
%%writefile ../projects/notes_api/pyproject.toml
[project]
name = "notes-api"
version = "0.1.0"
description = "The Phase 6 Notes API: a FastAPI service built step by step from Unit 6.2 to 6.8."
readme = "README.md"
requires-python = ">=3.15"
dependencies = [
    "fastapi[standard]>=0.163.1",
]

[dependency-groups]
dev = [
    "pytest>=9.2.0",
]

[build-system]
requires = ["uv_build>=0.13,<0.14"]
build-backend = "uv_build"

[tool.pytest.ini_options]
testpaths = ["tests"]
pythonpath = ["src"]
addopts = "-q"

Overwriting ../projects/notes_api/pyproject.toml


Ruff settings (inherit the course configuration) and the README:

In [184]:
%%writefile ../projects/notes_api/ruff.toml
# Extends the course configuration; notes_api (src/) and the tests are first-party imports.
extend = "../../pyproject.toml"
src = ["src", "tests"]

Overwriting ../projects/notes_api/ruff.toml


In [185]:
%%writefile ../projects/notes_api/README.md
# Notes API

A FastAPI service for creating, searching and organizing notes. It is built step by step in
Phase 6 of the course (`notebooks/phase6_web_fastapi.ipynb`); the notebook writes these files.

| Version | Unit | Adds |
| --- | --- | --- |
| v1 | 6.2 | CRUD path operations, Pydantic v2 schemas, filtering and pagination, OpenAPI |

## Run

```bash
cd projects/notes_api
uv run uvicorn notes_api.main:app --reload   # creates .venv; http://127.0.0.1:8000/docs
uv run pytest
```

## Endpoints

| Method | Path | Success | Notes |
| --- | --- | --- | --- |
| `GET` | `/health` | 200 | liveness probe |
| `POST` | `/notes` | 201 | `Location` header; body `NoteCreate` |
| `GET` | `/notes` | 200 | `q`, `tag` (repeatable), `pinned`, `sort`, `limit` (1-100), `offset` |
| `GET` | `/notes/{note_id}` | 200 | 404 if unknown |
| `PUT` | `/notes/{note_id}` | 200 | replaces every field |
| `PATCH` | `/notes/{note_id}` | 200 | only the fields sent; explicit `null` is rejected |
| `DELETE` | `/notes/{note_id}` | 204 | |

## Layout

```text
src/notes_api/
├── __init__.py   # version
├── main.py       # the FastAPI app and its path operations
├── schemas.py    # Pydantic models: NoteCreate, NoteUpdate, NoteRead, NoteFilters, NotePage
└── store.py      # in-memory storage (replaced by a database in Unit 6.4)
tests/            # pytest + TestClient
```

Overwriting ../projects/notes_api/README.md


The package marker with the version that `/openapi.json` reports:

In [186]:
%%writefile ../projects/notes_api/src/notes_api/__init__.py
"""Notes API: a FastAPI service built step by step in Phase 6 of the course."""

__version__ = "0.1.0"

Overwriting ../projects/notes_api/src/notes_api/__init__.py


**`schemas.py`.** Reusable `Annotated` types (`Title`, `Tag`, `Tags`), the three
directions of the note (Create, Update, Read), the query-parameter model and the
page envelope. `NoteUpdate` rejects an explicit `null`, the hole in plain
`exclude_unset` PATCH handling noted in 6.2.6. `NoteRead.word_count` is a computed
field:

Part 1 of `schemas.py`: imports and the slug check for tags.

In [187]:
%%writefile ../projects/notes_api/src/notes_api/schemas.py
"""Request and response schemas for the Notes API (Pydantic v2).

One schema per direction: NoteCreate (POST/PUT body), NoteUpdate (PATCH body, every field
optional), NoteRead (responses, with server-owned fields), NoteFilters (query parameters) and
NotePage (one page of results).
"""

import re
from datetime import datetime
from typing import Annotated, Literal

from pydantic import (
    AfterValidator,
    BaseModel,
    ConfigDict,
    Field,
    StringConstraints,
    computed_field,
    field_validator,
    model_validator,
)

MAX_TAGS = 10


def require_slug(value: str) -> str:
    """Allow lowercase letters, digits and dashes (checked after stripping and lower-casing)."""
    if not re.fullmatch(r"[a-z0-9-]+", value):
        raise ValueError("tags use lowercase letters, digits and dashes")
    return value

Overwriting ../projects/notes_api/src/notes_api/schemas.py


Part 2: reusable `Annotated` types for titles, bodies, tags, sort orders.

In [188]:
%%writefile -a ../projects/notes_api/src/notes_api/schemas.py


def unique_in_order(tags: list[str]) -> list[str]:
    """Drop repeated tags, keeping the first occurrence."""
    return list(dict.fromkeys(tags))


Title = Annotated[str, StringConstraints(strip_whitespace=True, min_length=1, max_length=200)]
Body = Annotated[str, Field(max_length=10_000)]
Tag = Annotated[
    str,
    StringConstraints(strip_whitespace=True, to_lower=True, min_length=1, max_length=30),
    AfterValidator(require_slug),
]
Tags = Annotated[list[Tag], Field(max_length=MAX_TAGS)]
SortOrder = Literal["created_at", "-created_at", "updated_at", "-updated_at", "title", "-title"]

Appending to ../projects/notes_api/src/notes_api/schemas.py


Part 3: `NoteCreate`, the body of POST and PUT.

In [189]:
%%writefile -a ../projects/notes_api/src/notes_api/schemas.py


class NoteCreate(BaseModel):
    """Body of POST /notes and PUT /notes/{note_id}: every field a client may set."""

    model_config = ConfigDict(
        extra="forbid",
        json_schema_extra={
            "examples": [{"title": "Buy tea", "body": "Assam, 500 g", "tags": ["shopping"]}]
        },
    )

    title: Title
    body: Body = ""
    tags: Tags = []
    pinned: bool = False

    @field_validator("tags")
    @classmethod
    def drop_duplicate_tags(cls, tags: list[str]) -> list[str]:
        """Keep each tag once."""
        return unique_in_order(tags)

Appending to ../projects/notes_api/src/notes_api/schemas.py


Part 4: `NoteUpdate`, the PATCH body, which rejects explicit `null`.

In [190]:
%%writefile -a ../projects/notes_api/src/notes_api/schemas.py


class NoteUpdate(BaseModel):
    """Body of PATCH /notes/{note_id}: only the fields being changed."""

    model_config = ConfigDict(extra="forbid", json_schema_extra={"examples": [{"pinned": True}]})

    title: Title | None = None
    body: Body | None = None
    tags: Tags | None = None
    pinned: bool | None = None

    @field_validator("tags")
    @classmethod
    def drop_duplicate_tags(cls, tags: list[str] | None) -> list[str] | None:
        """Keep each tag once."""
        return None if tags is None else unique_in_order(tags)

    @model_validator(mode="after")
    def reject_explicit_null(self) -> NoteUpdate:
        """None means "not sent"; an explicit null is an error (no field is nullable)."""
        nulled = sorted(name for name in self.model_fields_set if getattr(self, name) is None)
        if nulled:
            raise ValueError(f"fields cannot be null: {', '.join(nulled)}")
        return self

Appending to ../projects/notes_api/src/notes_api/schemas.py


Part 5: `NoteRead` (computed `word_count`) and the query model `NoteFilters`.

In [191]:
%%writefile -a ../projects/notes_api/src/notes_api/schemas.py


class NoteRead(BaseModel):
    """A note as the API returns it."""

    model_config = ConfigDict(from_attributes=True)

    id: int
    title: str
    body: str
    tags: list[str]
    pinned: bool
    created_at: datetime
    updated_at: datetime

    @computed_field
    @property
    def word_count(self) -> int:
        """Words in the body (derived, never stored)."""
        return len(self.body.split())


class NoteFilters(BaseModel):
    """Query parameters of GET /notes: filtering, sorting and pagination."""

    model_config = ConfigDict(extra="forbid")

    q: Annotated[str | None, Field(max_length=100, description="Text in title or body.")] = None
    tag: Annotated[list[Tag], Field(description="Only notes that have all these tags.")] = []
    pinned: bool | None = None
    sort: SortOrder = "-created_at"
    limit: Annotated[int, Field(ge=1, le=100)] = 20
    offset: Annotated[int, Field(ge=0)] = 0

Appending to ../projects/notes_api/src/notes_api/schemas.py


Part 6: the page envelope and the error body.

In [192]:
%%writefile -a ../projects/notes_api/src/notes_api/schemas.py


class NotePage(BaseModel):
    """One page of notes, plus what a client needs to ask for the next one."""

    items: list[NoteRead]
    total: int
    limit: int
    offset: int


class ErrorMessage(BaseModel):
    """Body of 404 responses (and every HTTPException)."""

    detail: str

Appending to ../projects/notes_api/src/notes_api/schemas.py


**`store.py`.** The storage layer: a `Note` dataclass (the server's record), a
`NoteNotFoundError`, and `NoteStore` with `add / get / search / replace / update /
delete`. The clock is injectable so tests can control time. Unit 6.4 swaps the dict
for SQLAlchemy behind the same method names, so `main.py` will barely change:

Part 1 of `store.py`: the stored record, the not-found error and the clock.

In [193]:
%%writefile ../projects/notes_api/src/notes_api/store.py
"""In-memory storage for notes. Unit 6.4 replaces it with SQLAlchemy behind the same methods."""

import dataclasses
import itertools
from collections.abc import Callable
from datetime import UTC, datetime

from notes_api.schemas import NoteCreate, NoteFilters, NoteUpdate


@dataclasses.dataclass(slots=True)
class Note:
    """A stored note: the server's own record, not a schema."""

    id: int
    title: str
    body: str
    tags: list[str]
    pinned: bool
    created_at: datetime
    updated_at: datetime


class NoteNotFoundError(LookupError):
    """No note has the requested id."""

    def __init__(self, note_id: int) -> None:
        """Remember the id; the message is what the API sends as the 404 detail."""
        super().__init__(f"note {note_id} not found")
        self.note_id = note_id


def utc_now() -> datetime:
    """The current time in UTC (the default clock)."""
    return datetime.now(UTC)

Overwriting ../projects/notes_api/src/notes_api/store.py


Part 2: the filtering and sorting helpers.

In [194]:
%%writefile -a ../projects/notes_api/src/notes_api/store.py


def matches(note: Note, filters: NoteFilters) -> bool:
    """True if the note passes every filter."""
    if filters.pinned is not None and note.pinned != filters.pinned:
        return False
    if not set(filters.tag) <= set(note.tags):
        return False
    if filters.q:
        needle = filters.q.casefold()
        return needle in note.title.casefold() or needle in note.body.casefold()
    return True


def sort_key(field_name: str) -> Callable[[Note], tuple[object, int]]:
    """Sort by one field (text ignores case), then by id so the order is always stable."""

    def key(note: Note) -> tuple[object, int]:
        """The sort key of one note."""
        value = getattr(note, field_name)
        return (value.casefold() if isinstance(value, str) else value, note.id)

    return key

Appending to ../projects/notes_api/src/notes_api/store.py


Part 3: `NoteStore`, creating and reading notes.

In [195]:
%%writefile -a ../projects/notes_api/src/notes_api/store.py


class NoteStore:
    """Notes in a dict, ids from a counter. Used only from async handlers (one thread)."""

    def __init__(self, clock: Callable[[], datetime] = utc_now) -> None:
        """Start empty; clock supplies created_at/updated_at (injectable for tests)."""
        self.clock = clock
        self._notes: dict[int, Note] = {}
        self._ids = itertools.count(1)

    def clear(self) -> None:
        """Remove every note and restart ids at 1."""
        self._notes.clear()
        self._ids = itertools.count(1)

    def add(self, data: NoteCreate) -> Note:
        """Store a new note."""
        now = self.clock()
        note = Note(id=next(self._ids), **data.model_dump(), created_at=now, updated_at=now)
        self._notes[note.id] = note
        return note

    def get(self, note_id: int) -> Note:
        """The note with this id, or NoteNotFoundError."""
        try:
            return self._notes[note_id]
        except KeyError:
            raise NoteNotFoundError(note_id) from None

Appending to ../projects/notes_api/src/notes_api/store.py


Part 4: search, replace, update and delete (this part continues the class).

In [196]:
%%writefile -a ../projects/notes_api/src/notes_api/store.py

    def search(self, filters: NoteFilters) -> tuple[list[Note], int]:
        """One page of matching notes, and how many notes match in total."""
        found = [note for note in self._notes.values() if matches(note, filters)]
        descending = filters.sort.startswith("-")
        found.sort(key=sort_key(filters.sort.removeprefix("-")), reverse=descending)
        return found[filters.offset : filters.offset + filters.limit], len(found)

    def replace(self, note_id: int, data: NoteCreate) -> Note:
        """Replace every client-owned field (PUT)."""
        note = self.get(note_id)
        replaced = dataclasses.replace(note, **data.model_dump(), updated_at=self.clock())
        self._notes[note_id] = replaced
        return replaced

    def update(self, note_id: int, changes: NoteUpdate) -> Note:
        """Apply only the fields the client sent (PATCH); updated_at moves only on a change."""
        note = self.get(note_id)
        sent = changes.model_dump(exclude_unset=True)
        if all(getattr(note, name) == value for name, value in sent.items()):
            return note
        updated = dataclasses.replace(note, **sent, updated_at=self.clock())
        self._notes[note_id] = updated
        return updated

    def delete(self, note_id: int) -> None:
        """Remove the note, or raise NoteNotFoundError."""
        self.get(note_id)
        del self._notes[note_id]

Appending to ../projects/notes_api/src/notes_api/store.py


**`main.py`.** The app with OpenAPI metadata and function-name operation ids, a
health check and the six note operations. Handlers return `Note` dataclasses;
`response_model=NoteRead` converts and filters them. The store is a module-level
object for now; Unit 6.3 replaces it with dependency injection (and splits the
routes into an `APIRouter`):

Part 1 of `main.py`: imports and the operation-id function.

In [197]:
%%writefile ../projects/notes_api/src/notes_api/main.py
"""The Notes API application.

Unit 6.2 version: path operations, Pydantic schemas, response models, status codes, errors and
OpenAPI metadata. Later units add dependency injection and routers (6.3), a database (6.4),
authentication (6.5), background jobs and caching (6.6), observability (6.7) and deployment (6.8).
"""

from typing import Annotated, Any

from fastapi import FastAPI, HTTPException, Path, Query, Response, status
from fastapi.routing import APIRoute

from notes_api import __version__
from notes_api.schemas import ErrorMessage, NoteCreate, NoteFilters, NotePage, NoteRead, NoteUpdate
from notes_api.store import Note, NoteNotFoundError, NoteStore


def operation_id(route: APIRoute) -> str:
    """Use the function name as the OpenAPI operationId (clean names in generated clients)."""
    return route.name

Overwriting ../projects/notes_api/src/notes_api/main.py


Part 2: the app and its OpenAPI metadata, the store, shared types, health.

In [198]:
%%writefile -a ../projects/notes_api/src/notes_api/main.py


app = FastAPI(
    title="Notes API",
    version=__version__,
    summary="Create, search and organize notes.",
    description="Built step by step in Phase 6 of the course. Timestamps are UTC, ISO 8601.",
    openapi_tags=[
        {"name": "notes", "description": "Create, read, update and delete notes."},
        {"name": "health", "description": "Liveness probe for load balancers and tests."},
    ],
    generate_unique_id_function=operation_id,
)

notes_store = NoteStore()  # module-level for now; Unit 6.3 turns it into a dependency

NoteId = Annotated[int, Path(ge=1, description="The note's id.")]
NOT_FOUND: dict[int | str, dict[str, Any]] = {
    404: {"model": ErrorMessage, "description": "No note has this id."}
}


def find_note(note_id: int) -> Note:
    """The stored note, or an HTTP 404."""
    try:
        return notes_store.get(note_id)
    except NoteNotFoundError as error:
        raise HTTPException(status.HTTP_404_NOT_FOUND, detail=str(error)) from error


@app.get("/health", tags=["health"])
async def health() -> dict[str, str]:
    """Liveness probe."""
    return {"status": "ok"}

Appending to ../projects/notes_api/src/notes_api/main.py


Part 3: create, list and fetch.

In [199]:
%%writefile -a ../projects/notes_api/src/notes_api/main.py


@app.post("/notes", response_model=NoteRead, status_code=status.HTTP_201_CREATED, tags=["notes"])
async def create_note(note: NoteCreate, response: Response) -> Note:
    """Create a note; the Location header points at it."""
    created = notes_store.add(note)
    response.headers["Location"] = app.url_path_for("get_note", note_id=str(created.id))
    return created


@app.get("/notes", response_model=NotePage, tags=["notes"])
async def list_notes(filters: Annotated[NoteFilters, Query()]) -> dict[str, object]:
    """Search, filter, sort and paginate notes."""
    notes, total = notes_store.search(filters)
    return {"items": notes, "total": total, "limit": filters.limit, "offset": filters.offset}


@app.get("/notes/{note_id}", response_model=NoteRead, responses=NOT_FOUND, tags=["notes"])
async def get_note(note_id: NoteId) -> Note:
    """Fetch one note."""
    return find_note(note_id)

Appending to ../projects/notes_api/src/notes_api/main.py


Part 4: replace, update and delete.

In [200]:
%%writefile -a ../projects/notes_api/src/notes_api/main.py


@app.put("/notes/{note_id}", response_model=NoteRead, responses=NOT_FOUND, tags=["notes"])
async def replace_note(note_id: NoteId, note: NoteCreate) -> Note:
    """Replace every field of a note (omitted fields get their defaults)."""
    find_note(note_id)
    return notes_store.replace(note_id, note)


@app.patch("/notes/{note_id}", response_model=NoteRead, responses=NOT_FOUND, tags=["notes"])
async def update_note(note_id: NoteId, changes: NoteUpdate) -> Note:
    """Change only the fields sent in the body."""
    find_note(note_id)
    return notes_store.update(note_id, changes)


@app.delete(
    "/notes/{note_id}",
    status_code=status.HTTP_204_NO_CONTENT,
    responses=NOT_FOUND,
    tags=["notes"],
)
async def delete_note(note_id: NoteId) -> None:
    """Delete a note."""
    find_note(note_id)
    notes_store.delete(note_id)

Appending to ../projects/notes_api/src/notes_api/main.py


Import the app from the files just written and walk through the API with
`TestClient`. Timestamps are left out of the printout because they change on every
run:

In [201]:
notes_main = reload_notes_api()
notes_main.notes_store.clear()


def short(note: dict[str, Any]) -> dict[str, Any]:
    """A note without its timestamps."""
    return {key: value for key, value in note.items() if not key.endswith("_at")}


with TestClient(notes_main.app) as client:
    created = client.post(
        "/notes",
        json={"title": "Buy tea", "body": "Assam, 500 g", "tags": ["Shopping"], "pinned": True},
    )
    print(created.status_code, created.headers["location"], short(created.json()))
    client.post("/notes", json={"title": "Call Ravi", "body": "about the trip", "tags": ["home"]})
    client.post("/notes", json={"title": "Pay the electricity bill", "tags": ["home", "bills"]})
    page = client.get("/notes", params={"tag": "home", "sort": "title", "limit": 10}).json()
    print("tag=home:", [note["title"] for note in page["items"]], "total", page["total"])
    print(
        "PATCH:",
        short(client.patch("/notes/2", json={"pinned": True, "tags": ["home", "travel"]}).json()),
    )
    print("PATCH null:", client.patch("/notes/2", json={"title": None}).json()["detail"][0]["msg"])
    print(
        "DELETE:", client.delete("/notes/3").status_code, "then GET:", client.get("/notes/3").json()
    )

201 /notes/1 {'id': 1, 'title': 'Buy tea', 'body': 'Assam, 500 g', 'tags': ['shopping'], 'pinned': True, 'word_count': 3}
tag=home: ['Call Ravi', 'Pay the electricity bill'] total 2
PATCH: {'id': 2, 'title': 'Call Ravi', 'body': 'about the trip', 'tags': ['home', 'travel'], 'pinned': True, 'word_count': 3}
PATCH null: Value error, fields cannot be null: title


DELETE: 204 then GET: {'detail': 'note 3 not found'}


**Tests.** The `client` fixture clears the module-level store before each test
(which is exactly the coupling Unit 6.3's dependency injection removes). The tests
cover every route, the validation rules, filtering and sorting, PATCH semantics and
the OpenAPI operation ids:

In [202]:
%%writefile ../projects/notes_api/tests/conftest.py
"""Shared fixtures: a TestClient whose in-memory store starts empty for every test."""

from collections.abc import Iterator

import pytest
from fastapi.testclient import TestClient

from notes_api.main import app, notes_store


@pytest.fixture
def client() -> Iterator[TestClient]:
    """A TestClient for the app, on an empty store."""
    notes_store.clear()
    with TestClient(app) as test_client:
        yield test_client

Overwriting ../projects/notes_api/tests/conftest.py


Part 1 of `test_notes.py`: two helpers and the health check.

In [203]:
%%writefile ../projects/notes_api/tests/test_notes.py
"""Behaviour of the Notes API v1 endpoints."""

from typing import Any

import pytest
from fastapi.testclient import TestClient


def create(client: TestClient, **fields: Any) -> dict[str, Any]:
    """POST a note (title defaults to "Buy tea") and return the created note."""
    response = client.post("/notes", json={"title": "Buy tea", **fields})
    assert response.status_code == 201, response.text
    return response.json()


def ids(response: Any) -> list[int]:
    """Ids of the notes on a page."""
    assert response.status_code == 200, response.text
    return [note["id"] for note in response.json()["items"]]


def test_health(client: TestClient) -> None:
    """GET /health answers ok."""
    assert client.get("/health").json() == {"status": "ok"}

Overwriting ../projects/notes_api/tests/test_notes.py


Part 2: creating notes, valid and invalid.

In [204]:
%%writefile -a ../projects/notes_api/tests/test_notes.py


def test_create_returns_201_location_and_normalized_fields(client: TestClient) -> None:
    """POST returns 201, a Location header and cleaned-up fields."""
    response = client.post(
        "/notes", json={"title": "  Buy tea  ", "body": "Assam, 500 g", "tags": ["Shop", "shop"]}
    )
    assert response.status_code == 201
    assert response.headers["location"] == "/notes/1"
    note = response.json()
    assert {key: note[key] for key in ("id", "title", "tags", "pinned", "word_count")} == {
        "id": 1,
        "title": "Buy tea",
        "tags": ["shop"],
        "pinned": False,
        "word_count": 3,
    }
    assert note["created_at"] == note["updated_at"]


@pytest.mark.parametrize(
    ("body", "location"),
    [
        ({}, ["body", "title"]),
        ({"title": "   "}, ["body", "title"]),
        ({"title": "x", "color": "red"}, ["body", "color"]),
        ({"title": "x", "tags": ["two words"]}, ["body", "tags", 0]),
        ({"title": "x", "tags": [f"tag-{number}" for number in range(11)]}, ["body", "tags"]),
    ],
)
def test_create_rejects_invalid_input(
    client: TestClient, body: dict[str, Any], location: list[Any]
) -> None:
    """Invalid bodies are 422 errors pointing at the bad field."""
    response = client.post("/notes", json=body)
    assert response.status_code == 422
    assert response.json()["detail"][0]["loc"] == location

Appending to ../projects/notes_api/tests/test_notes.py


Part 3: fetching one note, and pagination.

In [205]:
%%writefile -a ../projects/notes_api/tests/test_notes.py


def test_get_one_and_unknown_id(client: TestClient) -> None:
    """GET one note; unknown ids are 404, non-positive ids 422."""
    created = create(client)
    assert client.get("/notes/1").json() == created
    missing = client.get("/notes/99")
    assert (missing.status_code, missing.json()) == (404, {"detail": "note 99 not found"})
    assert client.get("/notes/0").status_code == 422


def test_list_paginates_and_counts_all_matches(client: TestClient) -> None:
    """limit/offset select a page; total counts every match."""
    for number in range(5):
        create(client, title=f"Note {number}")
    response = client.get("/notes", params={"limit": 2, "offset": 2, "sort": "created_at"})
    assert ids(response) == [3, 4]
    assert {key: response.json()[key] for key in ("total", "limit", "offset")} == {
        "total": 5,
        "limit": 2,
        "offset": 2,
    }

Appending to ../projects/notes_api/tests/test_notes.py


Part 4: default order and filters.

In [206]:
%%writefile -a ../projects/notes_api/tests/test_notes.py


def test_list_default_order_is_newest_first(client: TestClient) -> None:
    """Without sort, the newest note comes first."""
    for title in ("first", "second", "third"):
        create(client, title=title)
    assert ids(client.get("/notes")) == [3, 2, 1]


def test_list_filters_by_tag_pinned_and_text(client: TestClient) -> None:
    """tag (all of them), pinned and q filter the list."""
    create(client, title="Buy tea", tags=["shopping", "home"], pinned=True)
    create(client, title="Call Ravi", body="about the TEA estate trip", tags=["home"])
    create(client, title="Pay bill", tags=["shopping"])
    assert ids(client.get("/notes?tag=home&sort=created_at")) == [1, 2]
    assert ids(client.get("/notes?tag=home&tag=shopping")) == [1]
    assert ids(client.get("/notes?pinned=false&sort=created_at")) == [2, 3]
    assert ids(client.get("/notes?q=tea&sort=created_at")) == [1, 2]

Appending to ../projects/notes_api/tests/test_notes.py


Part 5: sorting, bad parameters and PUT.

In [207]:
%%writefile -a ../projects/notes_api/tests/test_notes.py


def test_list_sorts_titles_ignoring_case(client: TestClient) -> None:
    """sort=title ignores case."""
    for title in ("banana", "Apple", "cherry"):
        create(client, title=title)
    titles = [note["title"] for note in client.get("/notes?sort=title").json()["items"]]
    assert titles == ["Apple", "banana", "cherry"]


@pytest.mark.parametrize("query", ["limit=0", "limit=101", "offset=-1", "sort=newest", "x=1"])
def test_list_rejects_bad_parameters(client: TestClient, query: str) -> None:
    """Out-of-range or unknown query parameters are 422 errors."""
    assert client.get(f"/notes?{query}").status_code == 422


def test_put_replaces_every_field(client: TestClient) -> None:
    """PUT resets every field the body leaves out."""
    create(client, tags=["home"], pinned=True, body="old body")
    replaced = client.put("/notes/1", json={"title": "Buy green tea"})
    assert replaced.status_code == 200
    assert {key: replaced.json()[key] for key in ("title", "body", "tags", "pinned")} == {
        "title": "Buy green tea",
        "body": "",
        "tags": [],
        "pinned": False,
    }

Appending to ../projects/notes_api/tests/test_notes.py


Part 6: PATCH semantics and DELETE.

In [208]:
%%writefile -a ../projects/notes_api/tests/test_notes.py


def test_patch_changes_only_the_fields_sent(client: TestClient) -> None:
    """PATCH changes only the fields in the body."""
    created = create(client, body="Assam", tags=["home"])
    patched = client.patch("/notes/1", json={"pinned": True}).json()
    assert patched["pinned"] is True
    assert {key: patched[key] for key in ("title", "body", "tags")} == {
        key: created[key] for key in ("title", "body", "tags")
    }
    assert patched["updated_at"] >= created["updated_at"]


def test_patch_without_a_change_keeps_updated_at(client: TestClient) -> None:
    """A PATCH that changes nothing keeps updated_at."""
    created = create(client, pinned=True)
    for body in ({}, {"pinned": True}):
        assert client.patch("/notes/1", json=body).json()["updated_at"] == created["updated_at"]


@pytest.mark.parametrize("body", [{"title": None}, {"pinned": None}, {"colour": "red"}])
def test_patch_rejects_null_and_unknown_fields(client: TestClient, body: dict[str, Any]) -> None:
    """PATCH refuses explicit nulls and unknown fields."""
    create(client)
    assert client.patch("/notes/1", json=body).status_code == 422


def test_delete_then_gone(client: TestClient) -> None:
    """DELETE answers 204 and the note is gone."""
    create(client)
    deleted = client.delete("/notes/1")
    assert (deleted.status_code, deleted.content) == (204, b"")
    assert client.get("/notes/1").status_code == 404

Appending to ../projects/notes_api/tests/test_notes.py


Part 7: 404 for every method, and the operation ids.

In [209]:
%%writefile -a ../projects/notes_api/tests/test_notes.py


@pytest.mark.parametrize("method", ["put", "patch", "delete"])
def test_unknown_note_is_404_for_every_method(client: TestClient, method: str) -> None:
    """PUT, PATCH and DELETE on an unknown id are 404 errors."""
    kwargs = {} if method == "delete" else {"json": {"title": "x"}}
    response = client.request(method.upper(), "/notes/99", **kwargs)
    assert (response.status_code, response.json()) == (404, {"detail": "note 99 not found"})


def test_openapi_uses_function_names_as_operation_ids(client: TestClient) -> None:
    """operationIds are the handler function names."""
    paths = client.get("/openapi.json").json()["paths"]
    operation_ids = {op["operationId"] for ops in paths.values() for op in ops.values()}
    assert operation_ids == {
        "health",
        "create_note",
        "list_notes",
        "get_note",
        "replace_note",
        "update_note",
        "delete_note",
    }

Appending to ../projects/notes_api/tests/test_notes.py


Run the suite with the kernel's environment (`python -m pytest` inside the project
folder, where `pyproject.toml` configures pytest):

In [210]:
pytest_run = subprocess.run(
    [sys.executable, "-m", "pytest", "-p", "no:cacheprovider"],
    cwd=NOTES_API_DIR,
    capture_output=True,
    text=True,
    env=CHILD_ENV,
    timeout=300,
)
print(tidy_test_output(pytest_run.stdout + pytest_run.stderr).splitlines()[-1])
print(f"{pytest_run.returncode=}")

28 passed in <time>
pytest_run.returncode=0


And as a project of its own: `uv lock` resolves the dependencies into
`projects/notes_api/uv.lock` (unchanged on later runs unless `pyproject.toml`
changes), and `uv run pytest` creates the project's own `.venv` (git-ignored) and
runs the tests there, which is what CI and the Docker image in Unit 6.8 will do.
The first run downloads or links packages and takes a few seconds; offline, uv uses
its cache or the cell prints a skip message:

In [211]:
if HAS_UV:
    lock_run = subprocess.run(
        ["uv", "lock"],
        cwd=NOTES_API_DIR,
        capture_output=True,
        text=True,
        env=CHILD_ENV,
        timeout=600,
    )
    print(
        tidy_test_output(lock_run.stderr).splitlines()[-1]
        if lock_run.stderr
        else lock_run.returncode
    )
    if lock_run.returncode == 0:
        uv_pytest = subprocess.run(
            ["uv", "run", "--frozen", "pytest", "-p", "no:cacheprovider"],
            cwd=NOTES_API_DIR,
            capture_output=True,
            text=True,
            env=CHILD_ENV,
            timeout=600,
        )
        print(
            tidy_test_output(uv_pytest.stdout).splitlines()[-1],
            f"(exit code {uv_pytest.returncode})",
        )
    else:
        print("SKIP uv run pytest: uv lock failed (offline with an empty cache?)")
else:
    print("SKIP: the uv command is not on PATH")
print(f"{(NOTES_API_DIR / 'uv.lock').exists()=}")

Resolved 66 packages in <time>


28 passed in <time> (exit code 0)
(NOTES_API_DIR / 'uv.lock').exists()=True


Finally, the real thing: Uvicorn serves `notes_api.main:app` from `src/` in a child
process; the 6.2.9 generator turns its live `/openapi.json` into TypeScript types and
a client; a Node script uses the client to create and query notes (skipped without
`node`). This is the contract a frontend team would build against:

In [212]:
NOTES_DEMO_SOURCE = """
import { createNote, listNotes, updateNote } from "./notes_client.mjs";
const baseUrl = process.argv[2];
const tea = await createNote(baseUrl, { title: "Buy tea", tags: ["shopping"] });
await createNote(baseUrl, { title: "Plan the Goa trip", body: "tickets, hotel", tags: ["travel"] });
await updateNote(baseUrl, tea.id, { pinned: true });
const pinned = await listNotes(baseUrl, { pinned: true });
console.log("pinned:", pinned.items.map((note) => note.title), "total:", pinned.total);
const travel = await listNotes(baseUrl, { tag: ["travel"], sort: "title" });
console.log("travel:", travel.items.map((note) => `${note.title} (${note.word_count} words)`));
"""
with uvicorn_server("notes_api.main:app", app_dir=NOTES_SRC) as base_url:
    notes_openapi = httpx.get(f"{base_url}/openapi.json", trust_env=False).json()
    (PHASE6_DIR / "notes_types.d.ts").write_text(
        typescript_declarations(notes_openapi) + "\n", encoding="utf-8"
    )
    (PHASE6_DIR / "notes_client.mjs").write_text(
        javascript_client(notes_openapi, "notes_types.d.ts") + "\n", encoding="utf-8"
    )
    (PHASE6_DIR / "notes_demo.mjs").write_text(NOTES_DEMO_SOURCE.strip() + "\n", encoding="utf-8")
    if HAS_NODE:
        node_run = subprocess.run(
            ["node", "notes_demo.mjs", base_url],
            cwd=PHASE6_DIR,
            capture_output=True,
            text=True,
            timeout=60,
        )
        print(node_run.stdout + node_run.stderr)
    else:
        print("SKIP the Node demo: the node command is not installed")
print(
    re.search(
        r"export interface NoteRead \{.*?\}",
        (PHASE6_DIR / "notes_types.d.ts").read_text(encoding="utf-8"),
        re.S,
    )[0]
)

pinned: [ 'Buy tea' ] total: 1
travel: [ 'Plan the Goa trip (2 words)' ]

export interface NoteRead {
  id: number;
  title: string;
  body: string;
  tags: string[];
  pinned: boolean;
  created_at: string;
  updated_at: string;
  word_count: number;
}


What v1 deliberately leaves for later units: the module-level `notes_store` (tests
must reset it; Unit 6.3 injects it with `Depends`), one big module of routes (6.3:
`APIRouter`), no settings or CORS (6.3), memory-only storage (6.4: SQLAlchemy and
Alembic), no users (6.5: authentication and ownership), no background work (6.6),
no metrics or tracing (6.7), no container (6.8).

### Recap — Unit 6.2

| Concept | One-line takeaway | JS / TS equivalent |
| --- | --- | --- |
| Architecture | FastAPI = Starlette (ASGI, routing) + Pydantic (validation) + OpenAPI; it reads your signatures | Fastify + JSON Schema, NestJS |
| Path operations | one decorator per method; typed path params; `StrEnum`; `{x:path}`; first declared route wins | `app.get("/x/:id")` |
| Query parameters | non-path simple params; `Annotated[int, Query(ge=1, le=100)]`; lists need `Query()`; query models with `extra="forbid"` | `req.query` + zod |
| Request bodies | a model parameter is the body; several bodies are keyed by name; `Body(embed=True)` | `express.json()` + DTO |
| Pydantic v2 | types + `Field` constraints; `field_validator` / `model_validator` / `computed_field`; `model_dump` options; `ConfigDict`; lax vs strict; `TypeAdapter`; `errors()`; `Annotated` types | zod (`z.object`, `.refine`, `.transform`, `safeParse`) |
| Schema design | Create / Update / Read schemas; PATCH = `exclude_unset`; reject explicit `null` | separate DTOs |
| Responses | `response_model` filters output; `status_code`; `JSONResponse`, `StreamingResponse`, `FileResponse`, `RedirectResponse`; `response: Response` for headers/cookies; `HTTPException` | `res.status().json()`, `res.cookie()` |
| `async def` vs `def` | blocking code in `async def` freezes the worker; `def` runs in a 40-thread pool; offload with `run_in_threadpool` | (Node has no sync handlers) |
| OpenAPI | `/docs`, `/redoc`, `/openapi.json`; metadata, tags, examples, operation ids; generate typed clients | `@nestjs/swagger`, `openapi-typescript` |
| How it works | decorators register routes; `inspect.signature` + `Annotated` metadata classify parameters; validators compiled once | reflect-metadata in NestJS |

**Further reading**

- [FastAPI tutorial](https://fastapi.tiangolo.com/tutorial/) (path, query and body
  parameters, response models, status codes, `async`),
  [Starlette responses](https://www.starlette.io/responses/)
- [Pydantic: models](https://docs.pydantic.dev/latest/concepts/models/),
  [fields](https://docs.pydantic.dev/latest/concepts/fields/),
  [validators](https://docs.pydantic.dev/latest/concepts/validators/),
  [serialization](https://docs.pydantic.dev/latest/concepts/serialization/),
  [strict mode](https://docs.pydantic.dev/latest/concepts/strict_mode/),
  [type adapter](https://docs.pydantic.dev/latest/concepts/type_adapter/)
- [`inspect`](https://docs.python.org/3/library/inspect.html),
  [`typing.Annotated`](https://docs.python.org/3/library/typing.html#typing.Annotated),
  [`annotationlib`](https://docs.python.org/3/library/annotationlib.html),
  [`enum.StrEnum`](https://docs.python.org/3/library/enum.html#enum.StrEnum)
- [PEP 593](https://peps.python.org/pep-0593/) (`Annotated`),
  [PEP 649](https://peps.python.org/pep-0649/) / [PEP 749](https://peps.python.org/pep-0749/) (lazy annotations),
  [PEP 695](https://peps.python.org/pep-0695/) (`type` aliases)
- [OpenAPI 3.1 specification](https://spec.openapis.org/oas/v3.1.0),
  [openapi-typescript](https://openapi-ts.dev/)